# Swiss Citation — Anchor-Funnel v7.5 multi-query

Runs the full anchor-funnel pipeline on **every** query in `val.csv` with **no** per-id branching. The same code path retrieves for any query you drop in — val, test, or production. Final cap `topk_final = 50,000` (effectively returns the post-gate channel union, so the aggregate measures the **upper bound** of channel recall).

## Hard constraints (carried through the whole pipeline)

- **No hardcoded lists.** No DE_LEX, no fixed BGE list, no statute cluster table, no per-query branching.
- **No query-specific knowledge.** Architecture generalises to any val / test / production query.
- **No train data.** `train.csv` is never read.
- **Open-source models only.** Qwen3-32B (query expansion + HyDE) and Qwen3-Embedding-8B (vectors).

## Phase map

| Phase | Cells | Purpose |
|---|---|---|
| 1. Setup | env, drive, paths, knobs | runtime + IO + global config |
| 2. Corpus indexes | law-llm jsonl, court-v5 jsonl | one streaming pass → all in-memory indexes |
| 3. Citation graph | sqlite → `idx_graph_{in,out}` | 4-layer alias graph, 23.65 M edges |
| 4. Per-area bedrock + co-citation | corpus statistics | per-legal-area universal articles + co-cited canon clusters |
| 5. BM25 (FTS5 in-memory) | per-language FTS5 indices | lexical match in DE / FR / IT / EN |
| 6. Vector setup | Qwen3-Embedding-8B + E_GPU | brute-force GPU cosine over 2.65 M corpus rows |
| 7. **Query expansion + HyDE (Qwen3-32B)** | structured-JSON targets **AND** hypothetical-answer paragraphs | one model load, two passes per query |
| 7.6 Encode queries | raw + enriched + **HyDE** | three vector queries per row |
| 8. Channels (**PRF + dual-pass**) | `run_channels()` | 15 retrieval signals per query |
| 9. RRF fusion + master loop | weighted RRF + neg-gate + round-robin | top-50,000 final pool, looped over every query |
| 10. Aggregate + missed-gold diagnostic | per-query table, macro R@K curve, per-channel mean recall, miss breakdown | shows where the channel union is dark |
| 11. Save artifacts | per-query `final_topk_<qid>.json` + aggregate `summary_multiquery.json` | reproducibility |

## Channel summary (15 channels)

| # | Channel | Index used | Gap closed | Budget | RRF weight |
|---|---|---|---:|---:|---:|
| 1 | `law_direct_match` | `idx_law_direct` | LLM-named statutes (incl. corpus-related codes) → matching law rows | None | 1.2 |
| 2 | `court_statute` | `idx_court_statute` | LLM-named statute appears in row's `statute_anchors`. Specificity + paragraph_role boost. PRF augmented. | 8000 | 1.5 |
| 3 | `co_citation` | `co_neighbours` + law / court indexes | Statute cluster co-cited with LLM target. Specificity-weighted. PRF augmented. | 2500 | 0.7 |
| 4 | `per_area_bedrock` | `per_area_canon_count`, filtered by corpus-derived code family | Universal procedural articles per legal area. PRF augmented. | 1500 | 1.5 |
| 5 | `statute_backprop` | `doc_statute_anchors` + dual-pass seed | Caught court rows → law articles they cite. Rare-canon specificity weight. **Dual-pass**: pass-1 seed + PRF-expanded seed, merged. | 2000 | 2.5 |
| 6 | `sibling_expansion` | `idx_court_base` + `idx_judgment_importance` | All Es of caught judgments. Judgment-importance scoring. **Dual-pass**. | 5000 | 1.0 |
| 7 | `graph_forward` | `idx_graph_out` + `idx_judgment_importance` | Caught row → text-cited + case-level fan-out. **Dual-pass**. | 5000 | 2.0 |
| 8 | `graph_reverse` | `idx_graph_in` + `idx_judgment_importance` | Caught row ← rows that text-cite it. Landmark filter (importance ≥ 5). **Dual-pass**. | 3000 | 0.5 |
| 9 | `graph_2hop` (off) | `idx_graph_out` | 2-hop forward expansion. Disabled. | 1500 | 0.0 |
| 10 | `concept_en` | `idx_concept_en` | LLM concepts → English-tagged rows. Token-overlap matcher (stopwords filtered, shared-token weighted). PRF augmented. | 3000 | 1.8 |
| 11 | `term_orig` | `idx_term_orig` + `idx_term_lemma` | LLM DE / FR terms → original-language-tagged rows. German lemmatiser + substring overlap. PRF augmented. | 2500 | 1.2 |
| 12 | `bm25` | FTS5 per-language (de / fr / it / en) + `enhance()` | Per-language lexical match merged by max-score. Corpus-trained `enhance()`. | 2000 | 0.8 |
| 13 | `vector_raw` | `E_GPU` brute-force | Dense semantic match on raw query embedding. | 2000 | 1.0 |
| 14 | `vector_enriched` | `E_GPU` brute-force | Dense semantic match on keyword-enriched query. | 2000 | 1.0 |
| 15 | **`vector_hyde`** | `E_GPU` brute-force | Dense semantic match on **hypothetical-answer paragraph** (Qwen3-32B drafts a 4-6 sentence Swiss-Federal-Tribunal-style answer; embedded by Qwen3-Embedding-8B). Bridges the multilingual paragraph-vs-question vector gap. | 2000 | 1.5 |

### Guarantee channels — round-robin merged

7 channels, `guarantee_per_channel = 130` → up to 910 slots, rest of `topk_final` filled by weighted-RRF tail:

1. `law_direct_match`, 2. `per_area_bedrock`, 3. `statute_backprop`,
4. `concept_en`, 5. `graph_forward`, 6. `sibling_expansion`, 7. `term_orig`

### Role-aware negative gate

Substantive paragraph roles (`facts`, `reasoning`, `legal_standard`, `application`, `holding`, `citation`, `procedural_history`) override the noisy `is_notification_paragraph` enrichment flag, so substantive paragraphs are never dropped.

## Key v7.5 ideas (in order of build-up)

1. **Robust LLM expansion** — broad system prompt naming many Swiss codes, schema asks for 10-25 items per list, two diverse few-shots, 3-strategy JSON parser (markdown fence → brace-balanced scan → greedy), sampling retry at `temperature=0.4` if first pass is weak.
2. **Corpus-derived code-family expansion** — for each LLM-named code, find top-K corpus co-occurring codes (StPO → StBOG / BGG / BV / EMRK via corpus statistics). No hardcoded clusters.
3. **HyDE** — Qwen3-32B drafts a 4-6 sentence answer paragraph in Bundesgericht / Tribunal fédéral style, with inline German and French legal terms. Encoded as a third vector query. Bridges short-English-question vs long-multilingual-paragraph cosine gap.
4. **Corpus-side PRF** — after pass-1 retrieval, mini-fuse to pick the top-K court rows, aggregate their `statute_anchors` / `concepts_en` / `terms_original`, keep top-N new signals not in LLM targets. Pass-2 re-runs the canon / concept / term channels with augmented targets. Pure corpus signal — no hardcoded lists.
5. **Dual-pass graph channels** — sibling / graph_forward / graph_reverse / statute_backprop run twice (pass-1 seed + PRF-expanded seed) and merged per channel. Prevents PRF from displacing pass-1 gold past budget caps.


# Phase 1 — Setup


In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GB")
else:
    print("[WARN] No CUDA GPU available — vector channels will be skipped.")

Python: 3.12.13
PyTorch: 2.10.0+cu128
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, 95.0 GB


## 1.2 Mount Google Drive

**What:** Mount Drive so we can read the corpus, embeddings, and citation
graph from `/content/drive/MyDrive/swiss_law/`.

**Why:** The 21 GB of fp16 embeddings, 24 GB unified retrieval SQLite, and
2.4 GB citation graph all live on Drive — too large to download per run.

**Expected:** "Mounted at /content/drive". Skip silently if running locally.

**Failure modes:**
- "Drive not authorized" → click the OAuth link Colab prints.
- Mount succeeds but `MyDrive/swiss_law` is empty → wrong account; remount.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"[skip] Not on Colab or drive already mounted: {e}")

Mounted at /content/drive


## 1.3 Resolve all data paths

**What:** Auto-detect `DATA_ROOT` (could be Drive, local, or Kaggle) and
build a `PATHS` dict pointing at every required artifact.

**Why:** Same notebook should run on Colab, Kaggle, or locally without code
changes. Each artifact is checked for existence with a clear OK/MISSING flag.

**Required inputs:**
- `data/val.csv` — 10 English queries with gold citations.
- `law_llm_descriptors_*.jsonl` — LLM enrichment of all 175k laws.
- `court_authority_cards_v5_unified.jsonl` — court enrichment (10.4 GB).
- `embeddings/qwen3_8b_unified_chunk*.npy` — 27 fp16 chunks (21 GB total).
- `embeddings/qwen3_8b_unified_manifest.parquet` — doc_id ↔ row_index.
- **v7 NEW:** `data_insights/citation_graph_extracted.sqlite` — 4-layer graph (~2.4 GB).

**Expected:** All flags `OK`. If `graph_db` is `MISSING`, graph channels
silently skip (recall drops back to v6 levels).

**Failure modes:**
- `MISSING law_llm` or `court_v5` → notebook can't build any index. Stop.
- `MISSING emb_dir` → vector channels skipped; BM25 + anchors still run.
- `MISSING graph_db` → graph channels skipped; sibling_expansion (court_base) only.

In [ ]:
from pathlib import Path

CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/swiss_law"),
    Path("/content/drive/MyDrive/swiss_citation_extraction"),
    Path("/content/swiss_citation_extraction"),
    Path(r"E:/swiss_citation_extraction"),
    Path.cwd(),
]

DATA_ROOT = None
for root in CANDIDATE_ROOTS:
    if (root / "data" / "val.csv").exists():
        DATA_ROOT = root; break
if DATA_ROOT is None:
    print("[warn] Could not auto-detect DATA_ROOT — defaulting to /content/drive/MyDrive/swiss_law")
    DATA_ROOT = Path("/content/drive/MyDrive/swiss_law")
print(f"DATA_ROOT = {DATA_ROOT}")

def first_existing(*paths):
    for p in paths:
        if p.exists(): return p
    return paths[0]

PATHS = {
    "val_csv": DATA_ROOT / "data" / "val.csv",
    "law_llm": first_existing(
        DATA_ROOT / "data" / "checkpoints" / "law_llm_descriptors_0000000_all.jsonl",
        DATA_ROOT / "law_json_llm_output" / "law_llm_descriptors_0000000_all.jsonl",
    ),
    "court_v5": first_existing(
        DATA_ROOT / "artifacts_v2" / "court_authority_cards_v5_unified.jsonl",
        DATA_ROOT / "artifacts" / "court_authority_cards_v5_unified.jsonl",
    ),
    "emb_dir":      DATA_ROOT / "artifacts" / "embeddings",
    "emb_manifest": DATA_ROOT / "artifacts" / "embeddings" / "qwen3_8b_unified_manifest.parquet",
    "graph_db": first_existing(
        DATA_ROOT / "data_insights" / "citation_graph_extracted.sqlite",
        DATA_ROOT / "citation_graph_extracted.sqlite",
    ),
    "out_dir":      DATA_ROOT / "research" / "anchor_funnel_val001_v7",
}
PATHS["out_dir"].mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    if k == "out_dir": continue
    flag = "OK     " if p.exists() else "MISSING"
    print(f"  {flag}  {k:<14} {p}")

EMB_AVAILABLE   = PATHS["emb_dir"].exists() and any(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
GRAPH_AVAILABLE = PATHS["graph_db"].exists()
print()
print(f"  EMB_AVAILABLE   = {EMB_AVAILABLE}")
print(f"  GRAPH_AVAILABLE = {GRAPH_AVAILABLE}")

DATA_ROOT = /content/drive/MyDrive/swiss_law
  OK       val_csv        /content/drive/MyDrive/swiss_law/data/val.csv
  OK       law_llm        /content/drive/MyDrive/swiss_law/data/checkpoints/law_llm_descriptors_0000000_all.jsonl
  OK       court_v5       /content/drive/MyDrive/swiss_law/artifacts_v2/court_authority_cards_v5_unified.jsonl
  OK       emb_dir        /content/drive/MyDrive/swiss_law/artifacts/embeddings
  OK       emb_manifest   /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_manifest.parquet
  OK       graph_db       /content/drive/MyDrive/swiss_law/data_insights/citation_graph_extracted.sqlite

  EMB_AVAILABLE   = True
  GRAPH_AVAILABLE = True


## 1.4 Knob panel (CONFIG)

**What:** All architecture knobs live in a single `CONFIG` dict — budgets,
RRF k, BM25 limits, vector top-k, query-expansion model, etc.

**Why:** Editing budgets without touching channel code is essential when
diagnosis suggests "channel X dropped this gold due to truncation, lift
budget". Every budget is rationalized in a comment.

**Sensitive knobs:**
- `budget_sibling = 2000` — was 500 in v6, dropped sibling Es non-deterministically.
- `budget_graph_forward = 1500` — graph 1-hop forward; case-level fan-out adds
  up to ~30 expansions per seed, 1500 covers 50 seeds × 30 siblings.
- `budget_graph_reverse = 1000` — co-citing rows; bounded to keep ranking signal.
- `enable_graph_2hop = False` — 2-hop tends to dump procedural articles already
  caught elsewhere; flip to True only after Phase 10 says so.

**Expected:** dict prints cleanly; nothing should be `None` except `budget_law_direct`.

In [ ]:
import json

CONFIG = {
    "topk_final": 50000,

    # --- Channel budgets ---------------------------------------------------
    # law_direct_match: every law row whose canonical citation matches a
    # (LLM-named OR co-cited) statute target. Tiny per canon; uncapped is safe.
    "budget_law_direct":     None,

    # court_statute: court rows annotated with one of the LLM-named statutes.
    "budget_court_statute":  8000,   # v7.5: lifted 600->8000 (multi-match scoring + role boost now have room)

    # concept_en: rows whose concepts_en token overlaps with LLM concept_targets.
    "budget_concept":        3000,   # v7.5: lifted (rank-200+ gold lost otherwise)

    # term_orig: rows whose terms_original (DE/FR/IT) overlap with LLM term_targets.
    "budget_term":           2500,   # v7.5: lifted (deep-rank gold survives)

    # per_area_bedrock: most-cited canonical statutes within the LLM-named legal_area.
    # Per-area_top_n is internal cap; this budget caps the bedrock channel output.
    "budget_per_area":       1500,   # v7.5: procedural cluster lives at rank 100-500

    # co_citation: for each LLM statute target, fetch top-K co-citation neighbours
    # and pull their law + court rows.
    "budget_co_citation":    2500,   # v7.5: more co-cited candidates

    # bm25: lexical match on enriched query text. enhance() adds top-K corpus-
    # associated codes to query.
    "budget_bm25":           2000,   # v7.5: multi-language each gets 500

    # vector_raw / vector_enriched: dense semantic match using Qwen3-Embedding-8B
    # against full-corpus E_GPU.
    "budget_vector":         2000,   # v7.5: Obs 3 ceiling 0.289 — need deeper pool
    "budget_vector_enriched":2000,
    "budget_vector_hyde":    2000,   # HyDE — hypothetical-answer vector


    # statute_backprop: each caught court row contributes its cited statutes;
    # score = number of distinct caught court rows citing that article.
    # Surfaces procedural cluster (Art. 100 BGG, Art. 422 StPO, etc.).
    "budget_backprop":       2000,   # v7.5: rare-specificity scoring + more law candidates

    # sibling_expansion (court_base): caught court row → all Es of same judgment.
    # v6 had 500 → non-deterministic slice dropped val_001 sibling Es. 2000 fits
    # 100 seeds × 20 Es each.
    "budget_sibling":        5000,   # v7.5: judgment-importance scoring + bigger pool

    # graph_forward / graph_reverse / graph_2hop: 1-hop and 2-hop traversal of
    # the citation graph (intra-judgment backrefs + date aliases + range +
    # case-level fan-out). budget_forward sized for case-level fan-out from
    # ~50 seeds × 30 sibling-fanout ≈ 1500.
    "budget_graph_forward":  5000,   # v7.5: case-level fan-out + judgment importance
    "budget_graph_reverse":  3000,
    "enable_graph_2hop":     False,
    "budget_graph_2hop":     1500,

    # --- RRF + guarantee ---------------------------------------------------
    "rrf_k": 60,
    # Channels whose hits are PREPENDED before the RRF tail in Phase 9.
    # Keep this list small — graph_forward/reverse compete via RRF, only the
    # high-precision channels are guaranteed.
    # v7.5: 7-channel guarantee with smaller per-channel cap. Round-robin
    # ensures each high-recall channel contributes regardless of RRF score.
    # 7 channels × cap=130 = 910 guarantee slots, leaves ~90 RRF tail.
    "guarantee_channels": [
        "law_direct_match",
        "per_area_bedrock",
        "statute_backprop",
        "concept_en",            # v7.5 NEW
        "graph_forward",         # v7.5 NEW
        "sibling_expansion",     # v7.5 NEW
        "term_orig",             # v7.5 NEW
    ],
    "guarantee_per_channel": 130,
    # v7.5 weights — updated based on v7.4 per-channel mean recall + structural
    # value (multi-channel role).
    "channel_weights": {
        "statute_backprop":  2.5,   # mean recall 0.453 — universal best
        "graph_forward":     2.0,   # mean recall 0.324 — case-level fan-out
        "concept_en":        1.8,   # v7.5 lifted — token-overlap matcher now strong
        "court_statute":     1.5,   # v7.5 lifted — specificity+role scoring
        "per_area_bedrock":  1.5,
        "vector_raw":        1.0,
        "vector_enriched":   1.0,
        "vector_hyde":       1.5,   # HyDE — strongest vector signal expected

        "term_orig":         1.2,   # lemma+substring scoring
        "law_direct_match":  1.2,
        "sibling_expansion": 1.0,
        "bm25":              0.8,
        "co_citation":       0.7,
        "graph_reverse":     0.5,
        "graph_2hop":        0.0,
    },
    # v7.5 NEW: corpus-derived code-family expansion for per_area_bedrock.
    # When LLM names "StPO", we ALSO admit StBOG/BGG/BV/EMRK canons via
    # corpus co-citation evidence (not a hardcoded list).
    "code_family_top_k": 8,

    # --- Pseudo-relevance feedback (PRF) --------------------------------
    # After pass-1 retrieval, mini-fuse to pick the top-K court rows, then
    # aggregate their statute_anchors / concepts_en / terms_original. The
    # top-N NEW ones (not in LLM targets) feed a pass-2 run of the
    # canon/concept/term-dependent channels. Pure corpus signal — no
    # hardcoded lists.
    "enable_prf":              True,
    "prf_top_k_for_extraction": 4000,
    "prf_new_statutes_k":         40,
    "prf_new_concepts_k":         60,
    "prf_new_terms_k":            60,


    # --- Per-area bedrock --------------------------------------------------
    "per_area_top_n": 1000,   # v7.5: procedural cluster (422/428/135 StPO) lives at rank 100-500

    # --- Co-citation -------------------------------------------------------
    "co_citation_top_k_per_target":    50,   # v7.5: more cluster expansion
    "co_citation_min_co_count":        50,
    # drop neighbours that are TOO globally common (Art. 36 BV cited everywhere
    # would pollute court_statute). 5000 = ~0.2% of 2.5M corpus.
    "co_citation_max_neighbour_count": 50000,  # v7.5: allow more common; specificity downranks noise

    # --- Concept matching --------------------------------------------------
    "concept_substring_top_k": 6,

    # --- BM25 --------------------------------------------------------------
    "bm25_max_query_terms": 60,    # cap to stop token-explosion from enriched query
    "bm25_min_token_len":   3,

    # --- Vector ------------------------------------------------------------
    "vector_emb_model": "Qwen/Qwen3-Embedding-8B",
    "vector_topk":      800,

    # --- Query expansion ---------------------------------------------------
    "qwen_query_model":    "Qwen/Qwen3-32B",
    "qwen_max_new_tokens": 1024,

    # --- enhance() — corpus-derived BM25 lexicon expansion -----------------
    "enhance_top_k_codes":   5,
    "enhance_repeat_count":  5,
    "enhance_min_idf":       1.0,

    # --- Negative gate -----------------------------------------------------
    "noise_paragraph_roles": {"notification", "header", "empty", "metadata"},

    "lowercase_concepts": True,
    "lowercase_terms":    True,
}

print(json.dumps({k: v for k, v in CONFIG.items() if not isinstance(v, set)}, indent=2, default=str))

{
  "topk_final": 50000,
  "budget_law_direct": null,
  "budget_court_statute": 8000,
  "budget_concept": 3000,
  "budget_term": 2500,
  "budget_per_area": 1500,
  "budget_co_citation": 2500,
  "budget_bm25": 2000,
  "budget_vector": 2000,
  "budget_vector_enriched": 2000,
  "budget_vector_hyde": 2000,
  "budget_backprop": 2000,
  "budget_sibling": 5000,
  "budget_graph_forward": 5000,
  "budget_graph_reverse": 3000,
  "enable_graph_2hop": false,
  "budget_graph_2hop": 1500,
  "rrf_k": 60,
  "guarantee_channels": [
    "law_direct_match",
    "per_area_bedrock",
    "statute_backprop",
    "concept_en",
    "graph_forward",
    "sibling_expansion",
    "term_orig"
  ],
  "guarantee_per_channel": 130,
  "channel_weights": {
    "statute_backprop": 2.5,
    "graph_forward": 2.0,
    "concept_en": 1.8,
    "court_statute": 1.5,
    "per_area_bedrock": 1.5,
    "vector_raw": 1.0,
    "vector_enriched": 1.0,
    "vector_hyde": 1.5,
    "term_orig": 1.2,
    "law_direct_match": 1.2,
    "sib

## 1.5 Load all queries from val.csv

No per-query branching: every row in `val.csv` is processed by the same code path.


In [ ]:
import pandas as pd

val_df = pd.read_csv(PATHS["val_csv"])
print(f"val.csv has {len(val_df)} queries\n")

ALL_QUERIES = []
for _, row in val_df.iterrows():
    qid = str(row["query_id"])
    qtext = str(row["query"])
    gold = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
    ALL_QUERIES.append({"query_id": qid, "query_text": qtext, "gold": gold})

for q in ALL_QUERIES:
    head = q["query_text"][:90] + ("..." if len(q["query_text"]) > 90 else "")
    print(f"  {q['query_id']:<8}  gold={len(q['gold']):>2}   {head}")


val.csv has 10 queries

  val_001   gold=42   May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 A...
  val_002   gold=36   A claimant holding a national vocational diploma in warehouse operations worked intermitte...
  val_003   gold=47   A. Rivera, a Peruvian national born in 1994 and with no prior convictions in the forum sta...
  val_004   gold=10   Mr. Dalton, born in 1941 and resident in a small lakeside town near Thun, executed a handw...
  val_005   gold=11   A parent, separated from their co-parent since 2008, has not had custody of the two childr...
  val_006   gold=18   On 3 March 2012, homeowners Ms. L and her partner Mr. M asked G, an installer they knew so...
  val_007   gold=19   An heirship claims title to a vintage pocket chronometer known as “The Meridian” that belo...
  val_008   gold=29   Has a member of the town council of the Borough of L., who chaired the board of a publicly...
  val_009   gold=14   A divorced custodial paren

# Phase 2 — Build all corpus indexes (one pass)

## 2.1 What this big cell does

This cell is the heart of the retrieval pipeline. It **streams both JSONL files**
(`law_llm_descriptors` + `court_authority_cards_v5`) and builds **all** in-memory
indexes the channels need:

| Index | Type | What it maps |
|---|---|---|
| `cit_to_doc_ids[citation]` | dict[str, list[str]] | citation string → list of doc_ids |
| `doc_meta[did]` | dict[str, dict] | doc_id → {citation, family, court_base, paragraph_role} |
| `idx_law_direct[canonical]` | dict[str, set] | "100 BGG" → law doc_ids |
| `idx_court_statute[canonical]` | dict[str, set] | "100 BGG" → court rows whose anchors include this |
| `idx_court_base[base]` | dict[str, set] | "137 IV 122" → all Es of judgment (used by sibling_expansion) |
| `idx_concept_en[token]` | dict[str, set] | English concept → doc_ids |
| `idx_term_orig[token]` | dict[str, set] | DE/FR/IT term → doc_ids |
| `search_text[did]` | dict[str, str] | doc_id → BM25-search text (concatenated enrichment) |
| `legal_area_per_doc[did]` | dict[str, str] | court doc → its `legal_area_static` |
| `co_citation_pairs` | Counter[(canon_a, canon_b)] | unordered pairs of statutes co-cited within same court row |
| `tlf[token][code]` | dict[str, Counter] | corpus-wide token → law-code association (for `enhance()`) |
| `doc_statute_anchors[did]` | dict[str, set] | court doc_id → set of canonical statutes it cites |

## 2.2 Why we need each one (channel attribution)

- `idx_law_direct` → channels: `law_direct_match`, `per_area_bedrock`, `statute_backprop`, `co_citation`
- `idx_court_statute` → channels: `court_statute`, `co_citation`
- `idx_court_base` → channel: `sibling_expansion` (own-judgment Es)
- `idx_concept_en` / `idx_term_orig` → channels: `concept_en`, `term_orig`
- `search_text` → channel: `bm25` (FTS5 indexed in Phase 5)
- `co_citation_pairs` → Phase 4 builds `co_neighbours` from this; channel: `co_citation`
- `tlf` → BM25 query enhancement in Phase 5
- `doc_statute_anchors` → channel: `statute_backprop`

## 2.3 Expected outputs
- Law: ~175k rows in ~10 s
- Court: ~2.47M rows in ~150 s
- Token→code association: ~90k tokens, avg ~10 codes/token
- Co-citation pairs: ~1.1M

## 2.4 Failure modes
- **Slower than 200 s for court** → Drive throttling; retry.
- **`Total docs ≠ ~2.65M`** → JSONL truncation; check file size matches local.
- **`tlf` very small (< 50k tokens)** → law tokenizer pattern wrong; check the
  regex split below.

In [ ]:
from collections import defaultdict, Counter
import re, json, time

# --- Statute / case canonicalizers --------------------------------------------
CODE_ALIAS = {
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    "STPO": "StPO", "OBG": "OR",
}
ART_RE = re.compile(r"art\.?\s*(\d+[a-z]?)", re.I)
CODE_RE = re.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")

def statute_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = ART_RE.search(s)
    if not m: return None
    cands = [c.strip(".") for c in CODE_RE.findall(s)
             if c.strip(".") not in ("Art","Abs","Ziff","lit","let","al","Bst")]
    if not cands: return None
    code = CODE_ALIAS.get(cands[-1], cands[-1])
    return f"{m.group(1)} {code}"

def article_num(raw):
    if not raw: return None
    m = ART_RE.search(raw.strip())
    return m.group(1) if m else None

LEGAL_AREA_DEFAULT_CODE = {
    "criminal law and criminal procedure": "StPO",
    "criminal procedure":                  "StPO",
    "criminal law":                        "StGB",
    "civil law":                           "ZGB",
    "obligations":                         "OR",
    "civil procedure":                     "ZPO",
    "constitutional and public law":       "BV",
    "constitutional law":                  "BV",
    "administrative law":                  "VwVG",
    "social insurance":                    "ATSG",
    "tax law":                             "DBG",
}

def canonicalize_row_anchors(raw_anchors, legal_area_static):
    canons = set()
    primary_code = None
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            primary_code = c.split()[1]; break
    fallback = primary_code
    if fallback is None and legal_area_static:
        la = legal_area_static.lower()
        for k, v in LEGAL_AREA_DEFAULT_CODE.items():
            if k in la:
                fallback = v; break
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            canons.add(c); continue
        n = article_num(sa)
        if n and fallback:
            canons.add(f"{n} {fallback}")
    return canons

CASE_BGE_RE    = re.compile(r"BGE\s+(\d+)\s+([IVX]+)\s+(\d+)")
CASE_DOCKET_RE = re.compile(r"\b(\d[A-Z]_\d+/\d{4})\b")

def case_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = CASE_BGE_RE.search(s)
    if m: return f"BGE {m.group(1)} {m.group(2)} {m.group(3)}"
    m = CASE_DOCKET_RE.search(s)
    if m: return m.group(1)
    return None

TOKEN_NORM_RE = re.compile(r"\s+")
def norm_token(s, lower):
    if not s: return None
    s = TOKEN_NORM_RE.sub(" ", s.strip())
    if not s: return None
    return s.lower() if lower else s

# --- German lemmatizer for term_orig channel ---------------------------------
# Drop one common inflectional suffix at a time. Only drop a suffix if the
# remaining stem is >= 4 chars; allow recursion (e.g. 'haftens' -> 'haften'
# -> 'haft'). Surface form is always kept in idx_term_orig as well, so this
# is purely additive.
_TERM_LEMMA_SUFFIXES = ("en", "es", "em", "er", "e", "n", "s")
def term_lemma(tok):
    if not tok: return tok
    cur = tok
    seen = {cur}
    while True:
        changed = False
        for suf in _TERM_LEMMA_SUFFIXES:
            if cur.endswith(suf) and len(cur) - len(suf) >= 4:
                stem = cur[: len(cur) - len(suf)]
                if stem not in seen:
                    cur = stem; seen.add(cur); changed = True; break
        if not changed:
            break
    return cur

# --- Indexes -----------------------------------------------------------------
cit_to_doc_ids       = defaultdict(list)
doc_meta             = {}
idx_law_direct       = defaultdict(set)
idx_court_statute    = defaultdict(set)
idx_case_anchor      = defaultdict(set)
idx_court_base       = defaultdict(set)
idx_concept_en       = defaultdict(set)
idx_term_orig        = defaultdict(set)
idx_term_lemma       = defaultdict(set)   # lemma-form -> set(doc_ids)
term_orig_keys       = set()              # all normalized surface-form keys (for substring scan)
legal_area_per_doc   = {}
search_text          = {}
co_citation_pairs    = Counter()
tlf                  = defaultdict(Counter)
token_doc_count      = Counter()
doc_statute_anchors  = {}
doc_language         = {}

DOC_ID_LAW   = lambda i: f"law:{i}"
DOC_ID_COURT = lambda i: f"court:{i}"

def _take_text(*parts, max_chars=2000):
    out = []
    for p in parts:
        if not p: continue
        if isinstance(p, list):
            for x in p:
                if isinstance(x, str): out.append(x)
                elif isinstance(x, dict):
                    for v in x.values():
                        if isinstance(v, str): out.append(v)
        elif isinstance(p, str):
            out.append(p)
    return (" ".join(out))[:max_chars]

# --- Stream law jsonl --------------------------------------------------------
t0 = time.time(); n_law = 0
with open(PATHS["law_llm"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_LAW(n_law)
        cit_to_doc_ids[cit].append(did)
        doc_meta[did] = {"citation": cit, "family": "law", "court_base": None,
                         "paragraph_role": None, "is_notification_paragraph": False}
        canon = statute_anchor_canonical(cit)
        if canon: idx_law_direct[canon].add(did)

        enr = obj.get("llm_enrichment") or {}
        terms_de = []; terms_en = []
        for t in enr.get("terms_de_to_en") or []:
            if isinstance(t, dict):
                de = norm_token(t.get("de",""), CONFIG["lowercase_terms"])
                en = norm_token(t.get("en",""), CONFIG["lowercase_terms"])
                if de:
                    idx_term_orig[de].add(did); terms_de.append(de)
                    term_orig_keys.add(de)
                    _lem_de = term_lemma(de)
                    if _lem_de and _lem_de != de: idx_term_lemma[_lem_de].add(did)
                    idx_term_lemma[de].add(did)
                if en:
                    idx_concept_en[en].add(did); terms_en.append(en)
        for c in enr.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        search_text[did] = _take_text(
            cit, enr.get("english_summary",""), enr.get("legal_rule",""),
            enr.get("legal_question",""), enr.get("applicability_conditions"),
            enr.get("concepts_en"), terms_de, terms_en,
        )
        legal_area_per_doc[did] = "law"
        doc_language[did] = (obj.get("language") or "de").lower()

        # token -> code association (only law rows have a clean canonical code).
        if canon and " " in canon:
            row_code = canon.split()[1].lower()
            row_text = search_text[did].lower()
            row_tokens = set()
            for tok in re.split(r"[^\w\d]+", row_text, flags=re.UNICODE):
                if len(tok) >= 3:
                    row_tokens.add(tok)
            for tok in row_tokens:
                tlf[tok][row_code] += 1
                token_doc_count[tok] += 1

        n_law += 1

print(f"Law: {n_law:,} rows indexed in {time.time()-t0:.1f}s")
print(f"Token->code association: {len(tlf):,} tokens, "
      f"avg codes/token = {sum(len(c) for c in tlf.values())/max(1,len(tlf)):.1f}")

# --- Stream court jsonl ------------------------------------------------------
t1 = time.time(); n_court = 0
with open(PATHS["court_v5"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_COURT(n_court)
        cit_to_doc_ids[cit].append(did)
        cb  = obj.get("court_base") or ""
        rag = obj.get("rag_enrichment") or {}
        legal_area_static = obj.get("legal_area_static") or rag.get("legal_area") or ""
        doc_meta[did] = {"citation": cit, "family": "court", "court_base": cb,
                         "paragraph_role": rag.get("paragraph_role"),
                         "is_notification_paragraph": bool(obj.get("is_notification_paragraph"))}
        legal_area_per_doc[did] = (legal_area_static or "").lower()
        _row_lang_raw = (obj.get("language") or "").lower()
        doc_language[did] = _row_lang_raw if _row_lang_raw in ("de", "fr", "it", "en") else "en"

        if cb:
            idx_court_base[cb].add(did)
            cb_canon = case_anchor_canonical(cb)
            if cb_canon: idx_case_anchor[cb_canon].add(did)

        row_canons = canonicalize_row_anchors(rag.get("statute_anchors") or [], legal_area_static)
        for canon in row_canons:
            idx_court_statute[canon].add(did)
        if row_canons:
            doc_statute_anchors[did] = row_canons
        rc = sorted(row_canons)
        for i in range(len(rc)):
            for j in range(i+1, len(rc)):
                co_citation_pairs[(rc[i], rc[j])] += 1

        for ca in rag.get("case_anchors") or []:
            canon = case_anchor_canonical(ca)
            if canon: idx_case_anchor[canon].add(did)
        for c in rag.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        for t in rag.get("terms_original") or []:
            tok = norm_token(t, CONFIG["lowercase_terms"])
            if tok:
                idx_term_orig[tok].add(did)
                term_orig_keys.add(tok)
                _lem = term_lemma(tok)
                if _lem and _lem != tok: idx_term_lemma[_lem].add(did)
                idx_term_lemma[tok].add(did)

        search_text[did] = _take_text(
            cit, obj.get("text_excerpt_original",""),
            rag.get("concepts_en"), rag.get("terms_original"),
            rag.get("micro_topic",""), rag.get("topic",""), rag.get("subtopic",""),
            rag.get("statute_anchors"),
        )

        n_court += 1
        if n_court % 500_000 == 0:
            print(f"  court progress: {n_court:,} rows ({time.time()-t1:.1f}s)")

print(f"Court: {n_court:,} rows indexed in {time.time()-t1:.1f}s")
# Pre-compute per-canon document counts for specificity weighting in channel_court_statute.
idx_court_statute_count = {canon: len(s) for canon, s in idx_court_statute.items()}
print(f"Total docs:        {len(doc_meta):,}")
print(f"Unique citations:  {len(cit_to_doc_ids):,}")
print(f"Index sizes:       law_direct={len(idx_law_direct):,}, court_statute={len(idx_court_statute):,}, "
      f"case={len(idx_case_anchor):,}, court_base={len(idx_court_base):,}, "
      f"concept={len(idx_concept_en):,}, term={len(idx_term_orig):,}, "
      f"term_lemma={len(idx_term_lemma):,}, term_keys={len(term_orig_keys):,}")
print(f"Co-citation pairs: {len(co_citation_pairs):,}")

Law: 173,033 rows indexed in 12.2s
Token->code association: 91,173 tokens, avg codes/token = 11.7
  court progress: 500,000 rows (33.5s)
  court progress: 1,000,000 rows (70.1s)
  court progress: 1,500,000 rows (104.3s)
  court progress: 2,000,000 rows (142.3s)
Court: 2,476,315 rows indexed in 178.3s
Total docs:        2,649,348
Unique citations:  2,158,211
Index sizes:       law_direct=49,288, court_statute=81,988, case=157,227, court_base=178,593, concept=292,946, term=363,331, term_lemma=521,146, term_keys=363,331
Co-citation pairs: 1,142,790


## 2.5 Map gold citations to doc_ids (per query)

For each query we resolve its gold citations against the corpus. This is the upper bound on R@K — anything not mapped here cannot be retrieved by any channel.


In [ ]:
ALL_GOLD_DOC_SET = {}
ALL_TOTAL_GOLD = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    gset = set()
    unmapped = []
    for g in q["gold"]:
        dids = cit_to_doc_ids.get(g, [])
        if not dids:
            unmapped.append(g)
        else:
            gset.update(dids)
    ALL_GOLD_DOC_SET[qid] = gset
    ALL_TOTAL_GOLD[qid] = len(q["gold"])
    warn = f"  [WARN unmapped={len(unmapped)}]" if unmapped else ""
    print(f"  {qid:<8}  gold={len(q['gold']):>2}  mapped={len(q['gold']) - len(unmapped):>2}  doc_ids={len(gset):>4}{warn}")


  val_001   gold=42  mapped=42  doc_ids=  42
  val_002   gold=36  mapped=36  doc_ids=  38
  val_003   gold=47  mapped=47  doc_ids=  47
  val_004   gold=10  mapped=10  doc_ids=  10
  val_005   gold=11  mapped=11  doc_ids=  11
  val_006   gold=18  mapped=18  doc_ids=  18
  val_007   gold=19  mapped=19  doc_ids=  19
  val_008   gold=29  mapped=29  doc_ids=  30
  val_009   gold=14  mapped=14  doc_ids=  14
  val_010   gold=25  mapped=25  doc_ids=  25


# Phase 3 — Citation graph (v7 NEW)

## 3.1 What this cell does

Loads `data_insights/citation_graph_extracted.sqlite` (built locally via 4
alias passes; ~2.4 GB; 23.65 M edges) and converts edge tuples (citation_str
→ citation_str) into doc_id-keyed in-memory dicts:
- `idx_graph_out[did]` → list of doc_ids the row text-cites or fans-out to
- `idx_graph_in[did]` → list of doc_ids whose text cites this row

## 3.2 Why we need this — the 4 layers

The graph encodes signal that's **invisible to BM25, vector, and concepts**:

1. **Intra-judgment back-references**
   - Court text uses bare back-refs like `(vgl. E. 6.2 hiervor)` — references
     to siblings of the same judgment. Original `extract_citation_graph.py`
     missed all of these (its CONSIDERATION_RE only fires after a docket).
   - Pass 1 (`extract_intra_judgment_backrefs.py`) handles 4 patterns × 3
     languages: `vgl./siehe E. N`, `E. N hiervor`, `E. N ci-dessus`, `cf. supra
     consid. N`, including `siehe oben E. N`, `hiervor E. N`, `vorstehend`.
   - Self-tested with 15 real-world cases; 4-layer audit on 500 marker-rows
     dropped zero-target rate from 335→175 (90% of remaining are TRUE false
     markers like `nach oben` = "to the top").

2. **Date-stripped aliases**
   - Corpus extraction stores dated form `1B_210/2023 12.05.2023 E. 3`, but
     val gold uses un-dated form `1B_210/2023 E. 3`.
   - Pass 2 adds alias edges from dated → un-dated forms (only when the
     un-dated form exists as a real corpus row).

3. **E.-range expansion**
   - Corpus stores ranges as one citation: `1B_90/2021 E. 2.1-2.4`. Gold uses
     individual Es: `E. 2.1`, `E. 2.2`, `E. 2.3`, `E. 2.4`.
   - Pass 3 enumerates ranges and adds aliases.

4. **Case-level fan-out**
   - When a source cites one E. of a judgment, gold may include OTHER Es of
     the same judgment that nobody text-cites by exact pinpoint. Treats
     "citing one E." as "this case is relevant".
   - Pass 4 fans out: edge → BASE E. X spawns alias edges to all `BASE E. Y`
     where Y exists as a real corpus row.
   - Largest pass: +18.77 M edges.

After all 4 passes: **0/42 val_001 gold orphan** (was 11 originally).

## 3.3 Why doc_id mapping skips synthetic targets

Some graph nodes are synthetic (e.g., a backref `E. 4 hiervor` resolves to
`{base} E. 4` which may not be a real corpus row). Such targets have no
`cit_to_doc_ids` entry and are skipped here — graph channels only retrieve
real corpus rows.

## 3.4 Expected outputs
- ~24 M edges loaded (some skipped because synthetic targets have no doc_id)
- Out-degree avg ~10–15 (reflects case-level fan-out per cited judgment)
- In-degree avg ~10–15
- Load time: 30–60 s

## 3.5 Failure modes
- `graph_db MISSING` → graph channels skipped; sibling_expansion still works.
- `0 edges loaded` → all citations failed to map; check `cit_to_doc_ids`
  was built before this cell, and citations strings are exact (case, spacing).
- `Out-degree avg < 3` → mapping mostly failed; check date format normalization.

In [ ]:
import sqlite3 as _sqlite3

idx_graph_out = defaultdict(list)
idx_graph_in  = defaultdict(list)
GRAPH_OK = bool(GRAPH_AVAILABLE)

if GRAPH_OK:
    _t = time.time()
    _cit_to_did = {cit: dids[0] for cit, dids in cit_to_doc_ids.items() if dids}
    print(f"Built citation->doc_id map ({len(_cit_to_did):,} entries)")

    _g = _sqlite3.connect(str(PATHS["graph_db"]))
    n_loaded = 0; n_skipped = 0
    for _src, _tgt in _g.execute(
        "SELECT source, target FROM edges WHERE dataset='court_considerations'"
    ):
        _s = _cit_to_did.get(_src)
        _t2 = _cit_to_did.get(_tgt)
        if _s is None or _t2 is None:
            n_skipped += 1; continue
        if _s == _t2: continue
        idx_graph_out[_s].append(_t2)
        idx_graph_in[_t2].append(_s)
        n_loaded += 1
    _g.close()
    print(f"Graph: {n_loaded:,} edges loaded, {n_skipped:,} skipped (cit not in corpus)")
    print(f"Graph: out-degree avg = {n_loaded/max(1,len(idx_graph_out)):.1f}, "
          f"in-degree avg = {n_loaded/max(1,len(idx_graph_in)):.1f}")
    print(f"Graph: load time {time.time()-_t:.1f}s")
else:
    print("[skip] Graph DB missing — graph channels will return empty lists.")

# --- Judgment-importance index (v7.4) -------------------------------------
# importance(court_base) = sum over did in idx_court_base[court_base] of
#                         len(idx_graph_in[did])
# i.e. total incoming citations across every E.-paragraph row of the
# judgment. Landmark BGE cases score thousands; obscure dockets score 0-50.
# Used by sibling_expansion / graph_forward / graph_reverse channels in
# cell 32 to break score=1 ties and prioritize widely-cited judgments.
idx_judgment_importance = {}
if GRAPH_OK:
    _ti = time.time()
    for _cb, _dids in idx_court_base.items():
        _s = 0
        for _d in _dids:
            _s += len(idx_graph_in.get(_d, ()))
        idx_judgment_importance[_cb] = _s
    _imp_vals = list(idx_judgment_importance.values())
    if _imp_vals:
        _imp_vals_sorted = sorted(_imp_vals)
        _n = len(_imp_vals_sorted)
        print(f"Judgment importance: {len(idx_judgment_importance):,} judgments, "
              f"median={_imp_vals_sorted[_n//2]}, "
              f"p75={_imp_vals_sorted[3*_n//4]}, "
              f"max={_imp_vals_sorted[-1]}")
    print(f"Judgment importance: built in {time.time()-_ti:.1f}s")
else:
    print("[skip] idx_judgment_importance empty — graph not loaded.")

Built citation->doc_id map (2,158,211 entries)
Graph: 20,494,436 edges loaded, 3,155,263 skipped (cit not in corpus)
Graph: out-degree avg = 27.2, in-degree avg = 14.0
Graph: load time 103.2s
Judgment importance: 178,593 judgments, median=7, p75=44, max=204686
Judgment importance: built in 1.1s


# Phase 4 — Per-area bedrock + co-citation neighbours

## 4.1 Per-area bedrock — why we need it

For a query in legal area "criminal procedure", certain articles are
**universally cited** by every BGer detention decision (Art. 100 BGG,
Art. 42 BGG, Art. 66 BGG — the procedural/cost cluster). The LLM rarely
names these because they're "implicit" to lawyers but they're often gold.

Per-area bedrock is **corpus-derived**: from `legal_area_per_doc`, count
which canonical statutes appear most often in court rows of each area.

**Filter:** v6 added a critical fix — restrict per-area bedrock to canonicals
whose code matches one of the LLM-named codes (e.g., StPO + BGG for val_001).
v4 returned BGG-dominated lists across ALL areas because BGG appeal articles
are cited everywhere.

**Expected:** ~26 distinct legal areas; `criminal procedure and coercive measures`
top-8 should include 66 BGG, 78 BGG, 81 BGG.

**Failure modes:**
- `0 areas` → `legal_area_per_doc` is empty; check court enrichment has
  `legal_area_static` set.

In [ ]:
print("Building per-area bedrock index...")
_t = time.time()
per_area_canon_count = defaultdict(Counter)
for did, area in legal_area_per_doc.items():
    if not area or area == "law": continue
    canons = doc_statute_anchors.get(did, ())
    for canon in canons:
        per_area_canon_count[area][canon] += 1
print(f"  built in {time.time()-_t:.1f}s; areas: {len(per_area_canon_count)}")
for area in list(per_area_canon_count.keys())[:4]:
    print(f"  area={area!r}: top 8 = {per_area_canon_count[area].most_common(8)}")

# v7.5 NEW: derive code-family from corpus co-citation. For each pair of
# canons (a, b) co-cited in a court row, record the (code_a, code_b) pair
# weight. Used at retrieval time to expand statute_target_codes from
# LLM-named codes to corpus-related codes (no hardcoded statute cluster).
print("Building corpus-derived code-pair statistics...")
_tc = time.time()
code_pair_count = Counter()
for (a, b), n in co_citation_pairs.items():
    ca = a.split()[1] if " " in a else None
    cb = b.split()[1] if " " in b else None
    if ca and cb and ca != cb:
        code_pair_count[(ca, cb)] += n
        code_pair_count[(cb, ca)] += n   # symmetric
print(f"  code-pair statistics: {len(code_pair_count):,} directed pairs ({time.time()-_tc:.1f}s)")
print(f"  StPO's top related codes: {[(c, n) for (a, c), n in code_pair_count.most_common(2000) if a=='StPO'][:8]}")


Building per-area bedrock index...
  built in 2.1s; areas: 26
  area='constitutional and public law': top 8 = [('66 BGG', 13212), ('29 BV', 12189), ('89 BGG', 11479), ('82 BGG', 11464), ('42 BGG', 10836), ('9 BV', 9650), ('106 BGG', 9625), ('68 BGG', 8649)]
  area='administrative, tax, migration, and regulatory law': top 8 = [('42 BGG', 20301), ('106 BGG', 19460), ('66 BGG', 17471), ('105 BGG', 16033), ('95 BGG', 15722), ('83 BGG', 15231), ('68 BGG', 14674), ('89 BGG', 12187)]
  area='civil law': top 8 = [('63 OJ', 3189), ('8 ZGB', 2740), ('55 OJ', 2518), ('64 OJ', 2235), ('55 OG', 2085), ('63 OG', 1906), ('159 OG', 1876), ('9 BV', 1875)]
  area='criminal law and criminal procedure': top 8 = [('66 BGG', 28149), ('42 BGG', 20746), ('106 BGG', 19537), ('64 BGG', 13843), ('108 BGG', 12101), ('97 BGG', 12075), ('105 BGG', 11759), ('81 BGG', 11102)]
Building corpus-derived code-pair statistics...
  code-pair statistics: 54,790 directed pairs (0.7s)
  StPO's top related codes: [('BGG', 50026

## 4.2 Co-citation neighbours — why we need it

For each statute target the LLM names (e.g., `Art. 221 StPO`), find the top-K
canonical statutes that are **cited together** in the same court rows most
often. Surfaces statute clusters that move together in legal practice.

**Filter:** drop neighbours that are TOO globally common (Art. 36 BV cited
in nearly every criminal case). 5000 = ~0.2% of 2.5 M corpus.

**Expected:** for `221 StPO` neighbours: `212 StPO`, `237 StPO`, `5 StPO`,
`5 EMRK` (the detention statute cluster). NOT `36 BV` (filtered out).

**Failure modes:**
- All-empty neighbours → `co_citation_pairs` was empty; check court enrichment
  contained statute_anchors.

In [ ]:
co_neighbours = defaultdict(list)
canon_count = Counter()
for did, canons in doc_statute_anchors.items():
    for c in canons: canon_count[c] += 1

for (a, b), cnt in co_citation_pairs.items():
    if cnt < CONFIG["co_citation_min_co_count"]: continue
    if canon_count[b] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter b
    if canon_count[a] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter a
    co_neighbours[a].append((b, cnt))
    co_neighbours[b].append((a, cnt))

# Apply frequency filter on neighbour side and keep top-K per source
co_neighbours = {
    src: sorted(
        ((nb, n) for nb, n in nbs if canon_count[nb] <= CONFIG["co_citation_max_neighbour_count"]),
        key=lambda x: -x[1]
    )[:CONFIG["co_citation_top_k_per_target"] * 2]
    for src, nbs in co_neighbours.items()
}
print(f"Co-citation neighbours indexed for {len(co_neighbours):,} canonicals "
      f"(after frequency filter: max global count = {CONFIG['co_citation_max_neighbour_count']}).")
print("Sample - neighbours of '221 StPO' AFTER frequency filter:")
for nb, n in co_neighbours.get("221 StPO", [])[:8]:
    print(f"  {nb}: co={n}, total_in_corpus={canon_count[nb]}")

Co-citation neighbours indexed for 2,282 canonicals (after frequency filter: max global count = 50000).
Sample - neighbours of '221 StPO' AFTER frequency filter:
  36 BV: co=915, total_in_corpus=7216
  31 BV: co=783, total_in_corpus=4339
  10 BV: co=731, total_in_corpus=5241
  212 StPO: co=714, total_in_corpus=1337
  221 BV: co=646, total_in_corpus=670
  237 StPO: co=487, total_in_corpus=1467
  5 BV: co=393, total_in_corpus=9685
  5 StPO: co=272, total_in_corpus=1955


# Phase 5 — BM25 (FTS5 in-memory)

## 5.1 What this cell does

Build SQLite FTS5 over `search_text[did]` for all 2.65 M docs. Provides
`bm25_search(query_text, k)` which returns top-k doc_ids ranked by BM25.

## 5.2 Why FTS5 (and not Whoosh / Lucene)

- Pure stdlib; no extra install.
- ~80 s build for 2.6 M short docs.
- Returns BM25-scored top-K in <50 ms.
- We can pass any expanded query text and get a stable ranking.

## 5.3 enhance() — corpus-derived query enrichment

Untitled75's reference notebook used a `enhance()` from train data that we
forbid. We replicate the IDEA (boost query with code names most associated
with query tokens) but train it on the **corpus** (laws_de) instead of train.

For each token in query, look up `tlf[token]` (a Counter mapping legal codes
to row counts). The top-K codes with highest score get appended to the query
multiple times, biasing BM25 toward law rows of those codes.

Example: query contains "detention" → boost `stpo` (high) more than `or`.

**Expected:** FTS5 build ~80 s. enhance() boost typically adds 5×5=25 token
repetitions to the query.

**Failure modes:**
- Slow build (>180 s) → swap MEMORY journal for OFF, or use a temp file.

In [ ]:
import sqlite3, math

# -----------------------------------------------------------------------------
# Phase 5.A — legacy single-language FTS (kept for back-compat / debugging)
# -----------------------------------------------------------------------------
print(f"Building in-memory FTS5 (legacy, single index) over {len(search_text):,} docs...")
_t = time.time()
_fts = sqlite3.connect(":memory:")
_fts.execute("PRAGMA journal_mode = MEMORY")
_fts.execute("PRAGMA synchronous = OFF")
_fts.execute("CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, tokenize = 'unicode61 remove_diacritics 2')")
_inserted = 0
_batch = []
for did, txt in search_text.items():
    _batch.append((did, txt))
    if len(_batch) >= 50000:
        _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
        _inserted += len(_batch); _batch.clear()
        if _inserted % 500000 == 0:
            print(f"  inserted {_inserted:,} ({time.time()-_t:.1f}s)")
if _batch:
    _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
    _inserted += len(_batch)
_fts.commit()
print(f"FTS5 (legacy) built: {_inserted:,} rows in {time.time()-_t:.1f}s")


# -----------------------------------------------------------------------------
# Phase 5.B — per-language FTS5 indices
# -----------------------------------------------------------------------------
# Build one in-memory FTS5 per language. doc_language[did] was populated in
# cell 12 from the top-level `language` field (de/fr/it; anything else,
# including 'unknown' and missing, was normalised to 'en').
print("Building per-language FTS5 indices (de/fr/it/en)...")
_t_ml = time.time()
_LANGS = ("de", "fr", "it", "en")
_fts_by_lang = {}
_lang_doc_count = {}
for _L in _LANGS:
    _conn = sqlite3.connect(":memory:")
    _conn.execute("PRAGMA journal_mode = MEMORY")
    _conn.execute("PRAGMA synchronous = OFF")
    _conn.execute(
        "CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, "
        "tokenize = 'unicode61 remove_diacritics 2')"
    )
    _fts_by_lang[_L] = _conn
    _lang_doc_count[_L] = 0

# Group inserts by language. Stream search_text once, route per doc_language.
_batches = {L: [] for L in _LANGS}
_unknown_lang_did = 0
for did, txt in search_text.items():
    L = doc_language.get(did, "en")
    if L not in _fts_by_lang:
        # safety net for any unexpected value (shouldn't happen after cell 12)
        L = "en"
        _unknown_lang_did += 1
    _batches[L].append((did, txt))
    _lang_doc_count[L] += 1
    if len(_batches[L]) >= 50000:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()

for L in _LANGS:
    if _batches[L]:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()
    _fts_by_lang[L].commit()

print(f"Per-language FTS5 built in {time.time()-_t_ml:.1f}s")
for _L in _LANGS:
    print(f"  fts[{_L}]: {_lang_doc_count[_L]:,} docs")
if _unknown_lang_did:
    print(f"  (note) {_unknown_lang_did:,} docs had no language tag and were "
          f"routed to 'en'")


# -----------------------------------------------------------------------------
# Phase 5.C — enhance() (corpus-derived BM25 lexicon expansion); unchanged.
# -----------------------------------------------------------------------------
def _enhance_codes(text):
    text_lc = text.lower()
    tokens = set()
    for tok in re.split(r"[^\w\d]+", text_lc, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            tokens.add(tok)
    code_score = Counter()
    for tok in tokens:
        if tok not in tlf: continue
        n_docs = max(1, token_doc_count[tok])
        idf = math.log(1 + (max(1, len(search_text)) / n_docs))
        if idf < CONFIG["enhance_min_idf"]: continue
        for code, cnt in tlf[tok].most_common():
            code_score[code] += cnt * idf
    return [c for c, _ in code_score.most_common(CONFIG["enhance_top_k_codes"])]


# -----------------------------------------------------------------------------
# Phase 5.D — legacy bm25_search() (unchanged behaviour, single-index).
# -----------------------------------------------------------------------------
def bm25_search(query_text, k):
    boosted = _enhance_codes(query_text)
    enriched = query_text + " " + " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)
    fts_q = []
    for tok in re.split(r"[^\w\d]+", enriched, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]: break
    if not fts_q: return []
    fts_query = " OR ".join(f'"{t}"' for t in fts_q)
    rows = _fts.execute(
        "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? ORDER BY bm25(docs) LIMIT ?",
        (fts_query, k),
    ).fetchall()
    return [(did, -score) for did, score in rows]


# -----------------------------------------------------------------------------
# Phase 5.E — NEW multi-language BM25 search.
# -----------------------------------------------------------------------------
# For each language L, builds a language-specific query string from:
#   - the original English query (always present; English query tokens still
#     match against English concepts_en mixed into court search_text rows)
#   - language-appropriate enrichment terms from the LLM-produced `targets`
# Runs FTS5 on that index, takes its top-k_per_lang. Merges by `did` taking
# max-score across languages, then returns the global top k_total.
#
# Score normalisation:
#   FTS5 bm25() returns a NEGATIVE score (more negative = more relevant). We
#   flip sign first (bigger = more relevant). Then divide by
#   log(corpus_size_for_lang + math.e) so a tiny language (~94k IT rows) and
#   a huge one (~1.4M DE rows) produce comparable score magnitudes.
def _build_lang_query(english_query, targets, lang):
    """Build the FTS5 query string for a given language."""
    parts = [english_query or ""]
    targets = targets or {}
    concept_en = list(targets.get("concept_targets_en") or [])
    term_de = list(targets.get("term_targets_de") or [])
    term_fr = list(targets.get("term_targets_fr") or [])
    if lang == "de":
        parts.extend(term_de)
        parts.extend(concept_en)
    elif lang == "fr":
        parts.extend(term_fr)
        parts.extend(concept_en)
    elif lang == "it":
        # No it-specific targets in the qexp schema; fall back to de+fr+en.
        parts.extend(term_de)
        parts.extend(term_fr)
        parts.extend(concept_en)
    else:  # "en" and any unexpected language
        parts.extend(concept_en)
    return " ".join(p for p in parts if p)


def _split_budget(k_total, lang_doc_count):
    """Allocate per-language budget proportional to docs in that language,
    with a small floor so tiny languages still contribute. Returns dict."""
    total_docs = sum(max(1, lang_doc_count[L]) for L in _LANGS)
    floor = max(1, k_total // 16)  # at least ~6% of budget per language
    raw = {L: max(floor, int(round(k_total * lang_doc_count[L] / total_docs)))
           for L in _LANGS}
    # Trim if floors caused over-allocation; never below the floor though.
    over = sum(raw.values()) - k_total
    if over > 0:
        # subtract from largest first
        for L in sorted(_LANGS, key=lambda x: -raw[x]):
            take = min(over, raw[L] - floor)
            if take <= 0: continue
            raw[L] -= take; over -= take
            if over <= 0: break
    return raw


def bm25_search_multilang(query, targets, k_total):
    """Run BM25 per language, merge by did with max-score, return top k_total.

    Args:
        query    : original English query string.
        targets  : dict from Phase 7 (statute_targets, term_targets_de,
                   term_targets_fr, concept_targets_en, ...).
        k_total  : total budget (CONFIG['budget_bm25']).

    Returns:
        list of (did, score) tuples, sorted by score desc, length <= k_total.
    """
    if not _fts_by_lang:
        return []
    budgets = _split_budget(k_total, _lang_doc_count)
    # enhance() runs on the original English query only (the corpus-derived
    # codes are language-agnostic statute codes like "stpo", "bgg" — these are
    # appended to every language's query).
    boosted = _enhance_codes(query or "")
    boost_str = " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)

    merged = {}  # did -> best normalised score
    for L in _LANGS:
        if _lang_doc_count[L] == 0:
            continue
        per_lang_query = _build_lang_query(query, targets, L)
        if boost_str:
            per_lang_query = per_lang_query + " " + boost_str
        # Tokenise for FTS5: alpha/num tokens >= min_len; cap to budget.
        fts_q = []
        seen = set()
        for tok in re.split(r"[^\w\d]+", per_lang_query, flags=re.UNICODE):
            if len(tok) < CONFIG["bm25_min_token_len"]:
                continue
            tl = tok.lower()
            if tl in seen:
                continue
            seen.add(tl)
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]:
                break
        if not fts_q:
            continue
        fts_query = " OR ".join(f'"{t}"' for t in fts_q)
        # Cross-language comparability: divide raw score by log(N_lang + e).
        denom = math.log(_lang_doc_count[L] + math.e)
        try:
            rows = _fts_by_lang[L].execute(
                "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? "
                "ORDER BY bm25(docs) LIMIT ?",
                (fts_query, budgets[L]),
            ).fetchall()
        except sqlite3.OperationalError as _e:
            # malformed query (e.g. all stop-words) — skip this language.
            print(f"  bm25[{L}] skipped: {_e}")
            continue
        for did, raw_score in rows:
            # FTS5 bm25 is negative (lower = more relevant). Flip sign so
            # bigger = more relevant, then normalise by language size.
            norm = (-raw_score) / denom
            prev = merged.get(did)
            if prev is None or norm > prev:
                merged[did] = norm

    if not merged:
        return []
    out = sorted(merged.items(), key=lambda kv: -kv[1])[:k_total]
    return out


Building in-memory FTS5 (legacy, single index) over 2,649,348 docs...
  inserted 500,000 (11.5s)
  inserted 1,000,000 (27.4s)
  inserted 1,500,000 (44.4s)
  inserted 2,000,000 (61.9s)
  inserted 2,500,000 (78.2s)
FTS5 (legacy) built: 2,649,348 rows in 84.5s
Building per-language FTS5 indices (de/fr/it/en)...
Per-language FTS5 built in 70.7s
  fts[de]: 1,593,249 docs
  fts[fr]: 793,023 docs
  fts[it]: 125,583 docs
  fts[en]: 137,493 docs


# Phase 6 — Vector channel setup

## 6.1 What this cell does

Loads the 27 fp16 embedding chunks (~21 GB total) into a single `E_GPU`
tensor of shape (2.65 M, 4096). Provides `vector_search(q_emb, k)` doing
brute-force matmul on GPU.

## 6.2 Why brute-force GPU and not FAISS-IVF

- Blackwell has 95 GB VRAM; corpus E_GPU at fp16 fits in ~22 GB.
- Matmul `E_GPU @ q` is ~50 ms on GPU; comparable to FAISS-IVF query.
- No quantization recall loss, no index-build time.
- Per `personal_observations.md` Obs 3: dense embedding alone caps at
  R@1000 = 0.289 on val regardless of index — the bottleneck is "wrong kind
  of relationship for cosine similarity", not retrieval algorithm.

## 6.3 Manifest mapping

`qwen3_8b_unified_manifest.parquet` maps doc_id ↔ row_index in E_GPU.

**Expected:** 2.65 M rows in manifest, ~99.9% mapping coverage to our doc_ids.

**Failure modes:**
- `EMB_AVAILABLE = False` → vector channels skipped. Anchor + bedrock + BM25
  still run but recall drops.
- VRAM OOM during chunk concat → torch.cat allocates 2× peak; switch to
  in-place writes if chunk count grows.

In [ ]:
VECTOR_OK = False
E_GPU = None
my_did_for_row = None
row_for_did = None

if EMB_AVAILABLE:
    import torch as _torch, numpy as _np, pandas as _pd
    print("Loading manifest...")
    _t = time.time()
    _man = _pd.read_parquet(PATHS["emb_manifest"])
    print(f"  manifest rows: {len(_man):,}, cols: {list(_man.columns)}")
    # Build row_index -> our doc_id mapping
    row_for_did = {}; my_did_for_row = [None] * len(_man)
    for _, r in _man.iterrows():
        cit = r.get("citation") or ""
        fam = r.get("family") or ""
        ridx = int(r.get("row_index", -1))
        if ridx < 0: continue
        # Resolve to a doc_id in our cit_to_doc_ids based on family
        candidates = cit_to_doc_ids.get(cit, [])
        for d in candidates:
            if doc_meta.get(d, {}).get("family") == fam:
                row_for_did[d] = ridx
                if ridx < len(my_did_for_row):
                    my_did_for_row[ridx] = d
                break
    n_mapped = sum(1 for x in my_did_for_row if x is not None)
    print(f"  manifest->my_did mapping: {n_mapped:,}/{len(_man):,} ({100*n_mapped/len(_man):.1f}%) in {time.time()-_t:.1f}s")

    # Concat chunks to GPU
    print(f"  loading chunks to GPU (~21 GB)...")
    _t = time.time()
    _chunks = sorted(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
    arrs = []
    for cp in _chunks:
        arrs.append(_torch.from_numpy(_np.load(cp)).to("cuda", non_blocking=True))
    E_GPU = _torch.cat(arrs, dim=0); del arrs
    free_vram = (_torch.cuda.get_device_properties(0).total_memory
                 - _torch.cuda.memory_allocated()) / 1024**3
    print(f"  E_GPU shape={tuple(E_GPU.shape)} dtype={E_GPU.dtype}, "
          f"VRAM used={_torch.cuda.memory_allocated()/1024**3:.1f} GB, "
          f"free={free_vram:.1f} GB, "
          f"load {time.time()-_t:.1f}s")

    def vector_search(q_emb, k):
        if E_GPU is None: return []
        with _torch.no_grad():
            q = q_emb.to(E_GPU.device, dtype=E_GPU.dtype)
            q = q / (q.norm(dim=-1, keepdim=True) + 1e-9)
            scores = E_GPU @ q
            top_v, top_i = _torch.topk(scores, k=min(k, scores.shape[0]))
        out = []
        for s, i in zip(top_v.cpu().tolist(), top_i.cpu().tolist()):
            d = my_did_for_row[i]
            if d is not None: out.append((d, float(s)))
        return out

    VECTOR_OK = True
else:
    print("[skip] EMB_AVAILABLE = False; vector channels will be skipped.")
    def vector_search(q_emb, k): return []

Loading manifest...
  manifest rows: 2,652,248, cols: ['doc_id', 'family', 'citation', 'row_index']
  manifest->my_did mapping: 2,649,348/2,652,248 (99.9%) in 40.0s
  loading chunks to GPU (~21 GB)...
  E_GPU shape=(2652248, 4096) dtype=torch.float16, VRAM used=20.2 GB, free=74.7 GB, load 387.6s


## 6.4 Embedding model for query

**What:** Load Qwen3-Embedding-8B via SentenceTransformer for **query-side**
encoding (corpus-side is pre-encoded).

**Why:** We need the same model that produced the corpus embeddings, with
its canonical instruction prefix.

**VRAM:** ~16 GB. Loaded after Qwen3-32B is freed (Phase 7).

**Failure modes:**
- HF download blocked → preflight `huggingface_hub.snapshot_download` once.
- Model weights mismatch (8B vs 8B-Embedding) → ensure model id is exactly
  `Qwen/Qwen3-Embedding-8B`.

In [ ]:
EMB_MODEL = None
def encode_query(text):
    return EMB_MODEL.encode(
        [text],
        prompt_name="query",
        convert_to_tensor=True,
        normalize_embeddings=True,
    )[0]

# Phase 7 — Query expansion + HyDE (Qwen3-32B)

Load Qwen3-32B **once**, run two passes per query in the same model load, then free the model.

**Pass 1 — structured targets.** Broad system prompt naming many Swiss codes, schema asks for 10-25 items per list, two diverse few-shots, 3-strategy JSON parser (markdown fence → brace-balanced scan → greedy fallback), sampling retry at `temperature=0.4` if first pass returns empty or <10 total items. Produces `ALL_TARGETS[qid]` with `statute_targets`, `concept_targets_en`, `term_targets_de`, `term_targets_fr`, `legal_area_keywords`, `case_targets`. Raw responses persisted to `ALL_RAW_RESPONSES[qid]` for post-mortem.

**Pass 2 — HyDE (hypothetical document embedding).** Same model writes a 4-6 sentence answer paragraph in Bundesgericht / Tribunal fédéral style — citing Swiss articles inline as `Art. N CODE` and including inline German + French legal terms in parentheses. Goes into `ALL_HYDE[qid]`; encoded as a third vector query in Phase 7.6. Falls back to raw query text if the model produces <40 characters.

After both passes complete the model is freed (`del qmod, qtok, ...; torch.cuda.empty_cache()`).


In [ ]:
QEXP_PROMPT_SYSTEM = (
    "You are a Swiss legal-research assistant. You enumerate broadly across "
    "the most relevant Swiss federal codes (StPO, StGB, BGG, ZGB, OR, ZPO, BV, "
    "EMRK, IPRG, IRSG, AHVG, IVG, AsylG, AIG, DBG, StHG, KVG, UVG, AVIG, PatG, "
    "MSchG, URG, FINIG, FINMAG, BankG, KKG, SchKG, VwVG, FZG, BVG, BPV, BetmG, "
    "RPG, NHG, USG, GSchG, SVG, GwG, etc). You output ONLY a single JSON "
    "object — no prose, no markdown, no code fences."
)

QEXP_PROMPT_USER = '''Schema (exact keys, all required):

{
  "statute_targets":      [10-20 "Art. N CODE" strings, broad coverage of relevant codes],
  "case_targets":         [list of "BGE V D P" or "1B_N/Y" dockets — empty if uncertain],
  "concept_targets_en":   [15-25 English legal concepts (technical AND procedural)],
  "term_targets_de":      [15-25 German legal terms in original spelling],
  "term_targets_fr":      [10-20 French legal terms],
  "legal_area_keywords":  [3-6 short legal-area phrases]
}

Hard rules:
- Output ONLY the JSON object. No surrounding text, no markdown, no code fences.
- Each list MUST have at least the minimum number of items.
- Do NOT invent specific BGE volumes or dockets you are uncertain about.

Example A for "Does an unemployed worker keep insurance benefits when he refuses a job offer that is below his prior wage?":
{"statute_targets":["Art. 16 AVIG","Art. 17 AVIG","Art. 30 AVIG","Art. 30 Abs. 1 AVIG","Art. 16 Abs. 2 AVIG","Art. 22 AVIG","Art. 23 AVIG","Art. 8 AVIG","Art. 95 AVIG","Art. 11 AVIG"],
"case_targets":[],
"concept_targets_en":["unemployment insurance","suitable employment","wage protection","willingness to work","refusal of suitable work","reduction of benefits","good cause","unemployment compensation","insured earnings","obligation to accept","sanction","placement","right to compensation","intermediate earnings","admissibility threshold"],
"term_targets_de":["Arbeitslosenversicherung","zumutbare Arbeit","Lohnvergleich","Vermittlungsfähigkeit","Ablehnung","Einstellung in der Anspruchsberechtigung","Versicherungsleistungen","Arbeitslosenentschädigung","Zwischenverdienst","Vermittlungsbemühungen","Arbeitsbemühungen","versicherter Verdienst","Selbstverschulden","Sanktion","Verfügung"],
"term_targets_fr":["assurance-chômage","emploi convenable","comparaison de salaire","aptitude au placement","refus","suspension du droit à l'indemnité","indemnisation","gain intermédiaire","obligation","sanction"],
"legal_area_keywords":["unemployment insurance","social insurance","labour market","social security law"]}

Example B for "Is a will written on lined notebook paper and signed only on the last page valid under Swiss inheritance law?":
{"statute_targets":["Art. 505 ZGB","Art. 498 ZGB","Art. 499 ZGB","Art. 519 ZGB","Art. 520 ZGB","Art. 6 ZGB","Art. 467 ZGB","Art. 468 ZGB","Art. 522 ZGB","Art. 540 ZGB"],
"case_targets":[],
"concept_targets_en":["holographic will","testator","handwriting","signature","testamentary capacity","formal validity","formal requirements","invalidity","challenge of will","disposition mortis causa","heirship","forced heirship","compulsory portion","inheritance","reduction action"],
"term_targets_de":["eigenhändige Verfügung","Testament","Erblasser","Handschrift","Unterschrift","Verfügungsfähigkeit","Formvorschriften","Ungültigerklärung","Anfechtung","letztwillige Verfügung","Erbe","Pflichtteil","Erbschaft","Herabsetzung","Verfügung von Todes wegen"],
"term_targets_fr":["testament olographe","testateur","écriture","signature","capacité de disposer","conditions de forme","nullité","action en réduction","disposition pour cause de mort","héritage","réserve héréditaire"],
"legal_area_keywords":["inheritance law","succession","testamentary law","civil law"]}

Now produce the JSON for this query (JSON object only, no other text):
{QUERY}
'''

import time, gc
import re as _re
import json as _json
import torch as _torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def parse_targets_json(text):
    """Robust extractor: markdown-fence -> brace-balanced scan -> greedy."""
    if not text:
        return None
    # 1) ```json {...} ``` or ``` {...} ```
    fence = _re.search(r"```(?:json|JSON)?\s*(\{.*?\})\s*```", text, _re.S)
    if fence:
        try:
            return _json.loads(fence.group(1))
        except Exception:
            pass
    # 2) brace-balanced scan from each '{'
    n = len(text); i = 0
    while i < n:
        i = text.find("{", i)
        if i < 0:
            break
        depth = 0; in_str = False; esc = False; j = i
        while j < n:
            c = text[j]
            if esc:
                esc = False
            elif c == "\\":
                esc = True
            elif c == '"':
                in_str = not in_str
            elif not in_str:
                if c == "{":
                    depth += 1
                elif c == "}":
                    depth -= 1
                    if depth == 0:
                        cand = text[i:j+1]
                        try:
                            return _json.loads(cand)
                        except Exception:
                            break
            j += 1
        i += 1
    # 3) greedy fallback
    m = _re.search(r"\{.*\}", text, _re.S)
    if m:
        try:
            return _json.loads(m.group(0))
        except Exception:
            pass
    return None


def normalize_targets(d):
    """Coerce whatever the model returned into the expected schema."""
    keys = ("statute_targets", "case_targets", "concept_targets_en",
            "term_targets_de", "term_targets_fr", "legal_area_keywords")
    out = {k: [] for k in keys}
    if not isinstance(d, dict):
        return out
    for k in keys:
        v = d.get(k)
        if v is None:
            continue
        if isinstance(v, str):
            out[k] = [v.strip()] if v.strip() else []
        elif isinstance(v, list):
            flat = []
            for item in v:
                if isinstance(item, str) and item.strip():
                    flat.append(item.strip())
                elif isinstance(item, dict):
                    for vv in item.values():
                        if isinstance(vv, str) and vv.strip():
                            flat.append(vv.strip()); break
            out[k] = flat
    return out


_qmodel = CONFIG["qwen_query_model"]
print(f"[qexp] loading {_qmodel} (~65 GB bf16)...")
_t = time.time()
qtok = AutoTokenizer.from_pretrained(_qmodel)
qmod = AutoModelForCausalLM.from_pretrained(
    _qmodel,
    dtype=_torch.bfloat16,
    device_map="auto",
)
qmod.eval()
print(f"[qexp] model loaded in {time.time() - _t:.1f}s")

_input_device = next(p.device for p in qmod.parameters() if p.device.type != "meta")

ALL_TARGETS = {}
ALL_RAW_RESPONSES = {}

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    _t0 = time.time()

    _messages = [
        {"role": "system", "content": QEXP_PROMPT_SYSTEM},
        {"role": "user",   "content": QEXP_PROMPT_USER.replace("{QUERY}", qtext)},
    ]
    _inp = qtok.apply_chat_template(
        _messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )
    _inp = {k: v.to(_input_device) for k, v in _inp.items()}
    _prompt_len = _inp["input_ids"].shape[1]

    # Pass 1: deterministic.
    with _torch.no_grad():
        _out = qmod.generate(
            **_inp,
            max_new_tokens=CONFIG["qwen_max_new_tokens"],
            do_sample=False,
            pad_token_id=qtok.eos_token_id,
        )
    _resp = qtok.decode(_out[0][_prompt_len:], skip_special_tokens=True)
    raw = parse_targets_json(_resp)
    targets = normalize_targets(raw)
    n_items = sum(len(v) for v in targets.values())

    # Pass 2: sampling retry on parse failure OR weak result.
    retried = False
    if raw is None or n_items < 10:
        retried = True
        with _torch.no_grad():
            _out2 = qmod.generate(
                **_inp,
                max_new_tokens=int(CONFIG["qwen_max_new_tokens"] * 1.5),
                do_sample=True,
                temperature=0.4,
                top_p=0.95,
                pad_token_id=qtok.eos_token_id,
            )
        _resp2 = qtok.decode(_out2[0][_prompt_len:], skip_special_tokens=True)
        raw2 = parse_targets_json(_resp2)
        if raw2 is not None:
            targets2 = normalize_targets(raw2)
            n_items2 = sum(len(v) for v in targets2.values())
            if n_items2 > n_items:
                targets, _resp, n_items = targets2, _resp2, n_items2

    ALL_RAW_RESPONSES[qid] = _resp
    ALL_TARGETS[qid] = targets

    flag = "[retry]" if retried else "       "
    n_stat = len(targets["statute_targets"])
    n_ce   = len(targets["concept_targets_en"])
    n_td   = len(targets["term_targets_de"])
    n_tf   = len(targets["term_targets_fr"])
    n_la   = len(targets["legal_area_keywords"])
    print(f"[qexp] {qid}  {time.time()-_t0:5.1f}s {flag}  "
          f"stat={n_stat:>2} ce={n_ce:>2} td={n_td:>2} tf={n_tf:>2} la={n_la:>2}")

    if n_items == 0:
        # Print enough of the raw response to diagnose the failure.
        print(f"  [qexp][WARN] {qid}: empty targets even after retry. "
              f"Raw response (first 800 chars):")
        print("  " + repr(_resp[:800]))

# ---------------------------------------------------------------------------
# Multi-aspect HyDE — one LLM call identifies 2-5 distinct legal aspects of
# the query and writes one answer paragraph per aspect (Bundesgericht /
# Tribunal fédéral style). Each paragraph becomes its own vector query.
# Same model still loaded; identical input device.
# ---------------------------------------------------------------------------

QHYDE_PROMPT_SYSTEM = (
    "You are a Swiss legal expert. Given an English legal question, you "
    "identify the 2-5 distinct legal aspects it covers (different legal "
    "areas, NOT paraphrases of each other) and write a 3-5 sentence answer "
    "paragraph per aspect in Bundesgericht / Tribunal fédéral style. Each "
    "paragraph MUST cite relevant Swiss articles inline (Art. N CODE), use "
    "precise legal terminology, and include German AND French legal terms "
    "in parentheses where natural."
)

QHYDE_PROMPT_USER = (
    "Identify 2-5 distinct legal aspects of this question, then write one "
    "answer paragraph per aspect. Output ONLY in this exact format, no "
    "preamble, no markdown, no extra text:\n\n"
    "ASPECT 1: <legal area / sub-topic name>\n"
    "ANSWER 1: <3-5 sentences>\n\n"
    "ASPECT 2: <legal area / sub-topic name>\n"
    "ANSWER 2: <3-5 sentences>\n\n"
    "...\n\n"
    "Question: {QUERY}"
)

import re as _re_hyde

ALL_HYDE_ASPECTS = {}

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    _th = time.time()
    _msgs_h = [
        {"role": "system", "content": QHYDE_PROMPT_SYSTEM},
        {"role": "user",   "content": QHYDE_PROMPT_USER.replace("{QUERY}", qtext)},
    ]
    _inp_h = qtok.apply_chat_template(
        _msgs_h,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )
    _inp_h = {k: v.to(_input_device) for k, v in _inp_h.items()}
    _plen_h = _inp_h["input_ids"].shape[1]
    with _torch.no_grad():
        _out_h = qmod.generate(
            **_inp_h,
            max_new_tokens=1500,
            do_sample=False,
            pad_token_id=qtok.eos_token_id,
        )
    _resp_h = qtok.decode(_out_h[0][_plen_h:], skip_special_tokens=True).strip()

    # Parse every "ANSWER N: ..." block.
    aspects = []
    _matches = _re_hyde.findall(
        r"ANSWER\s+\d+\s*:\s*(.+?)(?=ASPECT\s+\d+\s*:|\Z)",
        _resp_h, _re_hyde.S | _re_hyde.I)
    for _m in _matches:
        _text = _m.strip()
        if len(_text) >= 60:
            aspects.append(_text)

    # Fallback: parser failed -> use raw response if substantial, else query.
    if not aspects:
        if len(_resp_h) >= 60:
            aspects = [_resp_h]
            print(f"[hyde] {qid}  [WARN parse-failed, single-paragraph fallback]")
        else:
            aspects = [qtext]
            print(f"[hyde] {qid}  [WARN response too short ({len(_resp_h)} chars), query-text fallback]")

    ALL_HYDE_ASPECTS[qid] = aspects
    print(f"[hyde] {qid}  {time.time()-_th:5.1f}s  {len(aspects)} aspect(s)")
    for _i, _a in enumerate(aspects[:3]):
        print(f"    [{_i+1}] {_a[:120].replace(chr(10), ' ')}...")
    if len(aspects) > 3:
        print(f"    ... and {len(aspects)-3} more")

# Free Qwen3-32B from VRAM.
del qmod, qtok, _inp, _out, _inp_h, _out_h
gc.collect()
_torch.cuda.empty_cache()
if _torch.cuda.is_available():
    _torch.cuda.synchronize()
    print(f"\n[qexp] freed Qwen3-32B; CUDA mem={_torch.cuda.memory_allocated() / 1024**3:.1f} GB")
else:
    print("\n[qexp] freed Qwen3-32B")


[qexp] loading Qwen/Qwen3-32B (~65 GB bf16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[qexp] model loaded in 213.9s
[qexp] val_001   54.5s          stat=20 ce=20 td=20 tf=20 la= 6
[qexp] val_002   52.8s          stat=20 ce=24 td=24 tf=24 la= 6
[qexp] val_003   51.9s          stat=20 ce=25 td=25 tf=23 la= 6
[qexp] val_004   44.2s          stat=20 ce=20 td=20 tf=19 la= 6
[qexp] val_005  104.0s [retry]  stat=20 ce=20 td=20 tf=19 la= 5
[qexp] val_006   51.9s          stat=20 ce=24 td=24 tf=21 la= 6
[qexp] val_007   50.9s          stat=20 ce=25 td=25 tf=24 la= 6
[qexp] val_008   54.9s          stat=20 ce=25 td=26 tf=21 la= 6
[qexp] val_009   46.1s          stat=20 ce=25 td=25 tf=21 la= 5
[qexp] val_010   50.2s          stat=20 ce=20 td=20 tf=20 la= 6
[hyde] val_001   43.8s  5 aspect(s)
    [1] Under Art. 221 Abs. 1 lit. b StPO, the extension of pre-trial detention is permissible if there is a concrete risk of co...
    [2] Art. 221 Abs. 2 StPO mandates that the extension of pre-trial detention be based on a concrete and specific risk, not me...
    [3] The right to liberty a

## 7.6 Encode queries (Qwen3-Embedding-8B)

Load Qwen3-Embedding-8B **once**, encode three text variants per query: raw question, keyword-enriched (raw + first 60 LLM-named terms / concepts), and the HyDE answer paragraph. Populates `ALL_Q_EMB_RAW[qid]`, `ALL_Q_EMB_ENRICHED[qid]`, `ALL_Q_EMB_HYDE[qid]`.

These three embeddings feed the three vector channels (`vector_raw`, `vector_enriched`, `vector_hyde`) in `run_channels`.


In [ ]:
print("Loading Qwen3-Embedding-8B...")
from sentence_transformers import SentenceTransformer
EMB_MODEL = SentenceTransformer(CONFIG["vector_emb_model"])
print(f"  done")

ALL_Q_EMB_RAW = {}
ALL_Q_EMB_ENRICHED = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    tgt = ALL_TARGETS.get(qid, {})
    enriched_bits = (
        (tgt.get("term_targets_de") or [])
      + (tgt.get("term_targets_fr") or [])
      + (tgt.get("concept_targets_en") or [])
    )
    enriched_query = qtext + " " + " ".join(enriched_bits[:60])
    ALL_Q_EMB_RAW[qid] = encode_query(qtext)
    ALL_Q_EMB_ENRICHED[qid] = encode_query(enriched_query)
    print(f"  {qid:<8}  raw + enriched   ({len(qtext)} -> {len(enriched_query)} chars, "
          f"+{min(60, len(enriched_bits))} kw)")

# Encode each multi-aspect HyDE paragraph separately. The vector_hyde channel
# in cell 34 will RRF-fuse per-aspect dense searches into a single channel.
ALL_Q_EMB_HYDE_LIST = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    aspects = ALL_HYDE_ASPECTS.get(qid) or [q["query_text"]]
    ALL_Q_EMB_HYDE_LIST[qid] = [encode_query(a) for a in aspects]
    print(f"  {qid:<8}  encoded {len(aspects)} aspect(s)  "
          f"(total {sum(len(a) for a in aspects):,} chars)")


Loading Qwen3-Embedding-8B...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  done
  val_001   raw + enriched   (1068 -> 2309 chars, +60 kw)
  val_002   raw + enriched   (1636 -> 3014 chars, +60 kw)
  val_003   raw + enriched   (1677 -> 2740 chars, +60 kw)
  val_004   raw + enriched   (1147 -> 2159 chars, +59 kw)
  val_005   raw + enriched   (1581 -> 2785 chars, +59 kw)
  val_006   raw + enriched   (1437 -> 2774 chars, +60 kw)
  val_007   raw + enriched   (1742 -> 2791 chars, +60 kw)
  val_008   raw + enriched   (1527 -> 2622 chars, +60 kw)
  val_009   raw + enriched   (1116 -> 2227 chars, +60 kw)
  val_010   raw + enriched   (1695 -> 3015 chars, +60 kw)
  val_001   encoded 5 aspect(s)  (total 3,532 chars)
  val_002   encoded 4 aspect(s)  (total 3,503 chars)
  val_003   encoded 4 aspect(s)  (total 3,426 chars)
  val_004   encoded 4 aspect(s)  (total 2,709 chars)
  val_005   encoded 5 aspect(s)  (total 3,229 chars)
  val_006   encoded 5 aspect(s)  (total 3,190 chars)
  val_007   encoded 5 aspect(s)  (total 3,386 chars)
  val_008   encoded 5 aspect(s)  (total 3,

# Phase 8 — Channels

`run_channels()` produces **15 channels** per query using one identical code path — no per-id specialisation. The function is defined once in the next cell and called inside the master loop in Phase 9.

The 15 channels split into 4 families:

- **Statute-anchored (4):** `law_direct_match`, `court_statute`, `co_citation`, `per_area_bedrock` — fire from LLM-named statutes + corpus-derived code-family expansion + PRF-augmented canons.
- **Graph-derived (5):** `statute_backprop`, `sibling_expansion`, `graph_forward`, `graph_reverse`, `graph_2hop` — fire from the caught-court-rows **seed pool**. All run **dual-pass** (pass-1 seed + PRF-expanded seed, merged per channel) so PRF can never displace pass-1 hits past a budget cap.
- **Text-derived (3):** `concept_en`, `term_orig`, `bm25` — fire from LLM-named concepts / terms / query text (with `enhance()` corpus-trained code boosting). Statute / concept / term channels are also PRF-augmented in pass 2.
- **Vector-derived (3):** `vector_raw`, `vector_enriched`, `vector_hyde` — brute-force GPU cosine against the 2.65 M corpus embedding tensor.


In [ ]:
from collections import Counter

def channel_law_direct(canon_set, idx, budget):
    counter = Counter()
    for canon in canon_set:
        for did in idx[canon]: counter[did] += 1
    items = counter.most_common()
    return items if budget is None else items[:budget]

def channel_court_statute(statute_canons, idx, idx_count=None, doc_meta=None, budget=None):
    """Score court rows for an LLM-named statute canonical set.

    Score(did) = (sum over matched canons of 1/log(2 + global_count[canon]))
                 * (1 + 0.3 * (n_matches - 1))         # multi-match bonus
                 * paragraph_role_weight(did)          # 0.4 / 0.6 / 1.0 / 1.5

    Backwards-compatible: idx_count and doc_meta are optional. If idx_count is
    None (e.g. cell 12 hasn't been re-run with the new patch), counts are derived
    from idx on the fly. If doc_meta is None, role weighting is skipped (1.0).
    """
    import math as _math
    if idx_count is None:
        idx_count = {c: len(idx[c]) for c in statute_canons if c in idx}

    # paragraph_role -> multiplicative weight
    _ROLE_W = {
        "legal_standard": 1.5, "reasoning": 1.5,
        "application":    1.5, "holding":   1.5,
        "facts":              1.0, "procedural_history": 1.0,
        "citation":           1.0, "neutral_default":    1.0,
        "costs":        0.6, "disposition": 0.6, "notification": 0.6,
        "neutral":      0.4,
    }

    # Per-doc accumulator: {did: [matches_so_far, summed_specificity_weight]}
    per_doc = {}
    for canon in statute_canons:
        dids = idx.get(canon)
        if not dids:
            continue
        cnt = idx_count.get(canon)
        if cnt is None:
            cnt = len(dids)
        w_canon = 1.0 / _math.log(2 + cnt)
        for did in dids:
            slot = per_doc.get(did)
            if slot is None:
                per_doc[did] = [1, w_canon]
            else:
                slot[0] += 1
                slot[1] += w_canon

    if not per_doc:
        return []

    scored = []
    for did, (n_matches, base_w) in per_doc.items():
        score = base_w * (1.0 + 0.3 * (n_matches - 1))
        if doc_meta is not None:
            meta = doc_meta.get(did) or {}
            role = meta.get("paragraph_role")
            if role is None or role == "":
                rw = 0.4
            else:
                rw = _ROLE_W.get(role, 1.0)
            score *= rw
        scored.append((did, float(score)))

    # Stable deterministic ordering: descending score, ascending doc_id on ties.
    scored.sort(key=lambda x: (-x[1], x[0]))
    if budget is None:
        return scored
    return scored[:budget]

import math as _math_v74

_SUBSTANTIVE_ROLES = {"reasoning", "legal_standard", "application", "holding"}

def _judgment_factor(cb, idx_judgment_importance):
    """sqrt(1 + log(1 + importance)). Default 1.0 for unknown / 0 importance."""
    imp = idx_judgment_importance.get(cb, 0) if cb else 0
    if imp <= 0:
        return 1.0
    return _math_v74.sqrt(1.0 + _math_v74.log(1.0 + float(imp)))

def _role_boost(doc_meta, did):
    m = doc_meta.get(did) or {}
    role = (m.get("paragraph_role") or "").strip().lower()
    return 1.5 if role in _SUBSTANTIVE_ROLES else 1.0

def channel_sibling(seed_doc_ids, idx_court_base, idx_judgment_importance,
                    doc_meta, budget):
    # 1) seed_count[cb] = how many caught seeds share court_base cb
    seed_count = Counter()
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb = m.get("court_base")
        if cb: seed_count[cb] += 1
    # 2) emit every sibling row with score = seed_count * judgment_factor * role_boost
    seed_set = set(seed_doc_ids)
    scored = {}
    for cb, cnt in seed_count.items():
        factor = _judgment_factor(cb, idx_judgment_importance)
        for s in idx_court_base.get(cb, ()):
            if s in seed_set: continue
            sc = float(cnt) * factor * _role_boost(doc_meta, s)
            # keep best score per doc (a row only belongs to one cb)
            if sc > scored.get(s, 0.0):
                scored[s] = sc
    # 3) deterministic order: score desc, doc_id asc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_forward(seed_doc_ids, idx_graph_out, idx_judgment_importance,
                          doc_meta, budget):
    # edge-count per target (how many distinct seeds point to it)
    seed_set = set(seed_doc_ids)
    edge_count = Counter()
    for did in seed_doc_ids:
        for t in idx_graph_out.get(did, ()):
            edge_count[t] += 1
    for d in seed_set:
        edge_count.pop(d, None)
    # weight each target by importance of its OWN judgment + role boost
    scored = {}
    for t, cnt in edge_count.items():
        m = doc_meta.get(t) or {}
        cb_of_t = m.get("court_base")
        factor = _judgment_factor(cb_of_t, idx_judgment_importance)
        sc = float(cnt) * factor * _role_boost(doc_meta, t)
        scored[t] = sc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_reverse(seed_doc_ids, idx_graph_in, idx_judgment_importance,
                          doc_meta, budget,
                          landmark_imp_threshold=5):
    """Reverse-graph expansion gated to landmark seeds.
    Only follows incoming-edges from seeds whose judgment has
    importance >= landmark_imp_threshold (default 5). Each contributing
    edge is weighted by the SEED-judgment importance, so landmark seeds
    dominate.
    """
    seed_set = set(seed_doc_ids)
    scored = {}
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb_seed = m.get("court_base")
        imp_seed = idx_judgment_importance.get(cb_seed, 0) if cb_seed else 0
        if imp_seed < landmark_imp_threshold:
            continue
        factor = _judgment_factor(cb_seed, idx_judgment_importance)
        for s in idx_graph_in.get(did, ()):
            if s in seed_set: continue
            inc = factor * _role_boost(doc_meta, s)
            scored[s] = scored.get(s, 0.0) + inc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_2hop(seed_doc_ids, idx_graph_out, budget):
    seed_set = set(seed_doc_ids)
    intermediate = set()
    for did in seed_doc_ids:
        intermediate.update(idx_graph_out.get(did, ()))
    intermediate -= seed_set
    counter = Counter()
    for x in intermediate:
        for t in idx_graph_out.get(x, ()):
            if t in seed_set: continue
            counter[t] += 1
    for d in intermediate:
        counter.pop(d, None)
    return counter.most_common(budget)

_CONCEPT_STOPWORDS_EN = frozenset({
    "a","an","the","of","in","on","at","by","for","with","to","from",
    "and","or","but","is","are","was","were","be","been","being",
    "has","have","had","do","does","did","no","not",
})
_CONCEPT_TOKEN_SPLIT_RE = re.compile(r"[^\w]+", re.UNICODE)

def _concept_tokens(s):
    """Whitespace+punct split, lowercase. Returns full token list."""
    if not s: return []
    return [t for t in _CONCEPT_TOKEN_SPLIT_RE.split(s.lower()) if t]

def _concept_meaningful_tokens(s):
    return [t for t in _concept_tokens(s) if t not in _CONCEPT_STOPWORDS_EN]

def expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25):
    """Replace strict-substring matcher with a weighted scorer.

    Combines three signals against each corpus concept:
      1. Exact match               -> weight 1.0
      2. Substring (either dir)    -> weight 0.85 * shorter/longer
         (must also share >=1 meaningful token; otherwise a stopword
         only embedding like 'of detention' could hijack 'of').
      3. Token overlap             -> weight shared/max(q_tok, c_tok)
         after stripping English stopwords from BOTH sides.

    Returns list[(corpus_concept, weight)] sorted by weight desc,
    capped at top_k per query concept (max weight kept across
    multiple query concepts hitting the same corpus concept).
    """
    W_EXACT = 1.0
    W_SUBSTR_MAX = 0.85  # capped below exact so true matches dominate

    keys = list(corpus_concept_keys)
    # Pre-tokenize the corpus once: corpus_concept -> (set_of_tokens, n_tokens)
    corpus_meaningful = {}
    for k in keys:
        m = _concept_meaningful_tokens(k)
        if m:
            corpus_meaningful[k] = (set(m), len(m))

    best_weight = {}  # corpus_concept -> max weight across query concepts

    for raw in llm_concepts or []:
        c = (raw or "").lower().strip()
        if not c or len(c) < 4:
            continue
        c_meaningful = _concept_meaningful_tokens(c)
        if not c_meaningful:
            # Pure-stopword query (e.g. "of the") yields no matches.
            continue
        c_set = set(c_meaningful)
        c_len = len(c_meaningful)
        c_chars = len(c)

        per_query = []  # (weight, length_diff, corpus_concept)

        # 1. Exact match
        if c in corpus_concept_keys:
            per_query.append((W_EXACT, 0, c))

        # 2. Substring match (either direction), gated on shared
        #    meaningful token to block stopword-only bridges.
        for cv in keys:
            if cv == c:
                continue
            if c in cv or cv in c:
                cv_info = corpus_meaningful.get(cv)
                if not cv_info:
                    continue
                cv_set, cv_len = cv_info
                if not (c_set & cv_set):
                    # only stopword/character overlap; reject
                    continue
                shorter = min(c_chars, len(cv))
                longer  = max(c_chars, len(cv))
                if longer <= 0:
                    continue
                w = W_SUBSTR_MAX * (shorter / longer)
                per_query.append((w, abs(len(cv) - c_chars), cv))

        # 3. Token overlap (stopwords already stripped on both sides)
        for cv, (cv_set, cv_len) in corpus_meaningful.items():
            if cv == c:
                continue
            shared = c_set & cv_set
            if not shared:
                continue
            denom = max(c_len, cv_len)
            if denom <= 0:
                continue
            w = len(shared) / denom
            per_query.append((w, abs(cv_len - c_len), cv))

        # Dedupe within this query concept: max weight per corpus key.
        local_best = {}
        for w, ld, cv in per_query:
            cur = local_best.get(cv)
            if cur is None or w > cur[0] or (w == cur[0] and ld < cur[1]):
                local_best[cv] = (w, ld)

        # Rank: weight desc, then length-diff asc, then alphabetical.
        ranked = sorted(local_best.items(),
                        key=lambda kv: (-kv[1][0], kv[1][1], kv[0]))
        for cv, (w, _) in ranked[:top_k]:
            prior = best_weight.get(cv, 0.0)
            if w > prior:
                best_weight[cv] = w

    # Final ordering: weight desc, then alphabetical for determinism.
    return sorted(best_weight.items(), key=lambda kv: (-kv[1], kv[0]))

def channel_concept(expanded_weighted, idx, budget):
    """Consume list[(concept, weight)] from expand_concepts_weighted.

    Each doc accumulates the sum of weights from every matched
    corpus concept it carries. Top `budget` by accumulated score.
    Backward-compatible: tolerates a list of plain strings (legacy)
    by treating each as weight=1.0.
    """
    scores = {}
    for item in expanded_weighted or []:
        if isinstance(item, tuple):
            tok, w = item
        else:
            tok, w = item, 1.0
        if not tok:
            continue
        for did in idx.get(tok, ()):
            scores[did] = scores.get(did, 0.0) + float(w)
    if not scores:
        return []
    # Sort by score desc, then doc_id for determinism. Cap at budget.
    items = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    if budget is not None:
        items = items[:budget]
    return items

def channel_term(targets, idx, idx_lemma, budget,
                 corpus_keys=None, lemma_score=0.7,
                 min_substring_len=4):
    """Term_orig channel — exact + lemma + substring fused scorer.

    Score model (per query-term/corpus-term pair, MAX over paths):
      exact         -> 1.0
      lemma equal   -> `lemma_score` (default 0.7) (DE-only; we still try FR/IT
                       through term_lemma but lemma_score is conservative)
      Q in T        -> len(Q) / len(T)
      T in Q        -> len(T) / len(Q)   (only if len(T) >= min_substring_len)
    Per doc_id we sum scores across all matching pairs and return the
    top-`budget` doc_ids by sum.
    """
    if corpus_keys is None:
        # Fallback: derive surface-form keys from the exact-match index. Slower
        # to construct on the fly but keeps callers without the precomputed set
        # working.
        corpus_keys = list(idx.keys())
    else:
        corpus_keys = list(corpus_keys)

    # Collect normalized query terms (DE first, then FR; we treat both
    # symmetrically — substring matching is language-agnostic, lemma logic
    # is most reliable for DE but doesn't actively hurt FR/IT because the
    # lemma function is identity for tokens with no removable suffix).
    q_terms = []
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok:
                q_terms.append(tok)
    if not q_terms:
        return []

    # doc_id -> accumulated score (max-per-pair, summed across query terms)
    score = Counter()

    for Q in q_terms:
        Q_lemma = term_lemma(Q)
        Q_len = len(Q)
        # Per-doc max for this single query term — prevents two paths
        # (e.g. exact + lemma) from double-counting the same doc.
        per_q = {}
        def _bump(did, s):
            if s > per_q.get(did, 0.0):
                per_q[did] = s

        # 1. Exact match — strongest signal, score 1.0.
        if Q in idx:
            for did in idx[Q]:
                _bump(did, 1.0)

        # 2. Lemma match — only fire when lemma differs from surface or when
        #    Q's lemma keys are present. Score 0.7 (capped below exact).
        if Q_lemma and Q_lemma in idx_lemma:
            for did in idx_lemma[Q_lemma]:
                _bump(did, lemma_score)

        # 3. Substring scan — for each corpus term T, compute the longer
        #    of Q⊂T and T⊂Q. Skip the exact-equal case (already scored).
        if Q_len >= 1:
            for T in corpus_keys:
                if T == Q:
                    continue
                T_len = len(T)
                s = 0.0
                if Q in T:
                    s = Q_len / T_len      # len(Q) / len(T)
                elif T_len >= min_substring_len and T in Q:
                    s = T_len / Q_len      # len(T) / len(Q)
                if s <= 0.0:
                    continue
                # Each corpus term T may map to many doc_ids; bump them all.
                for did in idx.get(T, ()):  # exact-form posting is canonical
                    _bump(did, s)

        # Roll the per-Q max scores into the cross-Q sum.
        for did, s in per_q.items():
            score[did] += s

    # most_common-style ordering by score, with deterministic tie-break on doc_id.
    items = sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_per_area_bedrock(legal_area_keywords, statute_target_codes,
                             per_area_canon_count, idx_law_direct, budget):
    if not legal_area_keywords: return []
    keys = [k.lower() for k in legal_area_keywords]
    selected_areas = set()
    for area in per_area_canon_count.keys():
        for k in keys:
            if k in area: selected_areas.add(area); break
    if not selected_areas: return []
    canon_score = Counter()
    for area in selected_areas:
        for canon, n in per_area_canon_count[area].most_common(CONFIG["per_area_top_n"]):
            canon_score[canon] = max(canon_score[canon], n)
    out = []; seen = set()
    for canon, _ in canon_score.most_common():
        canon_code = canon.split()[1] if canon and " " in canon else None
        if statute_target_codes and canon_code not in statute_target_codes:
            continue
        for did in idx_law_direct.get(canon, set()):
            if did not in seen:
                out.append((did, canon_score[canon])); seen.add(did)
        if len(out) >= budget: break
    return out[:budget]

def channel_statute_backprop(seed_court_dids, doc_statute_anchors, idx_law_direct, budget):
    import math as _math
    canon_to_courts = defaultdict(set)
    for did in seed_court_dids:
        for canon in doc_statute_anchors.get(did, set()):
            canon_to_courts[canon].add(did)
    # Specificity weighting: rare canons (small global court count) score more per
    # caught row than common procedural canons (Art. 100 BGG, Art. 9 BV, ...).
    # Idempotent against court_statute fix agent which may also define this:
    _idx_canon_ct = (locals().get('idx_court_statute_count')
                     or globals().get('idx_court_statute_count')
                     or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for canon, court_set in canon_to_courts.items():
        n_caught = len(court_set)
        global_ct = _idx_canon_ct.get(canon, n_caught)
        score = n_caught * (1.0 / _math.log(2 + global_ct))
        for law_did in idx_law_direct.get(canon, set()):
            if counter[law_did] < score: counter[law_did] = score
    return counter.most_common(budget)

def channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute, budget):
    import math as _math
    # Use canon_count from cell 20 if available; otherwise derive from idx_court_statute.
    _canon_ct = (locals().get('canon_count')
                 or globals().get('canon_count')
                 or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for raw in targets.get("statute_targets", []) or []:
        canon = statute_anchor_canonical(raw)
        if not canon: continue
        for nb, n in co_neighbours.get(canon, []):
            spec = 1.0 / _math.log(2 + _canon_ct.get(nb, 1))
            score = n * spec
            for did in idx_law_direct.get(nb, set()):
                if counter[did] < score: counter[did] = score
            for did in idx_court_statute.get(nb, set()):
                if counter[did] < score: counter[did] = score
    return counter.most_common(budget)

def channel_vector(q_emb, k):
    return vector_search(q_emb, k)

## 8.4 Channel runner (function)

`run_channels(query_text, targets, q_emb_raw, q_emb_enriched, q_emb_hyde, gold_doc_set, total_gold)` returns the 15-channel list for one query. Two-stage:

1. **Pass 1.** Topical channels with LLM-named targets + the three vector queries + BM25.
2. **PRF extraction.** Mini-RRF across pass-1 channels (skip `law_direct` so the pool is court-row biased), take the top `prf_top_k_for_extraction` (=2000) court doc_ids. Aggregate their `statute_anchors`, `concepts_en` and `terms_original` annotations. Keep the top-N **new** signals not already in pass-1 targets (`prf_new_statutes_k`=15, `prf_new_concepts_k`=30, `prf_new_terms_k`=30). Derive new codes from new canons. Pure corpus signal — no hardcoded lists.
3. **Pass 2.** Re-run `law_direct_match`, `court_statute`, `concept_en`, `term_orig`, `per_area_bedrock`, `co_citation` with augmented targets. Merge per channel (union, keep max score per doc).
4. **Dual-pass graph.** Build `seed_p1` (court hits from pass-1 topical + vectors + bm25) and `seed_p2` (same, but from merged topical). Run `channel_sibling`, `channel_graph_forward`, `channel_graph_reverse`, `channel_statute_backprop` **twice** — once with `seed_p1`, once with `seed_p2` — and merge. This preserves pass-1 gold against budget-cap displacement when pass-2 finds many new candidates.
5. **Per-channel hit sets** are stored on `PER_QUERY[qid]["channel_hit_sets"]` so the missed-gold diagnostic in Phase 10.4 can identify gold that fell through every channel.

A one-time reverse map (`doc → concepts`, `doc → terms`) is built lazily on first call (~10s on 2.5M-row corpus, cached for the rest of the kernel session).


In [ ]:
# Channel-runner with corpus-side pseudo-relevance feedback (PRF).
#
# Two passes:
#   Pass 1  — Topical channels using the LLM's expansion (current logic).
#   Mini-fuse pass-1 hits, keep top-K court dids ("PRF pool").
#   Aggregate the PRF pool's annotations:
#       statute_anchors  -> top-K new canons not in pass-1 targets
#       concepts_en      -> top-K new corpus concepts
#       terms_original   -> top-K new corpus terms (DE/FR/IT)
#   Pass 2  — Re-run the canon/concept/term-dependent channels with augmented
#             targets.
#   Merge per-channel: union pass-1 + pass-2, keep best score per doc.
#   Build the seed pool from merged topical channels + vectors + bm25.
#   Run sibling / graph / backprop ONCE on the expanded seed.
#
# No per-query branching. The same code path runs for every query.


def _ensure_doc_reverse_maps():
    """Lazy build of doc -> concepts and doc -> terms reverse maps. Built once
    per kernel session and cached in globals. The forward maps idx_concept_en
    and idx_term_orig are keyed by signal -> set(dids); we need the inverse."""
    global _doc_to_concepts, _doc_to_terms
    if globals().get("_doc_to_concepts") is None:
        import time as _time
        _t = _time.time()
        _doc_to_concepts = defaultdict(set)
        for _k, _s in idx_concept_en.items():
            for _d in _s:
                _doc_to_concepts[_d].add(_k)
        _doc_to_terms = defaultdict(set)
        for _k, _s in idx_term_orig.items():
            for _d in _s:
                _doc_to_terms[_d].add(_k)
        print(f"  [PRF reverse maps] built in {_time.time()-_t:.1f}s "
              f"({len(_doc_to_concepts):,} concept-bearing dids, "
              f"{len(_doc_to_terms):,} term-bearing dids)")


def _merge_channel(a, b):
    """Merge two channel hit lists (list[(did, score)]). Keep the higher
    score per doc. Returns deterministic list sorted by score desc, did asc."""
    seen = {}
    for did, sc in a:
        seen[did] = float(sc)
    for did, sc in b:
        s = float(sc)
        if did not in seen or s > seen[did]:
            seen[did] = s
    return sorted(seen.items(), key=lambda kv: (-kv[1], kv[0]))


def run_channels(QUERY_TEXT, targets, q_emb_raw, q_emb_enriched, q_emb_hyde_list,
                 gold_doc_set, total_gold, verbose=False):
    # ---------- canonical sets from LLM targets ----------
    llm_statute_canons = set()
    for raw in targets.get("statute_targets", []) or []:
        c = statute_anchor_canonical(raw)
        if c:
            llm_statute_canons.add(c)
    co_expanded_canons = set(llm_statute_canons)
    for canon in llm_statute_canons:
        for nb, _ in co_neighbours.get(canon, []):
            co_expanded_canons.add(nb)

    # corpus-derived code-family expansion for per_area_bedrock
    statute_target_codes = set()
    for canon in llm_statute_canons:
        if " " in canon:
            statute_target_codes.add(canon.split()[1])
    llm_codes_only = set(statute_target_codes)
    _kfam = CONFIG.get("code_family_top_k", 8)
    for c in llm_codes_only:
        related = sorted(
            ((cc, n) for (a, cc), n in code_pair_count.items() if a == c),
            key=lambda x: -x[1])[:_kfam]
        for cc, _ in related:
            statute_target_codes.add(cc)

    # concept expansion
    llm_concepts = (targets.get("concept_targets_en") or []) + (targets.get("legal_area_keywords") or [])
    corpus_concept_keys = set(idx_concept_en.keys())
    expanded_weighted = expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25)

    # ---------- PASS 1: topical channels ----------
    ch_law_direct = channel_law_direct(co_expanded_canons, idx_law_direct, CONFIG["budget_law_direct"])
    ch_court_stat = channel_court_statute(llm_statute_canons, idx_court_statute,
                                          idx_court_statute_count, doc_meta,
                                          CONFIG["budget_court_statute"])
    ch_concept    = channel_concept(expanded_weighted, idx_concept_en, CONFIG["budget_concept"])
    ch_term       = channel_term(targets, idx_term_orig, idx_term_lemma,
                                  CONFIG["budget_term"], corpus_keys=term_orig_keys)
    ch_per_area   = channel_per_area_bedrock(targets.get("legal_area_keywords", []),
                                              statute_target_codes, per_area_canon_count,
                                              idx_law_direct, CONFIG["budget_per_area"])
    ch_cocit      = channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute,
                                         CONFIG["budget_co_citation"])
    ch_bm25 = bm25_search_multilang(QUERY_TEXT, targets, CONFIG["budget_bm25"])

    ch_vector  = channel_vector(q_emb_raw,      CONFIG["budget_vector"])           if VECTOR_OK else []
    ch_venrich = channel_vector(q_emb_enriched, CONFIG["budget_vector_enriched"])  if VECTOR_OK else []
    # Multi-aspect HyDE: per-aspect dense search + RRF fusion -> single channel.
    if VECTOR_OK and q_emb_hyde_list:
        _aspect_hits = [channel_vector(_emb, CONFIG["budget_vector_hyde"])
                        for _emb in q_emb_hyde_list]
        _rrf_h = defaultdict(float)
        for _hits in _aspect_hits:
            for _rank, (_did, _) in enumerate(_hits):
                _rrf_h[_did] += 1.0 / (60 + _rank + 1)
        ch_vhyde = sorted(_rrf_h.items(), key=lambda kv: (-kv[1], kv[0]))[:CONFIG["budget_vector_hyde"]]
        if verbose:
            print(f"  vector_hyde: {len(q_emb_hyde_list)} aspect(s) -> "
                  f"{len(ch_vhyde):,} fused hits")
    else:
        ch_vhyde = []

    # ---------- PRF: aggregate signals from top-K pass-1 court dids ----------
    if CONFIG.get("enable_prf", True):
        _ensure_doc_reverse_maps()

        # Mini-RRF across pass-1 channels (skip law_direct since we want court rows).
        _pass1_for_prf = [
            ch_court_stat, ch_concept, ch_term, ch_per_area, ch_cocit,
            ch_bm25, ch_vector, ch_venrich, ch_vhyde,
        ]
        _mini_score = defaultdict(float)
        _RRF_K = 60
        for _hits in _pass1_for_prf:
            for _rank, (_did, _) in enumerate(_hits):
                _mini_score[_did] += 1.0 / (_RRF_K + _rank + 1)
        _prf_ranked = sorted(_mini_score.items(), key=lambda x: -x[1])
        _K_pool = CONFIG.get("prf_top_k_for_extraction", 2000)
        prf_court_dids = []
        for _did, _ in _prf_ranked:
            if doc_meta.get(_did, {}).get("family") == "court":
                prf_court_dids.append(_did)
                if len(prf_court_dids) >= _K_pool:
                    break

        # Aggregate annotations from prf_court_dids.
        _stat_ct = Counter()
        _conc_ct = Counter()
        _term_ct = Counter()
        for _did in prf_court_dids:
            for _canon in doc_statute_anchors.get(_did, ()) or ():
                _stat_ct[_canon] += 1
            for _c in _doc_to_concepts.get(_did, ()) or ():
                _conc_ct[_c] += 1
            for _t in _doc_to_terms.get(_did, ()) or ():
                _term_ct[_t] += 1

        # Existing target signals — we only want NEW ones.
        _existing_canons = set(co_expanded_canons)
        _existing_concept_keys = {c for c, _ in expanded_weighted}
        _existing_q_terms = set()
        for _key in ("term_targets_de", "term_targets_fr"):
            for _raw in (targets.get(_key) or []):
                _tok = norm_token(_raw, CONFIG["lowercase_terms"])
                if _tok:
                    _existing_q_terms.add(_tok)

        _K_stat = CONFIG.get("prf_new_statutes_k", 15)
        _K_conc = CONFIG.get("prf_new_concepts_k", 30)
        _K_term = CONFIG.get("prf_new_terms_k", 30)
        new_canons = [c for c, _ in _stat_ct.most_common() if c not in _existing_canons][:_K_stat]
        new_concepts = [c for c, _ in _conc_ct.most_common() if c not in _existing_concept_keys][:_K_conc]
        new_terms = [t for t, _ in _term_ct.most_common() if t not in _existing_q_terms][:_K_term]

        # Derive new codes from new canons (corpus-evident peripheral codes).
        new_codes = set()
        for _canon in new_canons:
            if " " in _canon:
                new_codes.add(_canon.split()[1])

        if verbose:
            print(f"  PRF pool: {len(prf_court_dids):,} pass-1 court dids  "
                  f"-> +{len(new_canons)} canons, +{len(new_codes)} codes, "
                  f"+{len(new_concepts)} concepts, +{len(new_terms)} terms")
            if new_canons:
                print(f"    new canons (top 8): {new_canons[:8]}")
            if new_codes:
                print(f"    new codes:          {sorted(new_codes)}")

        # ---------- PASS 2: re-run topical channels with augmented targets ----------
        augmented_canons_co       = co_expanded_canons | set(new_canons)
        augmented_canons_court    = llm_statute_canons | set(new_canons)
        augmented_codes           = statute_target_codes | new_codes
        # new concepts entered as low-weight (0.5) so they're admitted but don't
        # outscore strong LLM-named ones.
        augmented_weighted        = list(expanded_weighted) + [(c, 0.5) for c in new_concepts]

        # term channel: inject new_terms into term_targets_de (language-agnostic
        # via the lemma+substring matcher, so a wrong-language label is harmless).
        augmented_targets = dict(targets)
        augmented_targets["term_targets_de"] = list(targets.get("term_targets_de", [])) + new_terms

        # Re-run the canon/concept/term/area/cocit channels with augmented inputs.
        ch_law_direct2 = channel_law_direct(augmented_canons_co, idx_law_direct, CONFIG["budget_law_direct"])
        ch_court_stat2 = channel_court_statute(augmented_canons_court, idx_court_statute,
                                                idx_court_statute_count, doc_meta,
                                                CONFIG["budget_court_statute"])
        ch_concept2    = channel_concept(augmented_weighted, idx_concept_en, CONFIG["budget_concept"])
        ch_term2       = channel_term(augmented_targets, idx_term_orig, idx_term_lemma,
                                       CONFIG["budget_term"], corpus_keys=term_orig_keys)
        ch_per_area2   = channel_per_area_bedrock(targets.get("legal_area_keywords", []),
                                                    augmented_codes, per_area_canon_count,
                                                    idx_law_direct, CONFIG["budget_per_area"])
        # co_citation reads statute_targets via canonicalization; build an augmented dict.
        ch_cocit2      = channel_co_citation(
            {"statute_targets": [c for c in augmented_canons_court]},
            co_neighbours, idx_law_direct, idx_court_statute,
            CONFIG["budget_co_citation"])

        # ---------- merge pass-1 + pass-2 topical channels ----------
        # _p1 references stay readable so the seed-pool / graph code below can
        # build BOTH the pass-1 seed (original) AND the pass-2 seed (expanded).
        ch_law_direct_p1, ch_law_direct = ch_law_direct, _merge_channel(ch_law_direct, ch_law_direct2)
        ch_court_stat_p1, ch_court_stat = ch_court_stat, _merge_channel(ch_court_stat, ch_court_stat2)
        ch_concept_p1,    ch_concept    = ch_concept,    _merge_channel(ch_concept,    ch_concept2)
        ch_term_p1,       ch_term       = ch_term,       _merge_channel(ch_term,       ch_term2)
        ch_per_area_p1,   ch_per_area   = ch_per_area,   _merge_channel(ch_per_area,   ch_per_area2)
        ch_cocit_p1,      ch_cocit      = ch_cocit,      _merge_channel(ch_cocit,      ch_cocit2)
    else:
        # PRF disabled — pass-1 == merged.
        ch_law_direct_p1 = ch_law_direct
        ch_court_stat_p1 = ch_court_stat
        ch_concept_p1    = ch_concept
        ch_term_p1       = ch_term
        ch_per_area_p1   = ch_per_area
        ch_cocit_p1      = ch_cocit

    # ---------- seed pools ----------
    def _court_hits(hits):
        return {d for d, _ in hits if doc_meta.get(d, {}).get("family") == "court"}
    seed_p1 = (
        _court_hits(ch_court_stat_p1) | _court_hits(ch_law_direct_p1)
      | _court_hits(ch_concept_p1)    | _court_hits(ch_term_p1)
      | _court_hits(ch_per_area_p1)   | _court_hits(ch_cocit_p1)
      | _court_hits(ch_bm25)
      | _court_hits(ch_vector)        | _court_hits(ch_venrich)
      | _court_hits(ch_vhyde)
    )
    # Expanded seed: union with pass-2 hits (only differs from seed_p1 if PRF
    # enabled). When PRF off, seed_p2 == seed_p1 and the second graph pass is
    # idempotent — the merge call below is cheap.
    seed_p2 = seed_p1 | (
        _court_hits(ch_court_stat) | _court_hits(ch_law_direct)
      | _court_hits(ch_concept)    | _court_hits(ch_term)
      | _court_hits(ch_per_area)   | _court_hits(ch_cocit)
    )

    # ---------- sibling / graph / backprop  (DUAL-PASS, then merge) ----------
    # Pass-1 seed: preserves original results so PRF can never DROP gold past
    # a budget cap. Pass-2 seed: gives PRF a chance to ADD new candidates.
    # Final per channel = union (max score per doc).
    ch_sibling_p1 = channel_sibling(seed_p1, idx_court_base, idx_judgment_importance,
                                     doc_meta, CONFIG["budget_sibling"])
    ch_sibling_p2 = channel_sibling(seed_p2, idx_court_base, idx_judgment_importance,
                                     doc_meta, CONFIG["budget_sibling"])
    ch_sibling = _merge_channel(ch_sibling_p1, ch_sibling_p2)

    if GRAPH_OK:
        ch_graph_fwd_p1 = channel_graph_forward(seed_p1, idx_graph_out, idx_judgment_importance,
                                                 doc_meta, CONFIG["budget_graph_forward"])
        ch_graph_fwd_p2 = channel_graph_forward(seed_p2, idx_graph_out, idx_judgment_importance,
                                                 doc_meta, CONFIG["budget_graph_forward"])
        ch_graph_fwd = _merge_channel(ch_graph_fwd_p1, ch_graph_fwd_p2)

        ch_graph_rev_p1 = channel_graph_reverse(seed_p1, idx_graph_in, idx_judgment_importance,
                                                 doc_meta, CONFIG["budget_graph_reverse"])
        ch_graph_rev_p2 = channel_graph_reverse(seed_p2, idx_graph_in, idx_judgment_importance,
                                                 doc_meta, CONFIG["budget_graph_reverse"])
        ch_graph_rev = _merge_channel(ch_graph_rev_p1, ch_graph_rev_p2)

        ch_graph_2h = (channel_graph_2hop(seed_p2, idx_graph_out, CONFIG["budget_graph_2hop"])
                        if CONFIG["enable_graph_2hop"] else [])
    else:
        ch_graph_fwd = []; ch_graph_rev = []; ch_graph_2h = []

    backprop_seed_p1 = seed_p1 | _court_hits(ch_sibling_p1) | (
        _court_hits(ch_graph_fwd_p1) if GRAPH_OK else set()) | (
        _court_hits(ch_graph_rev_p1) if GRAPH_OK else set())
    backprop_seed_p2 = seed_p2 | _court_hits(ch_sibling) | _court_hits(ch_graph_fwd) | _court_hits(ch_graph_rev)
    ch_backprop_p1 = channel_statute_backprop(backprop_seed_p1, doc_statute_anchors, idx_law_direct,
                                                CONFIG["budget_backprop"])
    ch_backprop_p2 = channel_statute_backprop(backprop_seed_p2, doc_statute_anchors, idx_law_direct,
                                                CONFIG["budget_backprop"])
    ch_backprop = _merge_channel(ch_backprop_p1, ch_backprop_p2)

    CHANNELS = [
        ("law_direct_match",  ch_law_direct),
        ("court_statute",     ch_court_stat),
        ("co_citation",       ch_cocit),
        ("per_area_bedrock",  ch_per_area),
        ("statute_backprop",  ch_backprop),
        ("sibling_expansion", ch_sibling),
        ("graph_forward",     ch_graph_fwd),
        ("graph_reverse",     ch_graph_rev),
        ("graph_2hop",        ch_graph_2h),
        ("concept_en",        ch_concept),
        ("term_orig",         ch_term),
        ("bm25",              ch_bm25),
        ("vector_raw",        ch_vector),
        ("vector_enriched",   ch_venrich),
        ("vector_hyde",       ch_vhyde),
    ]

    channel_recalls = {}
    for name, hits in CHANNELS:
        found = {d for d, _ in hits}
        g = len(found & gold_doc_set) if gold_doc_set else 0
        channel_recalls[name] = (g, len(hits), g / max(1, total_gold))

    union_did = set()
    for _, hits in CHANNELS:
        union_did.update(d for d, _ in hits)
    union_gold = len(union_did & gold_doc_set) if gold_doc_set else 0

    if verbose:
        print(f"  {'channel':<22}  {'size':>6}  {'gold_in_ch':>11}  recall")
        print("  " + "-" * 58)
        for name in [c for c, _ in CHANNELS]:
            g, sz, r = channel_recalls[name]
            print(f"  {name:<22}  {sz:>6}  {g:>11}  {100*r:5.1f}%")
        print(f"  Union: {len(union_did):,} unique doc_ids   gold-in-union: {union_gold}/{total_gold}")

    return {
        "channels": CHANNELS,
        "channel_recalls": channel_recalls,
        "union_size": len(union_did),
        "union_gold": union_gold,
        "seed_size": len(seed_p2),
    }

print("run_channels() defined (with corpus-side PRF).")


run_channels() defined (with corpus-side PRF).


# Phase 9 — RRF fusion + master loop

Three pieces defined inline:

1. **`rrf_fuse(channels, k, weights)`** — weighted reciprocal-rank fusion: `score(did) = sum_ch weights[ch] / (k + rank + 1)`.
2. **`apply_neg_gate(doc_ids, doc_meta, noise_roles)`** — drops noisy paragraph roles (e.g. `notification`, `header`, `empty`, `metadata`) UNLESS the doc carries a substantive role like `reasoning` / `facts` / `legal_standard` / `application` / `holding` / `citation` / `procedural_history` — those override the buggy `is_notification_paragraph` flag.
3. **`round_robin_guarantee(...)`** — round-robin emission from guarantee channels, up to `per_channel_cap` each, until `total_cap` reached. Ensures every high-recall channel contributes regardless of weighted-RRF score.
4. **`fuse_for_query(channels, gold_doc_set, total_gold)`** — runs RRF → gate → guarantee → fill from RRF tail → truncate at `topk_final = 50,000`. Also samples the post-gate union at K ∈ {50, 100, …, 25000, 35000, 50000} for the macro R@K curve.

The **master loop** then iterates `ALL_QUERIES` and calls `run_channels` + `fuse_for_query` per query. Per-query results accumulate into `PER_QUERY[qid]` with `channel_recalls`, `channel_hit_sets`, `missed_gold_dids`, `R_at_K`, the K-curve, and the `final_topk` list. Identical code path for every row in val.csv.


In [ ]:
def rrf_fuse(channels, k, weights=None):
    """v7.4 weighted RRF: score(did) = sum_ch weights[ch] / (k + rank + 1)."""
    if weights is None: weights = {}
    score = defaultdict(float)
    for name, hits in channels:
        w = weights.get(name, 1.0)
        if w == 0: continue
        for rank, (did, _) in enumerate(hits):
            score[did] += w / (k + rank + 1)
    return score

SUBSTANTIVE_ROLES = {
    "facts", "reasoning", "legal_standard", "application",
    "holding", "citation", "procedural_history",
}

def apply_neg_gate(doc_ids, doc_meta, noise_roles):
    keep = []
    for did in doc_ids:
        m = doc_meta.get(did) or {}
        pr = (m.get("paragraph_role") or "").lower()
        if pr in SUBSTANTIVE_ROLES:
            keep.append(did); continue
        if m.get("is_notification_paragraph"): continue
        if pr in noise_roles: continue
        keep.append(did)
    return keep

def round_robin_guarantee(channels_by_name, guarantee_channel_names,
                           per_channel_cap, total_cap):
    iters = {cn: iter(channels_by_name.get(cn, [])) for cn in guarantee_channel_names}
    counts = {cn: 0 for cn in guarantee_channel_names}
    out = []; seen = set()
    while iters and len(out) < total_cap:
        exhausted = []
        for cn in list(iters.keys()):
            if counts[cn] >= per_channel_cap:
                exhausted.append(cn); continue
            try:
                did, _ = next(iters[cn])
                while did in seen:
                    did, _ = next(iters[cn])
                out.append(did); seen.add(did); counts[cn] += 1
                if len(out) >= total_cap: break
            except StopIteration:
                exhausted.append(cn)
        for cn in exhausted:
            if cn in iters: del iters[cn]
    return out

# R@K sampling points — fixed list, identical for every query.
K_SAMPLES = [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000,
             5000, 7500, 10000, 15000, 20000, 25000, 35000, 50000]

def fuse_for_query(CHANNELS, gold_doc_set, total_gold):
    rrf_scores = rrf_fuse(CHANNELS, CONFIG["rrf_k"], weights=CONFIG.get("channel_weights"))
    ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    ranked_dids = [d for d, _ in ranked]
    ranked_gated = apply_neg_gate(ranked_dids, doc_meta, CONFIG["noise_paragraph_roles"])

    ch_by_name = dict(CHANNELS)
    guarantee = round_robin_guarantee(
        ch_by_name, CONFIG["guarantee_channels"],
        CONFIG.get("guarantee_per_channel", 400),
        CONFIG["topk_final"],
    )
    guarantee = apply_neg_gate(guarantee, doc_meta, CONFIG["noise_paragraph_roles"])

    PASS_K = CONFIG["topk_final"]
    final_topk = list(guarantee); seen_f = set(final_topk)
    for did in ranked_gated:
        if len(final_topk) >= PASS_K: break
        if did not in seen_f:
            final_topk.append(did); seen_f.add(did)
    final_topk = final_topk[:PASS_K]

    all_ranked = guarantee + [d for d in ranked_gated if d not in set(guarantee)]
    curve = {}
    for K in K_SAMPLES:
        K_use = min(K, len(all_ranked))
        g = len(set(all_ranked[:K_use]) & gold_doc_set) if gold_doc_set else 0
        curve[K_use] = (g, g / max(1, total_gold))

    union_after_gate = set(all_ranked)
    return {
        "final_topk": final_topk,
        "gold_in_top": len(set(final_topk) & gold_doc_set) if gold_doc_set else 0,
        "R_at_K": (len(set(final_topk) & gold_doc_set) / max(1, total_gold)) if gold_doc_set else 0.0,
        "curve": curve,
        "pre_gate_size": len(ranked_dids),
        "post_gate_size": len(ranked_gated),
        "guarantee_size": len(guarantee),
        "union_after_gate_size": len(union_after_gate),
        "union_after_gate_gold": len(union_after_gate & gold_doc_set) if gold_doc_set else 0,
    }

# Master loop — iterate every query in ALL_QUERIES with identical code path.
import time as _time
print("=" * 80)
print(f"Running pipeline on {len(ALL_QUERIES)} queries  (topk_final={CONFIG['topk_final']:,})")
print("=" * 80)
PER_QUERY = {}
for q in ALL_QUERIES:
    qid = q["query_id"]
    g_set = ALL_GOLD_DOC_SET[qid]
    g_tot = ALL_TOTAL_GOLD[qid]
    t0 = _time.time()
    print(f"\n[{qid}]  gold={g_tot}  gold_doc_ids={len(g_set)}")
    ch_out = run_channels(q["query_text"], ALL_TARGETS[qid],
                          ALL_Q_EMB_RAW[qid], ALL_Q_EMB_ENRICHED[qid],
                          ALL_Q_EMB_HYDE_LIST[qid],
                          g_set, g_tot, verbose=True)
    fuse_out = fuse_for_query(ch_out["channels"], g_set, g_tot)
    # Per-channel doc_id sets — needed for missed-gold diagnostic.
    chan_hit_sets = {name: {d for d, _ in hits} for name, hits in ch_out["channels"]}
    union_did_set = set()
    for s in chan_hit_sets.values():
        union_did_set |= s
    missed_gold = (g_set - union_did_set) if g_set else set()
    PER_QUERY[qid] = {
        "gold": g_tot,
        "gold_doc_ids": len(g_set),
        "channel_recalls": ch_out["channel_recalls"],
        "channel_hit_sets": chan_hit_sets,
        "union_did_set": union_did_set,
        "missed_gold_dids": sorted(missed_gold),
        "union_size": ch_out["union_size"],
        "union_gold": ch_out["union_gold"],
        "post_gate_union_size": fuse_out["union_after_gate_size"],
        "post_gate_union_gold": fuse_out["union_after_gate_gold"],
        "gold_in_top": fuse_out["gold_in_top"],
        "R_at_K": fuse_out["R_at_K"],
        "curve": fuse_out["curve"],
        "final_topk": fuse_out["final_topk"],
    }
    print(f"  -> R@{CONFIG['topk_final']:,} = {fuse_out['R_at_K']:.3f}  "
          f"({fuse_out['gold_in_top']}/{g_tot})    "
          f"post-gate union gold = {fuse_out['union_after_gate_gold']}/{g_tot}    "
          f"({_time.time()-t0:.1f}s)")


Running pipeline on 10 queries  (topk_final=50,000)

[val_001]  gold=42  gold_doc_ids=42
  vector_hyde: 5 aspect(s) -> 2,000 fused hits
  PRF pool: 4,000 pass-1 court dids  -> +40 canons, +6 codes, +60 concepts, +60 terms
    new canons (top 8): ['29 BV', '9 BV', '13 BV', '26 BV', '8 EMRK', '106 BGG', '27 BV', '42 BGG']
    new codes:          ['BGG', 'BV', 'CEDH', 'EMRK', 'GE', 'StPO']
  channel                   size   gold_in_ch  recall
  ----------------------------------------------------------
  law_direct_match           218            7   16.7%
  court_statute            15631           12   28.6%
  co_citation               2500            0    0.0%
  per_area_bedrock          1500           17   40.5%
  statute_backprop          2390           17   40.5%
  sibling_expansion         5319            4    9.5%
  graph_forward             7281            9   21.4%
  graph_reverse             3104            2    4.8%
  graph_2hop                   0            0    0.0%
  concept

# Phase 10 — Aggregate per-query table

Per-query summary: gold, R@K, caught/gold, channel-union gold, post-gate-union gold. Plus macro mean and micro-pooled recall across all queries.


In [ ]:
# Aggregation header — per-query table + macro mean.
print("=" * 80)
print(f"AGGREGATE SUMMARY  (topk_final={CONFIG['topk_final']:,})")
print("=" * 80)
print()
hdr = f"{'query':<10}  {'gold':>4}  {'R@K':>7}  {'caught':>9}  {'union':>9}  {'gate_union':>11}"
print(hdr)
print("-" * len(hdr))
sum_R = 0.0; sum_union_gold = 0; sum_gate_union_gold = 0; sum_gold = 0
for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    print(f"{qid:<10}  {r['gold']:>4}  {r['R_at_K']:>7.3f}  "
          f"{r['gold_in_top']:>3}/{r['gold']:<5}  "
          f"{r['union_gold']:>3}/{r['gold']:<5}  "
          f"{r['post_gate_union_gold']:>3}/{r['gold']:<7}")
    sum_R += r["R_at_K"]
    sum_union_gold += r["union_gold"]
    sum_gate_union_gold += r["post_gate_union_gold"]
    sum_gold += r["gold"]
n = max(1, len(PER_QUERY))
print("-" * len(hdr))
print(f"{'MEAN':<10}        {sum_R/n:>7.3f}                  "
      f"{sum_union_gold:>3}/{sum_gold:<5}  "
      f"{sum_gate_union_gold:>3}/{sum_gold:<7}   "
      f"(micro union={sum_union_gold/max(1,sum_gold):.3f}, "
      f"gate union={sum_gate_union_gold/max(1,sum_gold):.3f})")


AGGREGATE SUMMARY  (topk_final=50,000)

query       gold      R@K     caught      union   gate_union
------------------------------------------------------------
val_001       42    0.929   39/42      39/42      39/42     
val_002       36    0.806   29/36      29/36      29/36     
val_003       47    0.766   36/47      36/47      36/47     
val_004       10    1.000   10/10      10/10      10/10     
val_005       11    1.000   11/11      11/11      11/11     
val_006       18    0.944   17/18      17/18      17/18     
val_007       19    0.895   17/19      17/19      17/19     
val_008       29    0.862   25/29      25/29      25/29     
val_009       14    0.929   13/14      13/14      13/14     
val_010       25    0.880   22/25      22/25      22/25     
------------------------------------------------------------
MEAN                0.901                  219/251    219/251       (micro union=0.873, gate union=0.873)


## 10.2 Macro mean R@K curve


In [ ]:
# Macro mean R@K curve across all queries.
print("=" * 80)
print("MACRO MEAN R@K CURVE (averaged across all queries)")
print("=" * 80)
all_K = sorted({K for r in PER_QUERY.values() for K in r["curve"].keys()})
print(f"{'K':>6}  {'mean recall':>12}  {'min':>7}  {'max':>7}")
print("-" * 40)
for K in all_K:
    rs = []
    for r in PER_QUERY.values():
        keys = sorted(r["curve"].keys())
        kk = max((k for k in keys if k <= K), default=None)
        if kk is not None:
            rs.append(r["curve"][kk][1])
    if rs:
        print(f"{K:>6}  {sum(rs)/len(rs):>11.3f}   "
              f"{min(rs):>6.3f}  {max(rs):>6.3f}")


MACRO MEAN R@K CURVE (averaged across all queries)
     K   mean recall      min      max
----------------------------------------
    50        0.052    0.000   0.194
   100        0.121    0.000   0.273
   200        0.179    0.040   0.455
   300        0.353    0.128   0.643
   500        0.415    0.128   0.643
   750        0.436    0.170   0.643
  1000        0.530    0.170   0.800
  1500        0.571    0.234   0.900
  2000        0.616    0.255   0.900
  3000        0.663    0.383   0.900
  5000        0.728    0.404   0.909
  7500        0.768    0.489   0.909
 10000        0.822    0.617   1.000
 15000        0.839    0.638   1.000
 20000        0.852    0.660   1.000
 25000        0.867    0.681   1.000
 35000        0.891    0.745   1.000
 40748        0.891    0.745   1.000
 40953        0.891    0.745   1.000
 41310        0.894    0.745   1.000
 41408        0.894    0.745   1.000
 43179        0.894    0.745   1.000
 43992        0.894    0.745   1.000
 44138        0.89

## 10.3 Per-channel mean recall


In [ ]:
# Per-channel mean recall across all queries.
print("=" * 80)
print("PER-CHANNEL MEAN RECALL (across all queries)")
print("=" * 80)
all_channels = list(next(iter(PER_QUERY.values()))["channel_recalls"].keys())
print(f"{'channel':<22}  {'mean recall':>11}  {'mean size':>10}")
print("-" * 50)
ranked = []
for cn in all_channels:
    rs = [PER_QUERY[qid]["channel_recalls"].get(cn, (0, 0, 0.0))[2] for qid in PER_QUERY]
    sizes = [PER_QUERY[qid]["channel_recalls"].get(cn, (0, 0, 0.0))[1] for qid in PER_QUERY]
    ranked.append((cn, sum(rs)/len(rs), sum(sizes)/len(sizes)))
ranked.sort(key=lambda x: -x[1])
for cn, r, sz in ranked:
    print(f"{cn:<22}  {r:>10.3f}  {sz:>9.0f}")


PER-CHANNEL MEAN RECALL (across all queries)
channel                 mean recall   mean size
--------------------------------------------------
statute_backprop             0.592       2417
law_direct_match             0.395        168
graph_forward                0.344       7246
vector_hyde                  0.340       2000
per_area_bedrock             0.330       1060
vector_enriched              0.220       2000
vector_raw                   0.213       2000
concept_en                   0.116       5348
bm25                         0.097       2000
court_statute                0.082      11715
term_orig                    0.062       4582
sibling_expansion            0.037       5813
co_citation                  0.031       2040
graph_reverse                0.021       3472
graph_2hop                   0.000          0


# Phase 10.4 — Missed-gold diagnostic

For every query where R@K < 1.0, list the gold doc_ids that **no channel returned** (i.e. fell through every one of the 15 channels). For each missed doc: citation, family (law / court), language, court_base, paragraph_role, and a 300-char text excerpt — so the failure type (procedural / multi-aspect / cross-lingual / etc.) is visible without re-reading the corpus.

Plus aggregate buckets — by family, language, paragraph_role, and court_base prefix — for the whole batch.

The point is to choose the next fix based on observed misses, not speculation.


In [ ]:
# Missed-gold diagnostic — what fell through every channel?
print("=" * 80)
print("MISSED-GOLD DIAGNOSIS  (gold doc_ids absent from every channel)")
print("=" * 80)

# Aggregate breakdown buckets.
from collections import Counter as _C
agg_family = _C(); agg_lang = _C(); agg_role = _C(); agg_courtbase_kind = _C()
total_missed = 0

for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    missed = r.get("missed_gold_dids", [])
    if not missed:
        print(f"\n[{qid}]  R@K={r['R_at_K']:.3f}   no missed gold")
        continue
    total_missed += len(missed)
    print(f"\n[{qid}]  R@K={r['R_at_K']:.3f}   gold={r['gold']}   "
          f"missed={len(missed)}   (channels all dark for these doc_ids)")
    for did in missed:
        m = doc_meta.get(did, {}) or {}
        cit  = m.get("citation", did)
        fam  = m.get("family", "?")
        lang = (m.get("language") or m.get("language_code") or "?")
        cb   = m.get("court_base") or ""
        pr   = (m.get("paragraph_role") or "")
        txt  = (search_text.get(did, "") or "")[:300].replace("\n", " ")
        agg_family[fam] += 1
        agg_lang[lang] += 1
        agg_role[pr or "(none)"] += 1
        # Court-base kind: e.g. BGE / 1B / 1C / 2C / district court / etc.
        if cb:
            kind = cb.split("_")[0] if "_" in cb else (cb.split()[0] if " " in cb else cb)
            agg_courtbase_kind[kind] += 1
        else:
            agg_courtbase_kind["(no court_base)"] += 1
        print(f"   - {cit}")
        print(f"       family={fam}  lang={lang}  court_base={cb}  paragraph_role={pr!r}")
        if txt:
            print(f"       text: {txt}")

print()
print("=" * 80)
print(f"AGGREGATE MISSED-GOLD BREAKDOWN  (total missed across all queries: {total_missed})")
print("=" * 80)
print("\nBy family:")
for k, n in agg_family.most_common():
    print(f"  {k:<10}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy language:")
for k, n in agg_lang.most_common():
    print(f"  {k:<10}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy paragraph_role:")
for k, n in agg_role.most_common():
    print(f"  {k:<25}  {n}  ({100*n/max(1,total_missed):.1f}%)")
print("\nBy court_base prefix (BGE / 1B / 5A / ... — empty means law row):")
for k, n in agg_courtbase_kind.most_common():
    print(f"  {k:<20}  {n}  ({100*n/max(1,total_missed):.1f}%)")


MISSED-GOLD DIAGNOSIS  (gold doc_ids absent from every channel)

[val_001]  R@K=0.929   gold=42   missed=3   (channels all dark for these doc_ids)
   - 7B_496/2025 E. 3.2
       family=court  lang=?  court_base=7B_496/2025  paragraph_role='procedural_history'
       text: 7B_496/2025 E. 3.2 3.2. Die Vorinstanz sieht weiterhin Kollusionsmöglichkeiten für den Beschwerdeführer. Zwar sei die Strafuntersuchung bereits weit fortgeschritten. Dennoch seien die zwei Mobiltelefone des Beschwerdeführers noch immer versiegelt und das Entsiegelungsverfahren sei noch im Gang. Bei 
   - Art. 37 Abs. 1 StBOG
       family=law  lang=?  court_base=  paragraph_role=''
       text: Art. 37 Abs. 1 StBOG The federal criminal courts' appellate chambers decide appeals referred to them by the StPO as competent. Federal criminal courts' appellate chambers decide appeals referred to them by the StPO. Which appeals are decided by the federal criminal courts' appellate chambers? Appeal
   - Art. 39 Abs. 1 StBOG
  

# Phase 11 — Save artifacts

Writes per-query `final_topk_<qid>.json` + aggregate `summary_multiquery.json` (macro mean, micro recall, per-query channel recalls, R@K curve), plus snapshots of `ALL_TARGETS` and the live `CONFIG`. All persisted under `PATHS["out_dir"]` for downstream reuse.


In [ ]:
import json as _json
out_dir = PATHS["out_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

did_to_cit = {did: m.get("citation", did) for did, m in doc_meta.items()}

# Per-query final top-K
for qid, r in PER_QUERY.items():
    final_top_records = [
        {"rank": i, "doc_id": did, "citation": did_to_cit.get(did, did)}
        for i, did in enumerate(r["final_topk"])
    ]
    (out_dir / f"final_topk_{qid}.json").write_text(
        _json.dumps(final_top_records, ensure_ascii=False), encoding="utf-8")

# Aggregate summary
n = max(1, len(PER_QUERY))
sum_gold = sum(r["gold"] for r in PER_QUERY.values())
sum_caught = sum(r["gold_in_top"] for r in PER_QUERY.values())
sum_union = sum(r["union_gold"] for r in PER_QUERY.values())
sum_gate_union = sum(r["post_gate_union_gold"] for r in PER_QUERY.values())
summary = {
    "topk_final": CONFIG["topk_final"],
    "n_queries": len(PER_QUERY),
    "macro_mean_R_at_K": sum(r["R_at_K"] for r in PER_QUERY.values()) / n,
    "micro_R_at_K": sum_caught / max(1, sum_gold),
    "micro_union_recall": sum_union / max(1, sum_gold),
    "micro_gate_union_recall": sum_gate_union / max(1, sum_gold),
    "per_query": {
        qid: {
            "gold": r["gold"],
            "gold_doc_ids": r["gold_doc_ids"],
            "R_at_K": r["R_at_K"],
            "gold_in_top": r["gold_in_top"],
            "union_gold": r["union_gold"],
            "post_gate_union_gold": r["post_gate_union_gold"],
            "channel_recalls": {
                cn: {"gold": v[0], "size": v[1], "recall": v[2]}
                for cn, v in r["channel_recalls"].items()
            },
            "curve": {str(K): {"gold": v[0], "recall": v[1]} for K, v in r["curve"].items()},
        }
        for qid, r in PER_QUERY.items()
    },
}
(out_dir / "summary_multiquery.json").write_text(
    _json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

(out_dir / "targets_multiquery.json").write_text(
    _json.dumps(ALL_TARGETS, ensure_ascii=False, indent=2), encoding="utf-8")

(out_dir / "config.json").write_text(_json.dumps(
    {k: v for k, v in CONFIG.items() if not isinstance(v, set)},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")

print(f"Wrote artifacts to {out_dir}")
print(f"  summary_multiquery.json")
print(f"  targets_multiquery.json")
print(f"  config.json")
print(f"  final_topk_<query_id>.json   x {len(PER_QUERY)}")


Wrote artifacts to /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7
  summary_multiquery.json
  targets_multiquery.json
  config.json
  final_topk_<query_id>.json   x 10


## 11.3 Cascade snapshot — one-time save for future warm-boot

Persists the corpus subset + per-query state needed by the cascade in cells 48+ to a new `snapshot/` subdirectory under `PATHS["out_dir"]`. Run this **once**, immediately after cell 46 finishes, in the same kernel session.

After that, the **Warm-Boot** cell below can reload everything in a future session — letting you skip cells 0-46 (the ~25 minute upstream pipeline) and jump straight to the cascade.

What gets saved (≈80-150 MB total):
- `corpus_snapshot.json.gz` — `search_text` + metadata + statute anchors + concepts + terms for the ~30-40k unique candidates in top-5000 of any query
- `per_query_snapshot.json` — `final_topk` (top-5000 per query) + R@K curves + channel recalls + gold counts
- `hyde_aspects.json` — multi-aspect HyDE paragraphs (Stage 4 aspect coverage uses these)
- `all_targets.json` — LLM-decomposed targets per query (Stage 2/3 prompts)
- `gold_doc_sets.json` — gold doc_id sets per query
- `config.json` — CONFIG snapshot
- `paths.json` — val.csv path pointer


In [ ]:
# Cascade Snapshot — saves corpus subset + per-query state for warm-boot.
# Run ONCE after cell 46 completes. Future sessions use the Warm-Boot cell.

import json as _j_snap, gzip as _gz_snap
from pathlib import Path
from collections import defaultdict as _dd_snap

SNAPSHOT_DIR = PATHS["out_dir"] / "snapshot"
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[snapshot] saving to {SNAPSHOT_DIR}")

# Union of doc_ids the cascade will see (top-5000 per query)
_cascade_dids = set()
for _qid, _r in PER_QUERY.items():
    _cascade_dids.update(_r["final_topk"][:5000])
print(f"  cascade pool: {len(_cascade_dids):,} unique doc_ids")

# Build _doc_to_concepts / _doc_to_terms reverse maps if not yet (used by Stages 2/3)
if "_doc_to_concepts" not in dir() or globals().get("_doc_to_concepts") is None:
    _doc_to_concepts = _dd_snap(set)
    for _k, _dids in idx_concept_en.items():
        for _d in _dids:
            _doc_to_concepts[_d].add(_k)
    _doc_to_terms = _dd_snap(set)
    for _k, _dids in idx_term_orig.items():
        for _d in _dids:
            _doc_to_terms[_d].add(_k)
    print(f"  built reverse maps: {len(_doc_to_concepts):,} concept-bearing, "
          f"{len(_doc_to_terms):,} term-bearing")

# Pack the minimal per-doc state the cascade reads
_corpus_snap = {}
for _did in _cascade_dids:
    _m = doc_meta.get(_did, {}) or {}
    _corpus_snap[_did] = {
        "ct":  (search_text.get(_did, "") or "")[:3000],   # corpus text
        "cit": _m.get("citation", _did) or _did,
        "fam": _m.get("family", "?"),
        "cb":  _m.get("court_base", "") or "",
        "pr":  _m.get("paragraph_role", "") or "",
        "ln":  _m.get("language", "?") or "?",
        "sa":  sorted(doc_statute_anchors.get(_did, set()) or set())[:15],
        "cn":  sorted(_doc_to_concepts.get(_did, set()))[:15],
        "tm":  sorted(_doc_to_terms.get(_did, set()))[:15],
    }
with _gz_snap.open(SNAPSHOT_DIR / "corpus_snapshot.json.gz", "wt", encoding="utf-8") as _f:
    _j_snap.dump(_corpus_snap, _f, ensure_ascii=False)
_sz_mb = (SNAPSHOT_DIR / "corpus_snapshot.json.gz").stat().st_size / 1024**2
print(f"  wrote corpus_snapshot.json.gz  ({_sz_mb:.1f} MB)")

# Per-query state (final_topk + curves + recalls)
_pq_snap = {}
for _qid, _r in PER_QUERY.items():
    _pq_snap[_qid] = {
        "final_topk":      _r["final_topk"][:5000],
        "curve":           {str(_k): list(_v) for _k, _v in _r.get("curve", {}).items()},
        "channel_recalls": {_ch: list(_v) for _ch, _v in _r.get("channel_recalls", {}).items()},
        "gold":            _r.get("gold", 0),
        "gold_doc_ids":    _r.get("gold_doc_ids", 0),
        "R_at_K":          _r.get("R_at_K", 0.0),
        "gold_in_top":     _r.get("gold_in_top", 0),
        "union_size":      _r.get("union_size", 0),
        "union_gold":      _r.get("union_gold", 0),
    }
(SNAPSHOT_DIR / "per_query_snapshot.json").write_text(
    _j_snap.dumps(_pq_snap, ensure_ascii=False), encoding="utf-8")
print(f"  wrote per_query_snapshot.json")

# HyDE aspects (Stage 4 aspect coverage)
if "ALL_HYDE_ASPECTS" in dir():
    (SNAPSHOT_DIR / "hyde_aspects.json").write_text(
        _j_snap.dumps(ALL_HYDE_ASPECTS, ensure_ascii=False), encoding="utf-8")
    print(f"  wrote hyde_aspects.json")

# LLM targets
if "ALL_TARGETS" in dir():
    (SNAPSHOT_DIR / "all_targets.json").write_text(
        _j_snap.dumps(ALL_TARGETS, ensure_ascii=False), encoding="utf-8")
    print(f"  wrote all_targets.json")

# Gold doc-id sets
(SNAPSHOT_DIR / "gold_doc_sets.json").write_text(
    _j_snap.dumps({_qid: sorted(_s) for _qid, _s in ALL_GOLD_DOC_SET.items()},
                   ensure_ascii=False),
    encoding="utf-8")
print(f"  wrote gold_doc_sets.json")

# CONFIG snapshot
(SNAPSHOT_DIR / "config.json").write_text(_j_snap.dumps(
    {_k: _v for _k, _v in CONFIG.items() if not isinstance(_v, set)},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(f"  wrote config.json")

# Paths pointer (val.csv location, for warm-boot)
(SNAPSHOT_DIR / "paths.json").write_text(_j_snap.dumps({
    "val_csv": str(PATHS["val_csv"]),
    "out_dir": str(PATHS["out_dir"]),
}, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"  wrote paths.json")

print(f"\n[snapshot] complete. {SNAPSHOT_DIR}")
print(f"[snapshot] Future sessions: run the Warm-Boot cell below to skip 0-46.")


[snapshot] saving to /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot
  cascade pool: 32,717 unique doc_ids
  wrote corpus_snapshot.json.gz  (14.8 MB)
  wrote per_query_snapshot.json
  wrote hyde_aspects.json
  wrote all_targets.json
  wrote gold_doc_sets.json
  wrote config.json
  wrote paths.json

[snapshot] complete. /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot
[snapshot] Future sessions: run the Warm-Boot cell below to skip 0-46.


## 11.4 Warm-Boot — load snapshot in a new session, skip cells 0-46

Run this cell ONCE at the start of a new session (after the snapshot above has been saved in a previous session). It reconstructs `PER_QUERY`, `search_text`, `doc_meta`, `doc_statute_anchors`, `_doc_to_concepts`, `_doc_to_terms`, `ALL_QUERIES`, `ALL_TARGETS`, `ALL_HYDE_ASPECTS`, `ALL_GOLD_DOC_SET`, `ALL_TOTAL_GOLD`, `CONFIG`, and `PATHS` from the snapshot in ~30-60 seconds.

After this cell runs, **jump straight to cell 50** (Stage 1 reranker, the first cascade cell). Cells 0-46 are not needed.

If `SNAPSHOT_DIR` below doesn't match where you saved, edit the path before running.


In [ ]:
# Warm-Boot — load saved snapshot and reconstruct all cascade-required state.
# Skip cells 0-46 after running this. Jump straight to Stage 1 (Phase 12).

import json as _j_wb, gzip as _gz_wb
from pathlib import Path
from collections import defaultdict as _dd_wb

# Mount Drive (Colab); no-op if already mounted or running locally.
try:
    from google.colab import drive as _drive_wb
    _drive_wb.mount("/content/drive")
except Exception:
    pass

# Snapshot location — adjust if you saved it elsewhere.
SNAPSHOT_DIR = Path(
    "/content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot"
)
if not SNAPSHOT_DIR.exists():
    raise SystemExit(
        f"[warm-boot] snapshot not found at {SNAPSHOT_DIR}\n"
        f"Run the Cascade Snapshot cell (11.3) in a prior session first."
    )
print(f"[warm-boot] loading from {SNAPSHOT_DIR}")

# CONFIG
CONFIG = _j_wb.loads((SNAPSHOT_DIR / "config.json").read_text(encoding="utf-8"))
print(f"  CONFIG loaded (topk_final={CONFIG.get('topk_final')})")

# PATHS
_paths_meta = _j_wb.loads((SNAPSHOT_DIR / "paths.json").read_text(encoding="utf-8"))
PATHS = {
    "val_csv": Path(_paths_meta["val_csv"]),
    "out_dir": Path(_paths_meta["out_dir"]),
}

# ALL_QUERIES from val.csv (only ~10 queries; trivial)
import pandas as pd
val_df = pd.read_csv(PATHS["val_csv"])
ALL_QUERIES = [
    {"query_id":   str(r["query_id"]),
     "query_text": str(r["query"]),
     "gold":       [c.strip() for c in str(r["gold_citations"]).split(";") if c.strip()]}
    for _, r in val_df.iterrows()
]
ALL_TOTAL_GOLD = {q["query_id"]: len(q["gold"]) for q in ALL_QUERIES}
print(f"  ALL_QUERIES: {len(ALL_QUERIES)}  (val.csv = {PATHS['val_csv']})")

# Corpus snapshot
with _gz_wb.open(SNAPSHOT_DIR / "corpus_snapshot.json.gz", "rt", encoding="utf-8") as _f:
    _c = _j_wb.load(_f)
search_text         = {_d: _r["ct"]  for _d, _r in _c.items()}
doc_meta            = {_d: {"citation":       _r["cit"],
                            "family":         _r["fam"],
                            "court_base":     _r["cb"],
                            "paragraph_role": _r["pr"],
                            "language":       _r["ln"]}
                       for _d, _r in _c.items()}
doc_statute_anchors = {_d: set(_r["sa"]) for _d, _r in _c.items()}
_doc_to_concepts    = {_d: set(_r["cn"]) for _d, _r in _c.items()}
_doc_to_terms       = {_d: set(_r["tm"]) for _d, _r in _c.items()}
print(f"  corpus_snapshot: {len(_c):,} docs")

# PER_QUERY
_pq = _j_wb.loads((SNAPSHOT_DIR / "per_query_snapshot.json").read_text(encoding="utf-8"))
PER_QUERY = {
    _qid: {
        "final_topk":      _r["final_topk"],
        "curve":           {int(_k): tuple(_v) for _k, _v in _r.get("curve", {}).items()},
        "channel_recalls": {_ch: tuple(_v) for _ch, _v in _r.get("channel_recalls", {}).items()},
        "gold":            _r.get("gold", 0),
        "gold_doc_ids":    _r.get("gold_doc_ids", 0),
        "R_at_K":          _r.get("R_at_K", 0.0),
        "gold_in_top":     _r.get("gold_in_top", 0),
        "union_size":      _r.get("union_size", 0),
        "union_gold":      _r.get("union_gold", 0),
    }
    for _qid, _r in _pq.items()
}
print(f"  PER_QUERY: {len(PER_QUERY)}")

# ALL_TARGETS
ALL_TARGETS = (_j_wb.loads((SNAPSHOT_DIR / "all_targets.json").read_text(encoding="utf-8"))
               if (SNAPSHOT_DIR / "all_targets.json").exists() else {})
print(f"  ALL_TARGETS: {len(ALL_TARGETS)}")

# ALL_HYDE_ASPECTS
ALL_HYDE_ASPECTS = (_j_wb.loads((SNAPSHOT_DIR / "hyde_aspects.json").read_text(encoding="utf-8"))
                    if (SNAPSHOT_DIR / "hyde_aspects.json").exists() else {})
print(f"  ALL_HYDE_ASPECTS: {len(ALL_HYDE_ASPECTS)}")

# ALL_GOLD_DOC_SET
_g = _j_wb.loads((SNAPSHOT_DIR / "gold_doc_sets.json").read_text(encoding="utf-8"))
ALL_GOLD_DOC_SET = {_qid: set(_lst) for _qid, _lst in _g.items()}
print(f"  ALL_GOLD_DOC_SET total: {sum(len(_s) for _s in ALL_GOLD_DOC_SET.values()):,} doc_ids")

# Sanity check — print a sample candidate's text
_sqid = ALL_QUERIES[0]["query_id"]
_sample_dids = PER_QUERY[_sqid]["final_topk"][:2]
print(f"\n  sanity check ({_sqid}, top-2 doc_ids = {_sample_dids}):")
for _d in _sample_dids:
    _t = (search_text.get(_d, "") or "")[:120].replace("\n", " ")
    print(f"    {_d}: \"{_t}...\"")

print()
print("[warm-boot] complete.")
print("[warm-boot] You can now run cell 50 onwards (Phase 12 cascade Stage 1).")
print("[warm-boot] Do NOT re-run cells 0-46 — their state is already loaded.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[warm-boot] loading from /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7/snapshot
  CONFIG loaded (topk_final=50000)
  ALL_QUERIES: 10  (val.csv = /content/drive/MyDrive/swiss_law/data/val.csv)
  corpus_snapshot: 32,717 docs
  PER_QUERY: 10
  ALL_TARGETS: 10
  ALL_HYDE_ASPECTS: 10
  ALL_GOLD_DOC_SET total: 254 doc_ids

  sanity check (val_001, top-2 doc_ids = ['law:128343', 'law:57057']):
    law:128343: "Art. 10 Abs. 1 StGB This article distinguishes between crimes and offenses based on the severity of penalties threatened..."
    law:57057: "Art. 66 Abs. 1 BGG The article states that court costs are generally imposed on the losing party, but the Federal Court ..."

[warm-boot] complete.
[warm-boot] You can now run cell 50 onwards (Phase 12 cascade Stage 1).
[warm-boot] Do NOT re-run cells 0-46 — their state is already loaded.


# Phase 12 — Hybrid 3-stage cascade (final F1 path)

The cells in Phases 0-11 produce the recall-ceiling pipeline: 15 channels → weighted RRF + neg-gate + 7-channel guarantee → `topk_final = 50,000`. The aggregate macro mean R@K=50k sits around 0.89 (218/251 gold across 10 queries).

That ceiling is the upper bound on what *any* reranking can achieve from this candidate pool — it tells us how much gold the channels could ever surface.

But for production / F1 we need a much shorter output (20-40 confident citations per query). The cascade below produces that **without modifying anything above** — it reads `PER_QUERY[qid]['final_topk']` and runs:

1. **Stage 1 — Cross-encoder rerank.** Qwen3-Reranker-8B (yes/no head) on the top-5000 fused candidates → top-500 by relevance probability.
2. **Stage 1B (optional, behind flag) — Translation A/B.** Translate top-200 candidates to English via Qwen3-32B, rerank, compare to original. Tests whether explicit MT helps a natively-multilingual reranker (literature: probably not, but we measure).
3. **Stage 2 — Listwise LLM scoring.** Qwen3-32B reads candidates in batches of 20 (each presented with full metadata: citation, statute anchors, concepts, terms, role, text excerpt) and scores each 1-5 with a grounded justification.
4. **Stage 3 — Pointwise evidence-quoted verdict.** Qwen3-32B does a deep judgment on top-100 — KEEP / MAYBE / DROP — with a verbatim evidence quote requirement. Without a quote, the candidate is forcibly DROP. This is the primary anti-hallucination gate.
5. **Stage 4 — Adaptive K + aspect coverage.** Selects top-K based on the confidence distribution (KEEP=5 always, KEEP=4 to a target, MAYBE only if needed), with a floor proportional to the number of decomposed query aspects, and an aspect-coverage check that promotes one candidate per uncovered aspect.

Each stage logs per-query R@K + timing so you can see where compression is gaining or losing recall.


In [ ]:
# =============================================================================
# Stage 1 — Cross-encoder rerank with Qwen3-Reranker-8B
# =============================================================================
# Read top-5000 candidates per query from PER_QUERY[qid]['final_topk'].
# Score each (query, candidate_text) pair with Qwen3-Reranker-8B's yes/no head.
# Output per-query: PER_QUERY[qid]['stage1_ranked'] = list[(did, score)] sorted
# desc by score, length up to 5000. Also stores R@K curve at K in
# {50,100,200,500,1000,2000,5000} so we can compare against the fusion baseline.
#
# Memory: frees EMB_MODEL + E_GPU first (vector retrieval is done) to reclaim
# ~38GB before loading reranker (~16GB bf16).
# =============================================================================

import gc
import time as _t_s1
import torch as _torch_s1
from transformers import AutoTokenizer, AutoModelForCausalLM

# Free embedding model + corpus matrix — done with vector retrieval.
for _name in ("EMB_MODEL", "E_GPU"):
    if _name in globals() and globals()[_name] is not None:
        try:
            del globals()[_name]
        except KeyError:
            pass
gc.collect()
_torch_s1.cuda.empty_cache()
print(f"[stage1] freed embedding + E_GPU; CUDA mem = "
      f"{_torch_s1.cuda.memory_allocated()/1024**3:.1f} GB")

# Load Qwen3-Reranker-8B
print("[stage1] loading Qwen/Qwen3-Reranker-8B (bf16) ...")
_t0 = _t_s1.time()
RERANKER_MODEL = "Qwen/Qwen3-Reranker-8B"
rrk_tok = AutoTokenizer.from_pretrained(RERANKER_MODEL, padding_side="left")
rrk_mod = AutoModelForCausalLM.from_pretrained(
    RERANKER_MODEL,
    dtype=_torch_s1.bfloat16,
    device_map="auto",
).eval()
print(f"[stage1] loaded in {_t_s1.time()-_t0:.1f}s; "
      f"CUDA mem = {_torch_s1.cuda.memory_allocated()/1024**3:.1f} GB")

# Yes/No token IDs for relevance probability extraction.
# CRITICAL: must use convert_tokens_to_ids (looks up exact vocab token),
# NOT tokenizer("yes").input_ids[0] (which BPE-tokenizes and can yield a
# different token like " yes" with leading space). Matches the official
# Qwen3-Reranker-8B model card example exactly.
_RRK_YES_ID = rrk_tok.convert_tokens_to_ids("yes")
_RRK_NO_ID  = rrk_tok.convert_tokens_to_ids("no")
print(f"[stage1] token_true_id (yes)={_RRK_YES_ID}, token_false_id (no)={_RRK_NO_ID}")

_RRK_INSTRUCT = (
    "Given an English legal question about Swiss federal law, determine "
    "whether the provided document (a Swiss law article OR a court paragraph "
    "in German / French / Italian) is a RELEVANT CITATION — meaning it "
    "answers, supports, or directly relates to the question."
)
_RRK_PREFIX = (
    "<|im_start|>system\n"
    "Judge whether the Document meets the requirements based on the Query and "
    "the Instruct provided. Note that the answer can only be \"yes\" or "
    "\"no\".<|im_end|>\n<|im_start|>user\n"
)
_RRK_SUFFIX = (
    "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
)
_RRK_PREFIX_IDS = rrk_tok.encode(_RRK_PREFIX, add_special_tokens=False)
_RRK_SUFFIX_IDS = rrk_tok.encode(_RRK_SUFFIX, add_special_tokens=False)
_RRK_MAX_LEN = 1024
_RRK_MAX_BODY = _RRK_MAX_LEN - len(_RRK_PREFIX_IDS) - len(_RRK_SUFFIX_IDS)


def _rrk_score_pairs(pairs, batch_size=8):
    """Score (query, doc) pairs in batches. Returns list[float] in [0,1]."""
    scores = []
    for _i in range(0, len(pairs), batch_size):
        _batch = pairs[_i:_i+batch_size]
        _bodies = [
            f"<Instruct>: {_RRK_INSTRUCT}\n<Query>: {q}\n<Document>: {d}"
            for q, d in _batch
        ]
        _inp = rrk_tok(
            _bodies,
            padding=False,
            truncation=True,
            max_length=_RRK_MAX_BODY,
            return_attention_mask=False,
            add_special_tokens=False,
        )
        for _j in range(len(_inp["input_ids"])):
            _inp["input_ids"][_j] = (
                _RRK_PREFIX_IDS + _inp["input_ids"][_j] + _RRK_SUFFIX_IDS
            )
        _inp = rrk_tok.pad(_inp, padding=True, return_tensors="pt")
        _inp = {k: v.cuda() for k, v in _inp.items()}
        with _torch_s1.no_grad():
            _logits = rrk_mod(**_inp).logits[:, -1, :]
            _yes = _logits[:, _RRK_YES_ID]
            _no  = _logits[:, _RRK_NO_ID]
            _stk = _torch_s1.stack([_no, _yes], dim=1).float()
            _prb = _torch_s1.nn.functional.softmax(_stk, dim=1)[:, 1]
        scores.extend(_prb.cpu().tolist())
    return scores


STAGE1_TOP_N = 5000
STAGE1_RRK_BATCH = 8

# === SANITY CHECK ===
# Score a (query, known-gold-doc) pair and an (query, known-irrelevant-doc) pair.
# A correct reranker should give the gold pair > 0.3 and irrelevant pair < 0.3.
# This catches token-ID lookup bugs immediately (catastrophic failure of
# previous run: score was effectively random because we used the wrong yes/no
# token IDs).
_sanity_qid = sorted(PER_QUERY.keys())[0]
_sanity_q = next(q["query_text"] for q in ALL_QUERIES if q["query_id"] == _sanity_qid)
_sanity_gold_dids = sorted(ALL_GOLD_DOC_SET[_sanity_qid])
_sanity_pool = PER_QUERY[_sanity_qid]["final_topk"][:5000]
_sanity_nongold = [d for d in _sanity_pool if d not in ALL_GOLD_DOC_SET[_sanity_qid]][:3]
_sanity_gold = [d for d in _sanity_pool if d in ALL_GOLD_DOC_SET[_sanity_qid]][:3]
if _sanity_gold and _sanity_nongold:
    _sanity_pairs = [(_sanity_q, (search_text.get(d, "") or "")[:2000])
                     for d in _sanity_gold + _sanity_nongold]
    _sanity_scores = _rrk_score_pairs(_sanity_pairs, batch_size=STAGE1_RRK_BATCH)
    print(f"\n[stage1 sanity] query: {_sanity_q[:90]}...")
    print(f"  gold scores: {[round(s,4) for s in _sanity_scores[:len(_sanity_gold)]]}")
    print(f"  non-gold scores: {[round(s,4) for s in _sanity_scores[len(_sanity_gold):]]}")
    _mean_gold = sum(_sanity_scores[:len(_sanity_gold)]) / len(_sanity_gold)
    _mean_nong = sum(_sanity_scores[len(_sanity_gold):]) / len(_sanity_nongold)
    print(f"  mean gold={_mean_gold:.4f}  mean non-gold={_mean_nong:.4f}  "
          f"separation={_mean_gold-_mean_nong:+.4f}")
    if _mean_gold <= _mean_nong:
        print(f"  [WARN] reranker is NOT separating gold from non-gold. "
              f"Check token IDs and prompt format.")
    elif _mean_gold - _mean_nong < 0.10:
        print(f"  [WARN] weak separation. Reranker may not help.")
    else:
        print(f"  [OK] separation is reasonable; proceeding.")

print(f"\n[stage1] reranking top-{STAGE1_TOP_N} per query "
      f"(batch_size={STAGE1_RRK_BATCH}) for {len(PER_QUERY)} queries ...")

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    r = PER_QUERY[qid]
    cands = r["final_topk"][:STAGE1_TOP_N]
    pairs = []
    for did in cands:
        _text = search_text.get(did, "")
        if not _text:
            _text = doc_meta.get(did, {}).get("citation", did) or did
        pairs.append((qtext, _text[:3000]))

    _t0 = _t_s1.time()
    _scores = _rrk_score_pairs(pairs, batch_size=STAGE1_RRK_BATCH)
    _ranked = sorted(zip(cands, _scores), key=lambda kv: -kv[1])
    r["stage1_ranked"] = _ranked

    _g_set = ALL_GOLD_DOC_SET[qid]
    _g_tot = ALL_TOTAL_GOLD[qid]
    r["stage1_R_at_K"] = {}
    for _K in [50, 100, 200, 500, 1000, 2000, 5000]:
        _K_use = min(_K, len(_ranked))
        _hit = len({d for d, _ in _ranked[:_K_use]} & _g_set)
        r["stage1_R_at_K"][_K] = (_hit, _hit / max(1, _g_tot))

    print(f"  [{qid}] {_t_s1.time()-_t0:5.1f}s  "
          f"R@50={r['stage1_R_at_K'][50][1]:.3f} "
          f"R@100={r['stage1_R_at_K'][100][1]:.3f} "
          f"R@200={r['stage1_R_at_K'][200][1]:.3f} "
          f"R@500={r['stage1_R_at_K'][500][1]:.3f} "
          f"R@1000={r['stage1_R_at_K'][1000][1]:.3f}")

# Comparison table — fusion vs stage1 reranker
print()
print("=" * 92)
print(f"  STAGE-1 vs FUSION baseline  (macro mean R@K)")
print("=" * 92)
print(f"  {'K':>6}  {'fusion':>8}  {'stage1':>8}  {'Δ':>+7}  {'rerank min':>10}  {'rerank max':>10}")
for _K in [50, 100, 200, 500, 1000, 2000, 5000]:
    _f = []
    _r = []
    for _qid in PER_QUERY:
        _curve = PER_QUERY[_qid]["curve"]
        _keys = sorted(_curve.keys())
        _kk = max((k for k in _keys if k <= _K), default=None)
        _f.append(_curve[_kk][1] if _kk is not None else 0.0)
        _r.append(PER_QUERY[_qid]["stage1_R_at_K"][_K][1])
    print(f"  {_K:>6}  {sum(_f)/len(_f):>8.3f}  {sum(_r)/len(_r):>8.3f}  "
          f"{(sum(_r)-sum(_f))/len(_f):>+7.3f}  {min(_r):>10.3f}  {max(_r):>10.3f}")

# Keep reranker loaded for Stage 1B if enabled; otherwise unload here.
# (Stage 1B's first action checks if rrk_mod exists and unloads it before
# loading Qwen3-32B.)
print(f"\n[stage1] complete. Reranker still in VRAM "
      f"({_torch_s1.cuda.memory_allocated()/1024**3:.1f} GB).")


[stage1] freed embedding + E_GPU; CUDA mem = 0.0 GB
[stage1] loading Qwen/Qwen3-Reranker-8B (bf16) ...


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

[stage1] loaded in 46.4s; CUDA mem = 15.3 GB
[stage1] token_true_id (yes)=9693, token_false_id (no)=2152

[stage1 sanity] query: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 A...
  gold scores: [0.4378, 0.0191, 0.0001]
  non-gold scores: [0.001, 0.0001, 0.0005]
  mean gold=0.1523  mean non-gold=0.0006  separation=+0.1518
  [OK] separation is reasonable; proceeding.

[stage1] reranking top-5000 per query (batch_size=8) for 10 queries ...
  [val_001] 312.9s  R@50=0.000 R@100=0.000 R@200=0.000 R@500=0.190 R@1000=0.286
  [val_002] 329.9s  R@50=0.000 R@100=0.000 R@200=0.000 R@500=0.000 R@1000=0.083
  [val_003] 334.7s  R@50=0.000 R@100=0.000 R@200=0.021 R@500=0.128 R@1000=0.149
  [val_004] 311.6s  R@50=0.200 R@100=0.200 R@200=0.200 R@500=0.300 R@1000=0.500
  [val_005] 323.9s  R@50=0.182 R@100=0.182 R@200=0.455 R@500=0.545 R@1000=0.636
  [val_006] 325.2s  R@50=0.222 R@100=0.333 R@200=0.389 R@500=0.389 R@1000=0.444
  [val_007] 336.1s  R@50=0.000 R@10

ValueError: Sign not allowed in string format specifier

## Stage 1B — Translation A/B (optional)

Tests the hypothesis: does translating DE/FR/IT candidate text to English BEFORE the reranker help Qwen3-Reranker-8B (which is already multilingual)?

Literature (arXiv:2511.19324, arXiv:2603.13301) predicts that explicit MT preprocessing hurts native multilingual rerankers — but we measure on our actual data rather than assume.

Cost-bounded: translates the top-200 candidates per query (2,000 translations total, batched on Qwen3-32B, ~30-45 min). After translation, we re-rerank and compare R@50 / R@100 / R@200 against the original-text Stage 1 result. Whichever wins feeds Stage 2.

Set `RUN_STAGE1B_TRANSLATION = False` inside the cell to skip and pass the original-text Stage 1 ranking straight to Stage 2.


In [ ]:
# =============================================================================
# Stage 1B (OPTIONAL) — Translation A/B
# =============================================================================
# Test the hypothesis: does translating DE/FR/IT candidate text to English BEFORE
# the reranker help Qwen3-Reranker-8B (which is already multilingual)?
#
# 2025-26 literature (arXiv:2511.19324, arXiv:2603.13301) predicts: NO. Native
# multilingual rerankers handle cross-lingual scoring directly; explicit MT
# preprocessing adds noise. But we measure rather than assume — this is the
# user's experimental design.
#
# Cost-bounded: translate only the top-200 candidates per query (after Stage 1).
# That's 2,000 translations total. Batched on Qwen3-32B, ~30-45 min.
#
# Set RUN_STAGE1B_TRANSLATION = False to skip and keep stage1 results.
# =============================================================================

RUN_STAGE1B_TRANSLATION = True   # ← flip to False to skip
STAGE1B_TOP_N           = 200    # how many top candidates per query to translate
STAGE1B_BATCH_SIZE      = 4      # Qwen3-32B generation batch size

if not RUN_STAGE1B_TRANSLATION:
    print("[stage1b] disabled (RUN_STAGE1B_TRANSLATION = False); "
          "Stage 2 will use stage1 (original-text) reranker output.")
    USE_TRANSLATED_RANKING = False
else:
    import gc
    import time as _t_s1b
    import torch as _torch_s1b
    from transformers import AutoTokenizer, AutoModelForCausalLM

    # Free reranker before loading Qwen3-32B
    if "rrk_mod" in globals():
        try:
            del rrk_mod, rrk_tok
        except NameError:
            pass
    gc.collect()
    _torch_s1b.cuda.empty_cache()
    if _torch_s1b.cuda.is_available():
        _torch_s1b.cuda.synchronize()
    print(f"[stage1b] freed reranker; CUDA mem = "
          f"{_torch_s1b.cuda.memory_allocated()/1024**3:.1f} GB")

    # Load Qwen3-32B for translation (re-uses query-model identifier from CONFIG)
    print(f"[stage1b] loading {CONFIG['qwen_query_model']} for translation ...")
    _t0 = _t_s1b.time()
    tr_tok = AutoTokenizer.from_pretrained(CONFIG["qwen_query_model"])
    tr_mod = AutoModelForCausalLM.from_pretrained(
        CONFIG["qwen_query_model"],
        dtype=_torch_s1b.bfloat16,
        device_map="auto",
    ).eval()
    print(f"[stage1b] loaded in {_t_s1b.time()-_t0:.1f}s; "
          f"CUDA mem = {_torch_s1b.cuda.memory_allocated()/1024**3:.1f} GB")

    _STAGE1B_SYS = (
        "You are a Swiss legal translator. Translate the following text into "
        "clean, formal English. CRITICAL: preserve all article references in "
        "their original Swiss form (Art. 221 StPO, Art. 100 BGG, Art. 519 ZGB, "
        "etc.), all case citations in original form (BGE 142 III 48, 1B_192/"
        "2022, etc.), and the case docket prefixes (BGE / 1B / 8C / 6B / 5A / "
        "etc.). For key legal terms, include the original-language term in "
        "parentheses on first occurrence, e.g. 'pre-trial detention "
        "(Untersuchungshaft / détention provisoire)'. If the input is already "
        "in English, output it unchanged. Output ONLY the translation — no "
        "preamble, no markdown, no commentary."
    )

    _tr_input_device = next(
        p.device for p in tr_mod.parameters() if p.device.type != "meta"
    )

    def _stage1b_translate(text):
        """Translate one text. Returns string."""
        if not text or not text.strip():
            return ""
        _msgs = [
            {"role": "system", "content": _STAGE1B_SYS},
            {"role": "user",   "content": text[:1800]},
        ]
        _inp = tr_tok.apply_chat_template(
            _msgs,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            enable_thinking=False,
        )
        _inp = {k: v.to(_tr_input_device) for k, v in _inp.items()}
        _plen = _inp["input_ids"].shape[1]
        with _torch_s1b.no_grad():
            _out = tr_mod.generate(
                **_inp,
                max_new_tokens=600,
                do_sample=False,
                pad_token_id=tr_tok.eos_token_id,
            )
        return tr_tok.decode(_out[0][_plen:], skip_special_tokens=True).strip()

    ALL_TRANSLATIONS = {}  # qid -> {did -> translated text}

    print(f"\n[stage1b] translating top-{STAGE1B_TOP_N} candidates per query ...")
    _total_trans = 0
    _t_total = _t_s1b.time()

    for q in ALL_QUERIES:
        qid = q["query_id"]
        r = PER_QUERY[qid]
        _top_dids = [d for d, _ in r["stage1_ranked"][:STAGE1B_TOP_N]]
        _texts = [search_text.get(did, "") or "" for did in _top_dids]

        _t_q = _t_s1b.time()
        _translations = {}
        for _did, _text in zip(_top_dids, _texts):
            _translations[_did] = _stage1b_translate(_text)
        ALL_TRANSLATIONS[qid] = _translations
        _total_trans += len(_translations)

        _elapsed = _t_s1b.time() - _t_q
        _per = _elapsed / max(1, len(_translations))
        print(f"  [{qid}] {len(_translations)} translations in {_elapsed:5.1f}s "
              f"({_per:.2f}s each)")

    print(f"[stage1b] {_total_trans} translations in "
          f"{(_t_s1b.time()-_t_total)/60:.1f} min")

    # Free Qwen3-32B before re-loading reranker
    del tr_mod, tr_tok
    gc.collect()
    _torch_s1b.cuda.empty_cache()
    if _torch_s1b.cuda.is_available():
        _torch_s1b.cuda.synchronize()
    print(f"[stage1b] freed Qwen3-32B; CUDA mem = "
          f"{_torch_s1b.cuda.memory_allocated()/1024**3:.1f} GB")

    # Reload reranker for second rerank pass on translated text
    print("[stage1b] reloading Qwen/Qwen3-Reranker-8B for re-rerank ...")
    _t0 = _t_s1b.time()
    rrk_tok = AutoTokenizer.from_pretrained(RERANKER_MODEL, padding_side="left")
    rrk_mod = AutoModelForCausalLM.from_pretrained(
        RERANKER_MODEL,
        dtype=_torch_s1b.bfloat16,
        device_map="auto",
    ).eval()
    print(f"[stage1b] reranker reloaded in {_t_s1b.time()-_t0:.1f}s")

    print(f"\n[stage1b] rescoring top-{STAGE1B_TOP_N} translated candidates ...")
    for q in ALL_QUERIES:
        qid = q["query_id"]
        qtext = q["query_text"]
        r = PER_QUERY[qid]
        _top_dids = [d for d, _ in r["stage1_ranked"][:STAGE1B_TOP_N]]
        _trans = ALL_TRANSLATIONS[qid]
        _pairs = [(qtext, _trans.get(d, "")[:3000]) for d in _top_dids]

        _t_q = _t_s1b.time()
        _scores = _rrk_score_pairs(_pairs, batch_size=STAGE1_RRK_BATCH)
        _ranked = sorted(zip(_top_dids, _scores), key=lambda kv: -kv[1])
        r["stage1b_ranked"] = _ranked

        _g_set = ALL_GOLD_DOC_SET[qid]
        _g_tot = ALL_TOTAL_GOLD[qid]
        r["stage1b_R_at_K"] = {}
        for _K in [50, 100, 200]:
            _K_use = min(_K, len(_ranked))
            _hit = len({d for d, _ in _ranked[:_K_use]} & _g_set)
            r["stage1b_R_at_K"][_K] = (_hit, _hit / max(1, _g_tot))

        print(f"  [{qid}] {_t_s1b.time()-_t_q:5.1f}s  "
              f"R@50={r['stage1b_R_at_K'][50][1]:.3f} "
              f"R@100={r['stage1b_R_at_K'][100][1]:.3f} "
              f"R@200={r['stage1b_R_at_K'][200][1]:.3f}")

    # A/B comparison — decide which to use for Stage 2
    print()
    print("=" * 80)
    print("  STAGE-1B A/B  (R@K on TOP-200 candidates, original vs translated)")
    print("=" * 80)
    print(f"  {'query':<10}  {'orig R@50':>10}  {'trans R@50':>11}  "
          f"{'orig R@100':>11}  {'trans R@100':>12}  {'orig R@200':>11}  "
          f"{'trans R@200':>12}")
    _orig_avg = {50: [], 100: [], 200: []}
    _trans_avg = {50: [], 100: [], 200: []}
    for qid in sorted(PER_QUERY.keys()):
        r = PER_QUERY[qid]
        _row = [qid]
        for _K in [50, 100, 200]:
            _o = r["stage1_R_at_K"][_K][1]
            _t = r["stage1b_R_at_K"][_K][1]
            _orig_avg[_K].append(_o)
            _trans_avg[_K].append(_t)
            _row.extend([_o, _t])
        print(f"  {_row[0]:<10}  {_row[1]:>10.3f}  {_row[2]:>11.3f}  "
              f"{_row[3]:>11.3f}  {_row[4]:>12.3f}  "
              f"{_row[5]:>11.3f}  {_row[6]:>12.3f}")
    print(f"  {'MEAN':<10}  "
          f"{sum(_orig_avg[50])/len(_orig_avg[50]):>10.3f}  "
          f"{sum(_trans_avg[50])/len(_trans_avg[50]):>11.3f}  "
          f"{sum(_orig_avg[100])/len(_orig_avg[100]):>11.3f}  "
          f"{sum(_trans_avg[100])/len(_trans_avg[100]):>12.3f}  "
          f"{sum(_orig_avg[200])/len(_orig_avg[200]):>11.3f}  "
          f"{sum(_trans_avg[200])/len(_trans_avg[200]):>12.3f}")

    # Decide: use translated ranking if mean R@200 > 0.01 better
    _orig_200 = sum(_orig_avg[200]) / len(_orig_avg[200])
    _trans_200 = sum(_trans_avg[200]) / len(_trans_avg[200])
    USE_TRANSLATED_RANKING = (_trans_200 > _orig_200 + 0.01)
    print(f"\n  Decision: Stage 2 will use "
          f"{'TRANSLATED' if USE_TRANSLATED_RANKING else 'ORIGINAL'} ranking "
          f"(orig R@200={_orig_200:.3f}, trans R@200={_trans_200:.3f})")

# Free reranker — Stage 2 needs Qwen3-32B
import gc as _gc_end
import torch as _torch_end
if "rrk_mod" in globals():
    try:
        del rrk_mod, rrk_tok
    except NameError:
        pass
_gc_end.collect()
_torch_end.cuda.empty_cache()
print(f"\n[stage1b] freed reranker for Stage 2; "
      f"CUDA mem = {_torch_end.cuda.memory_allocated()/1024**3:.1f} GB")


## Stage 2 — Listwise LLM scoring (Qwen3-32B)

The reranker is calibrated for surface relevance, not legal reasoning. Stage 2 brings the 32B model into the loop to score for *legal* relevance. Listwise (20 candidates per call) because comparative judgment is empirically more reliable than pointwise yes/no.

Each candidate is presented with: citation, family, court_base, language, paragraph role, statutes cited, pre-extracted concepts, and a 400-char text excerpt. The model outputs a JSON array `[{id, score 1-5, why <=25 words}, ...]`. Aggregated scores yield the top-100 that go into Stage 3.

System prompt embeds: Swiss code aliases (CC=ZGB, CO=OR, ...), explicit multilingual normalcy, and a hard-negative anchor example to calibrate away from term-overlap mirage.


In [ ]:
# =============================================================================
# Stage 2 — Listwise LLM scoring with Qwen3-32B
# =============================================================================
# Take top-500 reranked candidates per query (from Stage 1, or Stage 1B if it
# won the A/B). Present them to Qwen3-32B in batches of 20, ask the model to
# score each 1-5 for relevance and give a ≤25-word justification grounded in
# the candidate's text. Aggregate scores across batches; output top-100 by
# score.
#
# Anti-hallucination design:
#   - Multilingual reminders inline (CC=ZGB, CO=OR, ...)
#   - Hard-negative anchor example in system prompt
#   - Output schema strictly enforced (JSON array, parsed; malformed entries
#     are scored 0)
#   - "If text doesn't contain reasoning relevant to the query, score 1-2"
# =============================================================================

import gc
import json as _json_s2
import re as _re_s2
import time as _t_s2
import torch as _torch_s2
from transformers import AutoTokenizer, AutoModelForCausalLM

STAGE2_INPUT_TOP_N = 500       # how many candidates per query enter listwise
STAGE2_BATCH_SIZE  = 20        # candidates per LLM call
STAGE2_OUTPUT_TOP_N = 100      # how many candidates Stage 3 will see

_RANKING_KEY = "stage1b_ranked" if (
    "USE_TRANSLATED_RANKING" in globals() and USE_TRANSLATED_RANKING
) else "stage1_ranked"
print(f"[stage2] reading from PER_QUERY[*]['{_RANKING_KEY}']")

# Load Qwen3-32B
print(f"[stage2] loading {CONFIG['qwen_query_model']} ...")
_t0 = _t_s2.time()
s2_tok = AutoTokenizer.from_pretrained(CONFIG["qwen_query_model"])
s2_mod = AutoModelForCausalLM.from_pretrained(
    CONFIG["qwen_query_model"],
    dtype=_torch_s2.bfloat16,
    device_map="auto",
).eval()
print(f"[stage2] loaded in {_t_s2.time()-_t0:.1f}s; "
      f"CUDA mem = {_torch_s2.cuda.memory_allocated()/1024**3:.1f} GB")

_s2_input_device = next(
    p.device for p in s2_mod.parameters() if p.device.type != "meta"
)

S2_SYSTEM = (
    "You are a Swiss legal citation expert. You evaluate documents (Swiss law "
    "articles OR court paragraphs) for relevance to an English legal question. "
    "\n\nKEY FACTS:\n"
    "- Documents may be in German, French, or Italian. The query is English. "
    "Multilingual gold is normal and EXPECTED.\n"
    "- Swiss code aliases (treat as SAME code): CC=ZGB, CO=OR, CP=StGB, "
    "CPP=StPO, LP=SchKG, LIFD=DBG, LPGA=ATSG, LAI=IVG, LTF=BGG, LAA=UVG, "
    "Cst=BV, CEDH=EMRK.\n"
    "- Gold can be a law article OR a court paragraph (BGE 142 III 48 E. 4.1, "
    "1B_192/2022 E. 4.1.2, etc.).\n"
    "- A document that shares vocabulary with the query but lacks specific "
    "legal reasoning is NOT relevant. Topical overlap is not evidence.\n"
    "- A typical relevant citation either: (a) states the legal rule that "
    "answers the question, OR (b) applies/interprets that rule to facts "
    "similar to the question, OR (c) cites the central statute for the "
    "question's topic.\n\n"
    "HARD-NEGATIVE EXAMPLE (calibration): A paragraph that mentions "
    "\"detention\" only to say \"the suspect was not detained\" is NOT a "
    "relevant citation for a query about pre-trial detention grounds — it "
    "shares the term but contains no legal reasoning about detention.\n\n"
    "OUTPUT: A strict JSON array. Score each candidate 1-5 and give a ≤25-word "
    "justification that REFERENCES SPECIFIC CONTENT from the candidate's text."
)


def _s2_build_prompt(query_text, aspects, candidate_list):
    """Build the user prompt for a batch of candidates."""
    _aspects_str = "\n".join(f"  {i+1}. {a[:200]}" for i, a in enumerate(aspects[:6]))
    _cands_str = ""
    for idx, c in enumerate(candidate_list, start=1):
        _cands_str += (
            f"\n[{idx}]\n"
            f"  citation: {c['citation']}\n"
            f"  type: {c['family']}; court_base: {c['court_base']}; "
            f"role: {c['role']}; lang: {c['lang']}\n"
            f"  statutes cited: {c['anchors']}\n"
            f"  concepts: {c['concepts']}\n"
            f"  text excerpt: \"{c['text']}\"\n"
        )
    return (
        f"QUERY: {query_text}\n\n"
        f"QUERY ASPECTS (decomposed):\n{_aspects_str}\n\n"
        f"CANDIDATES (N={len(candidate_list)}):\n{_cands_str}\n"
        f"For each candidate above, output a JSON array of exactly "
        f"{len(candidate_list)} objects:\n"
        f"[{{\"id\": <int 1-{len(candidate_list)}>, \"score\": <int 1-5>, "
        f"\"why\": \"<≤25 words referencing candidate text>\"}}, ...]\n\n"
        f"Output ONLY the JSON array."
    )


def _s2_run_llm(prompt, max_new_tokens=4000):
    _msgs = [
        {"role": "system", "content": S2_SYSTEM},
        {"role": "user",   "content": prompt},
    ]
    _inp = s2_tok.apply_chat_template(
        _msgs,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    )
    _inp = {k: v.to(_s2_input_device) for k, v in _inp.items()}
    _plen = _inp["input_ids"].shape[1]
    with _torch_s2.no_grad():
        _out = s2_mod.generate(
            **_inp,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=s2_tok.eos_token_id,
        )
    return s2_tok.decode(_out[0][_plen:], skip_special_tokens=True).strip()


def _s2_parse_response(resp, n_expected):
    """Parse JSON array. Returns dict[batch_idx -> (score, why)]. Missing IDs
    default to score=0. Robust to extra text around JSON."""
    _result = {}
    _match = _re_s2.search(r"\[.*\]", resp, _re_s2.S)
    if not _match:
        return _result
    try:
        _arr = _json_s2.loads(_match.group(0))
    except Exception:
        return _result
    if not isinstance(_arr, list):
        return _result
    for _item in _arr:
        if not isinstance(_item, dict):
            continue
        try:
            _idx = int(_item.get("id"))
            _score = int(_item.get("score", 0))
            _why = str(_item.get("why", ""))[:200]
            if 1 <= _idx <= n_expected and 1 <= _score <= 5:
                _result[_idx] = (_score, _why)
        except (TypeError, ValueError):
            continue
    return _result


def _s2_make_candidate_record(did):
    """Pack a candidate's metadata + text excerpt for the prompt."""
    m = doc_meta.get(did, {}) or {}
    _text = (search_text.get(did, "") or "")[:400].replace("\n", " ")
    if not _text:
        _text = m.get("citation", did) or did
    _anchors = list(doc_statute_anchors.get(did, set()) or set())[:8]
    _anchors_str = ", ".join(_anchors) if _anchors else "(none)"
    _concepts = sorted(
        _doc_to_concepts.get(did, set()) if "_doc_to_concepts" in globals()
        else set()
    )[:6]
    _concepts_str = ", ".join(_concepts) if _concepts else "(none)"
    return {
        "did":       did,
        "citation":  m.get("citation", did) or did,
        "family":    m.get("family", "?"),
        "court_base":m.get("court_base", "") or "(n/a)",
        "role":      (m.get("paragraph_role") or "(n/a)"),
        "lang":      m.get("language") or m.get("language_code") or "?",
        "anchors":   _anchors_str,
        "concepts":  _concepts_str,
        "text":      _text,
    }


# Build the multi-aspect list to pass to the prompt
_query_aspects = {q["query_id"]: (ALL_HYDE_ASPECTS.get(q["query_id"], [])
                                   if "ALL_HYDE_ASPECTS" in globals() else [])
                  for q in ALL_QUERIES}

print(f"\n[stage2] listwise scoring "
      f"(top-{STAGE2_INPUT_TOP_N}, batch={STAGE2_BATCH_SIZE}, "
      f"output top-{STAGE2_OUTPUT_TOP_N}) ...")

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    r = PER_QUERY[qid]
    _cands_ranked = r[_RANKING_KEY][:STAGE2_INPUT_TOP_N]
    _dids = [d for d, _ in _cands_ranked]

    # Per-doc aggregated score (we score each doc exactly once here)
    _doc_scores = {}
    _doc_whys = {}

    _t_q = _t_s2.time()
    _n_batches = 0
    for _b in range(0, len(_dids), STAGE2_BATCH_SIZE):
        _batch_dids = _dids[_b:_b+STAGE2_BATCH_SIZE]
        _batch_recs = [_s2_make_candidate_record(d) for d in _batch_dids]
        _prompt = _s2_build_prompt(qtext, _query_aspects.get(qid, []), _batch_recs)
        _resp = _s2_run_llm(_prompt, max_new_tokens=20 * len(_batch_recs) + 200)
        _parsed = _s2_parse_response(_resp, len(_batch_recs))
        for _idx_in_batch, _did in enumerate(_batch_dids, start=1):
            if _idx_in_batch in _parsed:
                _sc, _w = _parsed[_idx_in_batch]
            else:
                _sc, _w = 0, ""
            _doc_scores[_did] = _sc
            _doc_whys[_did] = _w
        _n_batches += 1

    _ranked = sorted(_doc_scores.items(), key=lambda kv: -kv[1])
    r["stage2_ranked"] = _ranked
    r["stage2_whys"] = _doc_whys

    _g_set = ALL_GOLD_DOC_SET[qid]
    _g_tot = ALL_TOTAL_GOLD[qid]
    r["stage2_R_at_K"] = {}
    for _K in [50, 100, 200, 500]:
        _K_use = min(_K, len(_ranked))
        _hit = len({d for d, _ in _ranked[:_K_use]} & _g_set)
        r["stage2_R_at_K"][_K] = (_hit, _hit / max(1, _g_tot))

    _n_score_dist = {s: sum(1 for _, sc in _ranked if sc == s) for s in (5, 4, 3, 2, 1, 0)}
    print(f"  [{qid}] {_t_s2.time()-_t_q:6.1f}s  {_n_batches} batches  "
          f"score-dist (5/4/3/2/1/0)="
          f"{_n_score_dist[5]:>2}/{_n_score_dist[4]:>3}/"
          f"{_n_score_dist[3]:>3}/{_n_score_dist[2]:>3}/"
          f"{_n_score_dist[1]:>3}/{_n_score_dist[0]:>3}  "
          f"R@50={r['stage2_R_at_K'][50][1]:.3f} "
          f"R@100={r['stage2_R_at_K'][100][1]:.3f} "
          f"R@200={r['stage2_R_at_K'][200][1]:.3f}")

# Summary
print()
print("=" * 92)
print("  STAGE-2 LISTWISE vs STAGE-1 RERANK  (macro mean R@K)")
print("=" * 92)
print(f"  {'K':>6}  {'stage1':>8}  {'stage2':>8}  {'Δ':>+7}  "
      f"{'stage2 min':>10}  {'stage2 max':>10}")
for _K in [50, 100, 200, 500]:
    _r1 = [PER_QUERY[qid]["stage1_R_at_K"][_K][1] for qid in PER_QUERY]
    _r2 = [PER_QUERY[qid]["stage2_R_at_K"][_K][1] for qid in PER_QUERY]
    print(f"  {_K:>6}  {sum(_r1)/len(_r1):>8.3f}  {sum(_r2)/len(_r2):>8.3f}  "
          f"{(sum(_r2)-sum(_r1))/len(_r1):>+7.3f}  "
          f"{min(_r2):>10.3f}  {max(_r2):>10.3f}")
print(f"\n[stage2] complete. Qwen3-32B stays loaded for Stage 3.")


## Stage 3 — Pointwise evidence-quoted verdict (Qwen3-32B)

Deep judgment on the top-100 surviving Stage 2 candidates. 5 candidates per LLM call so each gets full attention.

The output schema enforces:
- `topic` — what the document is about (1 sentence)
- `relation` — how it relates to the query (≤2 sentences)
- `evidence_quote` — **VERBATIM** substring of the document text supporting relevance. If empty, the candidate is forcibly DROP regardless of confidence.
- `confidence` — 1-5
- `verdict` — KEEP / MAYBE / DROP, computed deterministically from confidence + evidence_quote (not trusted from the LLM).

The verbatim-quote requirement is the primary defense against fluent-prose hallucination — the model cannot slide on confident-sounding justifications without pointing to specific document text.


In [ ]:
# =============================================================================
# Stage 3 — Pointwise LLM judgment with evidence quoting
# =============================================================================
# Take top-100 candidates from Stage 2 (listwise). For each one, ask
# Qwen3-32B to do a structured deep analysis with evidence quoting. The
# evidence-quote requirement is the primary anti-hallucination gate — the
# model must extract a verbatim sentence from the candidate text supporting
# relevance. If it can't, the candidate is DROP regardless of fluent prose.
#
# Output per candidate (strict JSON):
#   topic          — one sentence, what the document is about
#   relation       — ≤2 sentences, how it relates to the query
#   evidence_quote — VERBATIM quote from the doc text (empty = drop)
#   confidence     — 1-5
#   verdict        — KEEP / MAYBE / DROP
#
# Batch size: 5 candidates per LLM call so each gets full attention.
# =============================================================================

import json as _json_s3
import re as _re_s3
import time as _t_s3
import torch as _torch_s3

STAGE3_INPUT_TOP_N = 100   # how many from Stage 2 we judge pointwise
STAGE3_BATCH_SIZE  = 5     # candidates per LLM call

S3_SYSTEM = (
    "You are a Swiss legal citation expert. For each Swiss legal document "
    "provided, you decide whether it is a relevant citation for the English "
    "legal question.\n\n"
    "CRITICAL ANTI-HALLUCINATION RULES — these are non-negotiable:\n"
    "1. Base your judgment ONLY on the document text shown. Do not invent "
    "citations, facts, or legal rules.\n"
    "2. The `evidence_quote` field MUST be a VERBATIM substring of the "
    "document text shown. If you cannot quote a specific sentence that "
    "proves the relation, set `evidence_quote = \"\"` and `verdict = "
    "\"DROP\"`.\n"
    "3. Lexical overlap with query terms is NOT evidence. Only legal "
    "reasoning that specifically addresses the question is evidence.\n"
    "4. Fluent-sounding prose without a verbatim quote is hallucination.\n\n"
    "DOMAIN FACTS:\n"
    "- Documents may be German / French / Italian. Multilingual is expected.\n"
    "- Swiss code aliases (treat as the SAME code): CC=ZGB, CO=OR, CP=StGB, "
    "CPP=StPO, LP=SchKG, LIFD=DBG, LPGA=ATSG, LAI=IVG, LTF=BGG, LAA=UVG, "
    "Cst=BV, CEDH=EMRK.\n"
    "- Both law articles and court paragraphs (BGE / 1B / 8C / ...) are valid "
    "citation targets.\n"
    "- A 'relevant citation' either: (a) states the rule that answers the "
    "question, (b) applies the rule to facts similar to the question, or "
    "(c) is the central statute authority for the question's topic.\n\n"
    "VERDICT RULES:\n"
    "  KEEP  = confidence ≥ 4 AND non-empty evidence_quote\n"
    "  MAYBE = confidence == 3 AND non-empty evidence_quote\n"
    "  DROP  = confidence ≤ 2  OR  empty evidence_quote\n\n"
    "OUTPUT: Strict JSON array, one object per candidate, no preamble."
)


def _s3_build_prompt(query_text, candidate_list):
    _cands_str = ""
    for idx, c in enumerate(candidate_list, start=1):
        _cands_str += (
            f"\n[{idx}]\n"
            f"  citation: {c['citation']}\n"
            f"  type: {c['family']}; court_base: {c['court_base']}; "
            f"role: {c['role']}; lang: {c['lang']}\n"
            f"  statutes cited: {c['anchors']}\n"
            f"  concepts: {c['concepts']}\n"
            f"  full text: \"{c['full_text']}\"\n"
        )
    _schema = (
        '[{"id": <int 1-N>, "topic": "<1 sentence>", "relation": '
        '"<≤2 sentences>", "evidence_quote": "<verbatim or empty>", '
        '"confidence": <1-5>, "verdict": "<KEEP|MAYBE|DROP>"}, ...]'
    )
    return (
        f"QUERY: {query_text}\n\n"
        f"CANDIDATES (N={len(candidate_list)}):\n{_cands_str}\n"
        f"For each candidate, output strict JSON in this exact schema:\n"
        f"{_schema}\n\n"
        f"Apply the verdict rules. Output ONLY the JSON array."
    )


def _s3_make_full_candidate_record(did):
    """Like Stage-2's record but with full (or longer) text."""
    m = doc_meta.get(did, {}) or {}
    _text = (search_text.get(did, "") or "")[:1500].replace("\n", " ")
    if not _text:
        _text = m.get("citation", did) or did
    _anchors = list(doc_statute_anchors.get(did, set()) or set())[:12]
    _anchors_str = ", ".join(_anchors) if _anchors else "(none)"
    _concepts = sorted(
        _doc_to_concepts.get(did, set()) if "_doc_to_concepts" in globals()
        else set()
    )[:10]
    _concepts_str = ", ".join(_concepts) if _concepts else "(none)"
    return {
        "did":       did,
        "citation":  m.get("citation", did) or did,
        "family":    m.get("family", "?"),
        "court_base":m.get("court_base", "") or "(n/a)",
        "role":      (m.get("paragraph_role") or "(n/a)"),
        "lang":      m.get("language") or m.get("language_code") or "?",
        "anchors":   _anchors_str,
        "concepts":  _concepts_str,
        "full_text": _text,
    }


def _s3_parse_response(resp, n_expected):
    """Parse strict JSON array; return dict[idx -> dict(...)]. Robust."""
    _result = {}
    _match = _re_s3.search(r"\[.*\]", resp, _re_s3.S)
    if not _match:
        return _result
    try:
        _arr = _json_s3.loads(_match.group(0))
    except Exception:
        return _result
    if not isinstance(_arr, list):
        return _result
    for _item in _arr:
        if not isinstance(_item, dict):
            continue
        try:
            _idx = int(_item.get("id"))
            if not (1 <= _idx <= n_expected):
                continue
            _conf = int(_item.get("confidence", 0))
            _verd = str(_item.get("verdict", "DROP")).upper()
            if _verd not in {"KEEP", "MAYBE", "DROP"}:
                _verd = "DROP"
            _ev = str(_item.get("evidence_quote", ""))[:600]
            # Verdict enforcement (re-check the rules even if LLM lies):
            if not _ev.strip():
                _verd = "DROP"
            elif _conf <= 2:
                _verd = "DROP"
            elif _conf == 3:
                _verd = "MAYBE"
            else:
                _verd = "KEEP"
            _result[_idx] = {
                "topic":          str(_item.get("topic", ""))[:300],
                "relation":       str(_item.get("relation", ""))[:500],
                "evidence_quote": _ev,
                "confidence":     max(1, min(5, _conf)),
                "verdict":        _verd,
            }
        except (TypeError, ValueError):
            continue
    return _result


print(f"\n[stage3] pointwise verdict on top-{STAGE3_INPUT_TOP_N} per query "
      f"(batch={STAGE3_BATCH_SIZE}) ...")

for q in ALL_QUERIES:
    qid = q["query_id"]
    qtext = q["query_text"]
    r = PER_QUERY[qid]
    _dids = [d for d, _ in r["stage2_ranked"][:STAGE3_INPUT_TOP_N]]

    _t_q = _t_s3.time()
    _verdicts = {}
    _n_batches = 0
    for _b in range(0, len(_dids), STAGE3_BATCH_SIZE):
        _batch_dids = _dids[_b:_b+STAGE3_BATCH_SIZE]
        _batch_recs = [_s3_make_full_candidate_record(d) for d in _batch_dids]
        _prompt = _s3_build_prompt(qtext, _batch_recs)
        _resp = _s2_run_llm(_prompt, max_new_tokens=4000)
        _parsed = _s3_parse_response(_resp, len(_batch_recs))
        for _idx_in_batch, _did in enumerate(_batch_dids, start=1):
            _verdicts[_did] = _parsed.get(_idx_in_batch, {
                "topic": "", "relation": "", "evidence_quote": "",
                "confidence": 0, "verdict": "DROP",
            })
        _n_batches += 1

    r["stage3_verdicts"] = _verdicts

    _keep = [d for d, v in _verdicts.items() if v["verdict"] == "KEEP"]
    _maybe = [d for d, v in _verdicts.items() if v["verdict"] == "MAYBE"]
    _drop = [d for d, v in _verdicts.items() if v["verdict"] == "DROP"]

    _g_set = ALL_GOLD_DOC_SET[qid]
    _g_tot = ALL_TOTAL_GOLD[qid]
    _keep_gold = len(set(_keep) & _g_set)
    _maybe_gold = len(set(_maybe) & _g_set)

    print(f"  [{qid}] {_t_s3.time()-_t_q:6.1f}s  {_n_batches} batches  "
          f"KEEP={len(_keep):>3}(gold {_keep_gold:>2}/{_g_tot})  "
          f"MAYBE={len(_maybe):>3}(gold {_maybe_gold:>2})  "
          f"DROP={len(_drop):>3}")

print("\n[stage3] complete. Qwen3-32B stays loaded if reused; otherwise free.")


## Stage 4 — Adaptive K + aspect coverage + F1

Final selection:

1. Take all `KEEP` with confidence 5.
2. Add `KEEP` confidence 4 up to a target K (proportional to query complexity — `target_min = max(15, n_aspects × 4)`, `target_max = max(40, n_aspects × 8)`).
3. Add `MAYBE` only if below target_min.
4. **Aspect coverage check**: for each decomposed aspect from `ALL_HYDE_ASPECTS`, ensure ≥1 KEEP relates to it (proxy: aspect keywords overlap with Stage-2 `why` text). If an aspect is uncovered but Stage-2 had a high-scoring candidate addressing it, force-promote.
5. Cap final K.

Outputs per-query P / R / F1, macro F1, and a head-to-head vs the fusion-baseline F1 at the same K (so we know whether the cascade is actually helping).


In [ ]:
# =============================================================================
# Stage 4 — Adaptive K + aspect coverage + F1 estimation
# =============================================================================
# Determine the final top-K per query from Stage 3's KEEP / MAYBE / DROP
# verdicts + confidence scores. Floor K based on query complexity (number of
# decomposed aspects). Apply aspect coverage check — if a decomposed aspect
# has zero KEEPs but had high-scoring Stage-2 candidates, force-promote one.
# Final: compute macro F1 per K + comparison to the fusion baseline.
# =============================================================================

import gc as _gc_s4
import re as _re_s4
import time as _t_s4
import torch as _torch_s4

# Free Qwen3-32B — Stage 3 was the last LLM-heavy stage.
for _name in ("s2_mod", "s2_tok"):
    if _name in globals():
        try:
            del globals()[_name]
        except KeyError:
            pass
_gc_s4.collect()
_torch_s4.cuda.empty_cache()
if _torch_s4.cuda.is_available():
    _torch_s4.cuda.synchronize()
print(f"[stage4] freed Qwen3-32B; CUDA mem = "
      f"{_torch_s4.cuda.memory_allocated()/1024**3:.1f} GB")


def _stage4_aspect_coverage(qid, stage2_ranked, stage3_verdicts, current_keep_dids):
    """For each decomposed aspect, ensure ≥1 keep exists. If an aspect is
    uncovered but Stage 2 had a non-zero scorer for it, promote that candidate
    (we approximate aspect-match by checking the stage2 'why' text against the
    aspect string — a cheap proxy)."""
    _aspects = ALL_HYDE_ASPECTS.get(qid, []) if "ALL_HYDE_ASPECTS" in globals() else []
    if not _aspects or len(_aspects) <= 1:
        return list(current_keep_dids), {}

    _whys = PER_QUERY[qid].get("stage2_whys", {}) or {}
    _aspect_terms = []
    for _a in _aspects:
        _toks = set(t for t in _re_s4.findall(r"[A-Za-zÄÖÜäöüß]{4,}", _a.lower()))
        _aspect_terms.append(_toks)

    _covered = set()
    for _did in current_keep_dids:
        _why = (_whys.get(_did) or "").lower()
        _why_toks = set(_re_s4.findall(r"[A-Za-zÄÖÜäöüß]{4,}", _why))
        for _ai, _a_toks in enumerate(_aspect_terms):
            if _a_toks & _why_toks:
                _covered.add(_ai)

    _missing = [i for i in range(len(_aspect_terms)) if i not in _covered]
    if not _missing:
        return list(current_keep_dids), {}

    _promoted = {}
    _stage2_dict = dict(stage2_ranked)
    for _ai in _missing:
        _best_did = None
        _best_score = -1
        for _did, _s2_score in stage2_ranked[:200]:
            if _did in current_keep_dids:
                continue
            _why = (_whys.get(_did) or "").lower()
            _why_toks = set(_re_s4.findall(r"[A-Za-zÄÖÜäöüß]{4,}", _why))
            if _aspect_terms[_ai] & _why_toks and _s2_score > _best_score:
                _best_did = _did
                _best_score = _s2_score
        if _best_did is not None:
            _promoted[_best_did] = _ai

    _new_keep = list(current_keep_dids)
    for _did in _promoted:
        if _did not in _new_keep:
            _new_keep.append(_did)
    return _new_keep, _promoted


print("\n[stage4] applying adaptive K + aspect coverage ...")

# Per-query final selection
for q in ALL_QUERIES:
    qid = q["query_id"]
    r = PER_QUERY[qid]
    _verdicts = r["stage3_verdicts"]
    _stage2 = r["stage2_ranked"]

    # Score-ordered KEEP / MAYBE pools
    _keep_5 = [d for d, v in _verdicts.items() if v["verdict"] == "KEEP" and v["confidence"] == 5]
    _keep_4 = [d for d, v in _verdicts.items() if v["verdict"] == "KEEP" and v["confidence"] == 4]
    _maybe  = [d for d, v in _verdicts.items() if v["verdict"] == "MAYBE"]

    # Sort each by stage2 score (higher first) to break ties deterministically
    _s2_score = dict(_stage2)
    _keep_5.sort(key=lambda d: -_s2_score.get(d, 0))
    _keep_4.sort(key=lambda d: -_s2_score.get(d, 0))
    _maybe.sort(key=lambda d: -_s2_score.get(d, 0))

    # Initial selection: all conf-5, then conf-4, then add MAYBE up to target
    _n_aspects = (len(ALL_HYDE_ASPECTS.get(qid, [])) if "ALL_HYDE_ASPECTS" in globals() else 1)
    _target_min = max(15, _n_aspects * 4)   # floor — multi-aspect needs more
    _target_max = max(40, _n_aspects * 8)   # cap — don't inflate

    _final = list(_keep_5)
    _final.extend(d for d in _keep_4 if d not in _final)
    if len(_final) < _target_min:
        _need = _target_min - len(_final)
        _final.extend(d for d in _maybe[:_need] if d not in _final)
    _final = _final[:_target_max]

    # Aspect coverage check — force-promote one candidate per uncovered aspect
    _final, _promoted = _stage4_aspect_coverage(qid, _stage2, _verdicts, _final)
    _final = _final[:_target_max + max(0, len(_promoted))]

    r["final_selected"] = _final
    r["stage4_promoted"] = _promoted

    _g_set = ALL_GOLD_DOC_SET[qid]
    _g_tot = ALL_TOTAL_GOLD[qid]
    _hit = len(set(_final) & _g_set)
    _prec = _hit / max(1, len(_final))
    _rec = _hit / max(1, _g_tot)
    _f1 = (2*_prec*_rec / (_prec+_rec)) if (_prec+_rec) > 0 else 0
    r["final_F1"] = _f1
    r["final_precision"] = _prec
    r["final_recall"] = _rec
    r["final_K"] = len(_final)

    print(f"  [{qid}] gold={_g_tot:>3}  K={len(_final):>3}  "
          f"caught={_hit:>3}  P={_prec:.3f}  R={_rec:.3f}  F1={_f1:.3f}  "
          f"(KEEP5={len(_keep_5):>2}, KEEP4={len(_keep_4):>2}, MAYBE={len(_maybe):>2}, "
          f"promoted={len(_promoted)})")

# ============================================================================
# Final F1 summary
# ============================================================================
print()
print("=" * 96)
print("  FINAL F1 SUMMARY  (adaptive top-K from cascade)")
print("=" * 96)
print(f"  {'query':<10}  {'gold':>4}  {'K':>4}  {'caught':>6}  "
      f"{'P':>6}  {'R':>6}  {'F1':>6}")
_macro_f1 = []
_macro_p = []
_macro_r = []
_total_tp = 0
_total_pred = 0
_total_gold = 0
for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    print(f"  {qid:<10}  {r['gold']:>4}  {r['final_K']:>4}  "
          f"{int(r['final_recall']*r['gold']):>6}  "
          f"{r['final_precision']:>6.3f}  {r['final_recall']:>6.3f}  "
          f"{r['final_F1']:>6.3f}")
    _macro_f1.append(r["final_F1"])
    _macro_p.append(r["final_precision"])
    _macro_r.append(r["final_recall"])
    _hit = int(r["final_recall"] * r["gold"])
    _total_tp += _hit
    _total_pred += r["final_K"]
    _total_gold += r["gold"]
print("-" * 96)
print(f"  {'MACRO':<10}                          "
      f"{sum(_macro_p)/len(_macro_p):>6.3f}  {sum(_macro_r)/len(_macro_r):>6.3f}  "
      f"{sum(_macro_f1)/len(_macro_f1):>6.3f}")
_micro_p = _total_tp / max(1, _total_pred)
_micro_r = _total_tp / max(1, _total_gold)
_micro_f1 = (2*_micro_p*_micro_r/(_micro_p+_micro_r)) if (_micro_p+_micro_r)>0 else 0
print(f"  {'MICRO':<10}  TP={_total_tp:>3}  pred={_total_pred:>3}  gold={_total_gold:>3}  "
      f"{_micro_p:>6.3f}  {_micro_r:>6.3f}  {_micro_f1:>6.3f}")

# ============================================================================
# Comparison: fusion-baseline F1 (at matching K) vs cascade F1
# ============================================================================
print()
print("=" * 96)
print("  CASCADE vs FUSION-BASELINE F1  (at matching K per query)")
print("=" * 96)
print(f"  {'query':<10}  {'gold':>4}  {'K':>4}  "
      f"{'fusion F1':>10}  {'cascade F1':>11}  {'Δ':>+7}")
_cascade_better = 0
_fusion_f1_per_q = []
for qid in sorted(PER_QUERY.keys()):
    r = PER_QUERY[qid]
    _K = r["final_K"]
    _curve = r["curve"]
    _keys = sorted(_curve.keys())
    _kk = max((k for k in _keys if k <= _K), default=None)
    _f_R = _curve[_kk][1] if _kk is not None else 0.0
    _f_caught = int(_f_R * r["gold"])
    _f_P = _f_caught / max(1, _K)
    _f_F1 = (2*_f_P*_f_R/(_f_P+_f_R)) if (_f_P+_f_R) > 0 else 0
    _fusion_f1_per_q.append(_f_F1)
    _c_F1 = r["final_F1"]
    if _c_F1 > _f_F1:
        _cascade_better += 1
    print(f"  {qid:<10}  {r['gold']:>4}  {_K:>4}  "
          f"{_f_F1:>10.3f}  {_c_F1:>11.3f}  {_c_F1-_f_F1:>+7.3f}")
print("-" * 96)
_macro_fusion_f1 = sum(_fusion_f1_per_q) / max(1, len(_fusion_f1_per_q))
_macro_cascade_f1 = sum(_macro_f1) / max(1, len(_macro_f1))
print(f"  cascade wins on {_cascade_better}/{len(PER_QUERY)} queries  "
      f"(macro fusion F1 = {_macro_fusion_f1:.3f}  vs  "
      f"macro cascade F1 = {_macro_cascade_f1:.3f})")

# ============================================================================
# Save cascade results
# ============================================================================
import json as _json_s4
_out = PATHS["out_dir"]
_out.mkdir(parents=True, exist_ok=True)
_summary = {
    "macro_F1":      sum(_macro_f1)/len(_macro_f1),
    "macro_P":       sum(_macro_p)/len(_macro_p),
    "macro_R":       sum(_macro_r)/len(_macro_r),
    "micro_F1":      _micro_f1,
    "micro_P":       _micro_p,
    "micro_R":       _micro_r,
    "n_queries":     len(PER_QUERY),
    "cascade_wins":  _cascade_better,
    "per_query": {
        qid: {
            "gold":        PER_QUERY[qid]["gold"],
            "K":           PER_QUERY[qid]["final_K"],
            "precision":   PER_QUERY[qid]["final_precision"],
            "recall":      PER_QUERY[qid]["final_recall"],
            "F1":          PER_QUERY[qid]["final_F1"],
            "selected":    PER_QUERY[qid]["final_selected"],
            "verdicts":    {d: v for d, v in PER_QUERY[qid]["stage3_verdicts"].items() if d in PER_QUERY[qid]["final_selected"]},
            "stage1_R_at_K_100": PER_QUERY[qid]["stage1_R_at_K"][100][1],
            "stage2_R_at_K_100": PER_QUERY[qid]["stage2_R_at_K"][100][1],
        }
        for qid in PER_QUERY
    },
}
(_out / "summary_cascade.json").write_text(
    _json_s4.dumps(_summary, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
print(f"\n[stage4] wrote {_out / 'summary_cascade.json'}")


## 11.2 Cleanup (free GPU memory)

**What:** Delete large GPU tensors so a notebook re-run starts clean.

**Why:** Colab keeps state across cells; without explicit cleanup, re-running
Phase 6 will OOM.

In [ ]:
import gc
try:
    del E_GPU
except NameError:
    pass
try:
    del EMB_MODEL
except NameError:
    pass
gc.collect()
import torch as _torch2
if _torch2.cuda.is_available():
    _torch2.cuda.empty_cache()
    print(f"VRAM after cleanup: {_torch2.cuda.memory_allocated()/1024**3:.2f} GB")
print("cleaned up.")